# Smol Centering with Protected Tabular Residuals

This ensemble adds a trained SmolLM3 centered readout to the language-model scores, then retains the original contextual balance and five-class Claude expert. TabPFN-3.5 and default-normalized TabICL v1.1 contribute sequential conditional geometric residuals on the thirteen non-Claude classes. Each residual preserves the Claude probability columns and expert-selected Claude rows; any family-crossing prediction is reverted.

The fixed residual weights are 0.10 for TabPFN128 and 0.05 for TabICL64. On rows without reversion the sequential conditional exponents are 0.855, 0.095 and 0.05. There is no extra probability calibration or global normalization. SmolLM3 contributes 0.03 times its centered log-score before the original balance.

The baseline builds on [APEX attribution ensemble](https://www.kaggle.com/code/saddamaditya/apex-attribution-multi-lora-claude-gate), [Centered StyleGram](https://www.kaggle.com/code/obstacledeveloper/local-candidate-adaptive-centered-stylegram), [Lucas Boesen's LoRA and adaptive centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering) and [Zbigniew Kirchner's Claude ensemble](https://www.kaggle.com/code/zbigniewkirchner/ntl-v6). The added contribution is the trained SmolLM3 centered readout and protected serial tabular combination. Checkpoint and package notices remain with their attached assets.

Execution includes feature extraction, supervised classical readouts, train-only tabular feature preparation and tabular context construction. The observed validation comparison uses one common heldout prompt partition, reused development data and an author-reported Claude adapter exclusion assumption. It does not establish independent five-fold performance or predict a leaderboard score.

The two GPUs have distinct roles: TabPFN uses the first GPU while SmolLM3 and then TabICL run in separate processes on the second GPU. The LLM process exits before TabICL starts.


In [ ]:
import base64,hashlib,json,ast
from pathlib import Path
here=Path('/kaggle/working/smol-protected-icl-code')
here.mkdir(exist_ok=False)
payload=json.loads("{\"ntl_reference.py\":\"...\",\"capture_runtime.py\":\"IiIiQ2FwdHVyZSBleGlzdGluZyBmcm96ZW4gZmVhdHVyZXMgYW5kIGJhbGFuY2VkIGJhc2VsaW5lIGludG8gZXBoZW1lcmFsIHN0b3JhZ2UuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIGZ1bmN0b29scyBpbXBvcnQgd3JhcHMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucApmcm9tIGNsYXVkZV9nYXRlIGltcG9ydCBzYXZlX2dhdGVfc3RhdGUKClZJRVdTID0geydxM184Yl9jYXQ3KzIyJzo4MTkyLCAnc3R5bGU0Yic6NTEyMH0KCmRlZiBjYWNoZV9kaXJlY3RvcnkoKToKICAgIGRpcmVjdG9yeSA9IFBhdGgob3MuZW52aXJvblsnTlRMX1RBQlVMQVJfVEVNUCddKS5yZXNvbHZlKCkKICAgIGFzc2VydCBkaXJlY3RvcnkuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQogICAgZGlyZWN0b3J5Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIGRpcmVjdG9yeQoKZGVmIHdyYXBfZml0X2hlYWQob3JpZ2luYWwsIHRyYWluLCB0ZXN0LCBsYWJlbHMpOgogICAgY2FjaGUgPSBjYWNoZV9kaXJlY3RvcnkoKQogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChjYWNoZS8nYWxpZ25tZW50Lm5weicsCiAgICAgICAgdHJhaW5faWRzPXRyYWluLmlkLnRvX251bXB5KGR0eXBlPXN0ciksIHRlc3RfaWRzPXRlc3QuaWQudG9fbnVtcHkoZHR5cGU9c3RyKSwKICAgICAgICBsYWJlbHM9bnAuYXNhcnJheShsYWJlbHMsZHR5cGU9c3RyKSwKICAgICAgICB5PXRyYWluLm1vZGVsLm1hcCh7bGFiZWw6aSBmb3IgaSxsYWJlbCBpbiBlbnVtZXJhdGUobGFiZWxzKX0pLnRvX251bXB5KGR0eXBlPW5wLmludDY0KSkKICAgIGNhcHR1cmVkID0gc2V0KCkKICAgIEB3cmFwcyhvcmlnaW5hbCkKICAgIGRlZiBmaXR0aW5nKGtleSwgRXRyLCBFdGUsIEMpOgogICAgICAgIGlmIGtleSBpbiBWSUVXUzoKICAgICAgICAgICAgYXNzZXJ0IGtleSBub3QgaW4gY2FwdHVyZWQKICAgICAgICAgICAgYXNzZXJ0IEV0ci5kdHlwZT09RXRlLmR0eXBlPT1ucC5mbG9hdDMyCiAgICAgICAgICAgIGFzc2VydCBFdHIuc2hhcGU9PShsZW4odHJhaW4pLFZJRVdTW2tleV0pIGFuZCBFdGUuc2hhcGU9PShsZW4odGVzdCksVklFV1Nba2V5XSkKICAgICAgICAgICAgYXNzZXJ0IG5wLmlzZmluaXRlKEV0cikuYWxsKCkgYW5kIG5wLmlzZmluaXRlKEV0ZSkuYWxsKCkKICAgICAgICAgICAgZm9yIHN1ZmZpeCx2YWx1ZSBpbiBbKCd0cmFpbicsRXRyKSwoJ3Rlc3QnLEV0ZSldOgogICAgICAgICAgICAgICAgbnAuc2F2ZShjYWNoZS8oa2V5KydfJytzdWZmaXgrJy5ucHknKSx2YWx1ZSxhbGxvd19waWNrbGU9RmFsc2UpCiAgICAgICAgICAgIGNhcHR1cmVkLmFkZChrZXkpCiAgICAgICAgICAgIChjYWNoZS8nY2FwdHVyZWRfdmlld3MuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzb3J0ZWQoY2FwdHVyZWQpKSkKICAgICAgICByZXR1cm4gb3JpZ2luYWwoa2V5LEV0cixFdGUsQykKICAgIHJldHVybiBmaXR0aW5nCgpkZWYgY2FwdHVyZV9iYWxhbmNlZChQLHRlc3QsbGFiZWxzLERDX1RFLExDX1RFLENMQVVERV9JRFgsZ2F0ZV9sYW0sbG9yYV9sYW0pOgogICAgY2FjaGU9Y2FjaGVfZGlyZWN0b3J5KCkKICAgIGFzc2VydCBzZXQoanNvbi5sb2FkcygoY2FjaGUvJ2NhcHR1cmVkX3ZpZXdzLmpzb24nKS5yZWFkX3RleHQoKSkpPT1zZXQoVklFV1MpCiAgICBhc3NlcnQgUC5kdHlwZT09bnAuZmxvYXQ2NCBhbmQgUC5zaGFwZT09KGxlbih0ZXN0KSxsZW4obGFiZWxzKSkKICAgIGFzc2VydCBucC5pc2Zpbml0ZShQKS5hbGwoKSBhbmQgKFA+PTApLmFsbCgpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKGNhY2hlLydiYWxhbmNlZF9iYXNlbGluZS5ucHonLHByb2JhYmlsaXRpZXM9UCwKICAgICAgICBpZHM9dGVzdC5pZC50b19udW1weShkdHlwZT1zdHIpLGxhYmVscz1ucC5hc2FycmF5KGxhYmVscyxkdHlwZT1zdHIpKQogICAgc2F2ZV9nYXRlX3N0YXRlKGNhY2hlLFAsdGVzdC5pZC50b19udW1weShkdHlwZT1zdHIpLGxhYmVscywKICAgICAgICBEQ19URSxMQ19URSxDTEFVREVfSURYLGdhdGVfbGFtLGxvcmFfbGFtKQogICAgcmVjb3Jkcz17fQogICAgZm9yIHBhdGggaW4gc29ydGVkKGNhY2hlLmdsb2IoJyoubnB5JykpOgogICAgICAgIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIHN0cmVhbTogZGlnZXN0PWhhc2hsaWIuZmlsZV9kaWdlc3Qoc3RyZWFtLCdzaGEyNTYnKS5oZXhkaWdlc3QoKQogICAgICAgIHJlY29yZHNbcGF0aC5uYW1lXT1kaWN0KGJ5dGVzPXBhdGguc3RhdCgpLnN0X3NpemUsc2hhMjU2PWRpZ2VzdCkKICAgICMgT25seSBzaXplcyBhbmQgaGFzaGVzIGFyZSBleHBvcnRhYmxlOyBhcnJheXMgYW5kIHRhcmdldHMgc3RheSB1bmRlciAva2FnZ2xlL3RlbXAuCiAgICBQYXRoKCcva2FnZ2xlL3dvcmtpbmcvdGFidWxhcl9jYXB0dXJlX2F1ZGl0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdCgKICAgICAgICBzdGF0dXM9J1RXT19FWElTVElOR19WSUVXU19DQVBUVVJFRF9XSVRIT1VUX0VYVFJBX0JBQ0tCT05FX1BBU1MnLAogICAgICAgIHRyYWluX3Jvd3M9bGVuKG5wLmxvYWQoY2FjaGUvJ2FsaWdubWVudC5ucHonLGFsbG93X3BpY2tsZT1GYWxzZSlbJ3RyYWluX2lkcyddKSwKICAgICAgICBxdWVyeV9yb3dzPWxlbih0ZXN0KSx2aWV3cz1yZWNvcmRzLGJhc2VsaW5lX2NsYXNzZXM9bGlzdChsYWJlbHMpLAogICAgICAgIHNvdXJjZT0nZXhhY3QgbnRsLXY2IHYyIHJhdyBmaXRfaGVhZCBpbnB1dHMgYmVmb3JlIHByZXByb2Nlc3NpbmcnLAogICAgICAgIGdhdGVfc3RhdGVfY2FwdHVyZWRfYmVmb3JlX29yaWdpbmFsX2FyZ21heD1UcnVlLAogICAgICAgIGRjX2R0eXBlPU5vbmUgaWYgRENfVEUgaXMgTm9uZSBlbHNlIHN0cihEQ19URS5kdHlwZSksCiAgICAgICAgbGNfZHR5cGU9Tm9uZSBpZiBMQ19URSBpcyBOb25lIGVsc2Ugc3RyKExDX1RFLmR0eXBlKSwKICAgICAgICBjbGF1ZGVfaW5kaWNlc19kdHlwZT1zdHIoQ0xBVURFX0lEWC5kdHlwZSksCiAgICAgICAgZXBoZW1lcmFsX3RyYWluaW5nX2FycmF5cz1UcnVlKSxpbmRlbnQ9MikpCg==\",\"claude_gate.py\":\"IiIiRXhhY3QgbnRsLXY2IGhhcmQgQ2xhdWRlIGRlY2lzaW9uLCB3aXRoIGxvc3NsZXNzIHRlbXBvcmFyeSBzdGF0ZSB0cmFuc3BvcnQuIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgdmFsaWRhdGVfZ2F0ZShQLCBsYWJlbHMsIERDX1RFLCBMQ19URSwgQ0xBVURFX0lEWCwgZ2F0ZV9sYW0sIGxvcmFfbGFtKToKICAgIGxhYmVscyA9IG5wLmFzYXJyYXkobGFiZWxzKQogICAgYXNzZXJ0IGxhYmVscy5uZGltID09IDEgYW5kIGxlbihsYWJlbHMpID09IGxlbihucC51bmlxdWUobGFiZWxzKSkgPT0gMTgKICAgIGFzc2VydCBQLmR0eXBlID09IG5wLmZsb2F0NjQgYW5kIFAuc2hhcGVbMTpdID09IChsZW4obGFiZWxzKSwpCiAgICBhc3NlcnQgbnAuaXNmaW5pdGUoUCkuYWxsKCkgYW5kIChQID49IDApLmFsbCgpCiAgICBleHBlY3RlZCA9IG5wLmFycmF5KFtpIGZvciBpLCBsYWJlbCBpbiBlbnVtZXJhdGUobGFiZWxzKSBpZiBsYWJlbC5zdGFydHN3aXRoKCdjbGF1ZGUnKV0pCiAgICBhc3NlcnQgbGVuKGV4cGVjdGVkKSA9PSA1CiAgICBhc3NlcnQgbnAuaXNzdWJkdHlwZShDTEFVREVfSURYLmR0eXBlLCBucC5pbnRlZ2VyKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoQ0xBVURFX0lEWCwgZXhwZWN0ZWQpCiAgICBhc3NlcnQgKGdhdGVfbGFtLCBsb3JhX2xhbSkgPT0gKDEuNSwgMS4wKQogICAgZm9yIHZhbHVlIGluIChEQ19URSwgTENfVEUpOgogICAgICAgIGlmIHZhbHVlIGlzIG5vdCBOb25lOgogICAgICAgICAgICBhc3NlcnQgbnAuaXNzdWJkdHlwZSh2YWx1ZS5kdHlwZSwgbnAuZmxvYXRpbmcpCiAgICAgICAgICAgIGFzc2VydCB2YWx1ZS5zaGFwZSA9PSAobGVuKFApLCBsZW4oQ0xBVURFX0lEWCkpIGFuZCBucC5pc2Zpbml0ZSh2YWx1ZSkuYWxsKCkKCgpkZWYgZ2F0ZWRfcHJlZGljdChQLCBsYWJlbHMsIERDX1RFLCBMQ19URSwgQ0xBVURFX0lEWCwgZ2F0ZV9sYW09MS41LCBsb3JhX2xhbT0xLjApOgogICAgdmFsaWRhdGVfZ2F0ZShQLCBsYWJlbHMsIERDX1RFLCBMQ19URSwgQ0xBVURFX0lEWCwgZ2F0ZV9sYW0sIGxvcmFfbGFtKQogICAgIyBLZWVwIHRoZSByZWZlcmVuY2Ugb3BlcmF0aW9ucywgZHR5cGVzLCBjbGlwcGluZywgdGllIG9yZGVyIGFuZCBvcHRpb25hbCBnYXRlIGV4YWN0bHkuCiAgICBwcmVkID0gUC5hcmdtYXgoMSkKICAgIGlmIERDX1RFIGlzIG5vdCBOb25lOgogICAgICAgIF9pc2MgPSBucC5pc2luKHByZWQsIENMQVVERV9JRFgpCiAgICAgICAgX3MgPSBucC5sb2cobnAuY2xpcChQWzosIENMQVVERV9JRFhdLCAxZS0xMiwgMSkpICsgZ2F0ZV9sYW0gKiBEQ19URQogICAgICAgIGlmIExDX1RFIGlzIG5vdCBOb25lOgogICAgICAgICAgICBfcyA9IF9zICsgbG9yYV9sYW0gKiBMQ19URQogICAgICAgIF9uZXcgPSBDTEFVREVfSURYW19zLmFyZ21heCgxKV0KICAgICAgICBwcmVkW19pc2NdID0gX25ld1tfaXNjXQogICAgcmV0dXJuIHByZWQKCgpkZWYgc2F2ZV9nYXRlX3N0YXRlKGRpcmVjdG9yeSwgUCwgaWRzLCBsYWJlbHMsIERDX1RFLCBMQ19URSwgQ0xBVURFX0lEWCwgZ2F0ZV9sYW0sIGxvcmFfbGFtKToKICAgIHZhbGlkYXRlX2dhdGUoUCwgbGFiZWxzLCBEQ19URSwgTENfVEUsIENMQVVERV9JRFgsIGdhdGVfbGFtLCBsb3JhX2xhbSkKICAgICMgRXhwbGljaXQgZmxhZ3MgYXZvaWQgb2JqZWN0IGFycmF5cyBhbmQgcHJlc2VydmUgdGhlIGFjdHVhbCBwcmVzZW50LWFycmF5IGR0eXBlcy4KICAgIG5wLnNhdmV6X2NvbXByZXNzZWQoUGF0aChkaXJlY3RvcnkpLydjbGF1ZGVfZ2F0ZV9zdGF0ZS5ucHonLAogICAgICAgIGlkcz1ucC5hc2FycmF5KGlkcywgZHR5cGU9c3RyKSwgbGFiZWxzPW5wLmFzYXJyYXkobGFiZWxzLCBkdHlwZT1zdHIpLAogICAgICAgIGRjX3ByZXNlbnQ9bnAuYXNhcnJheShEQ19URSBpcyBub3QgTm9uZSksIGxjX3ByZXNlbnQ9bnAuYXNhcnJheShMQ19URSBpcyBub3QgTm9uZSksCiAgICAgICAgRENfVEU9RENfVEUgaWYgRENfVEUgaXMgbm90IE5vbmUgZWxzZSBucC5lbXB0eSgoMCwgNSksIGR0eXBlPW5wLmZsb2F0NjQpLAogICAgICAgIExDX1RFPUxDX1RFIGlmIExDX1RFIGlzIG5vdCBOb25lIGVsc2UgbnAuZW1wdHkoKDAsIDUpLCBkdHlwZT1ucC5mbG9hdDY0KSwKICAgICAgICBDTEFVREVfSURYPUNMQVVERV9JRFgsIGdhdGVfbGFtPW5wLmFzYXJyYXkoZ2F0ZV9sYW0pLCBsb3JhX2xhbT1ucC5hc2FycmF5KGxvcmFfbGFtKSkKCgpkZWYgbG9hZF9nYXRlX3N0YXRlKGRpcmVjdG9yeSwgUCwgaWRzLCBsYWJlbHMpOgogICAgd2l0aCBucC5sb2FkKFBhdGgoZGlyZWN0b3J5KS8nY2xhdWRlX2dhdGVfc3RhdGUubnB6JywgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBzYXZlZDoKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChzYXZlZFsnaWRzJ10sIGlkcykKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChzYXZlZFsnbGFiZWxzJ10sIGxhYmVscykKICAgICAgICBzdGF0ZSA9IGRpY3QoRENfVEU9c2F2ZWRbJ0RDX1RFJ10uY29weSgpIGlmIGJvb2woc2F2ZWRbJ2RjX3ByZXNlbnQnXSkgZWxzZSBOb25lLAogICAgICAgICAgICBMQ19URT1zYXZlZFsnTENfVEUnXS5jb3B5KCkgaWYgYm9vbChzYXZlZFsnbGNfcHJlc2VudCddKSBlbHNlIE5vbmUsCiAgICAgICAgICAgIENMQVVERV9JRFg9c2F2ZWRbJ0NMQVVERV9JRFgnXS5jb3B5KCksCiAgICAgICAgICAgIGdhdGVfbGFtPXNhdmVkWydnYXRlX2xhbSddLml0ZW0oKSwgbG9yYV9sYW09c2F2ZWRbJ2xvcmFfbGFtJ10uaXRlbSgpKQogICAgdmFsaWRhdGVfZ2F0ZShQLCBsYWJlbHMsICoqc3RhdGUpCiAgICByZXR1cm4gc3RhdGUK\",\"masked_features.py\":\"IiIiRml4ZWQgc2hhcGUvZnVuY3Rpb24gVEYtSURGLVNWRCBibG9ja3M7IGFsbCBsZWFybmVkIHRyYW5zZm9ybXMgZml0IG9uIHRyYWluIG9ubHkuIiIiCgppbXBvcnQgcmUKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSBza2xlYXJuLmZlYXR1cmVfZXh0cmFjdGlvbi50ZXh0IGltcG9ydCBUZmlkZlZlY3Rvcml6ZXIKCmZyb20gc2tsZWFybi5kZWNvbXBvc2l0aW9uIGltcG9ydCBUcnVuY2F0ZWRTVkQKCkZVTkNUSU9OX1dPUkRTX1RFWFQ9J2EgYW4gdGhlIGFuZCBvciBidXQgaWYgd2hpbGUgYWx0aG91Z2ggYmVjYXVzZSBhcyBzbyB0aGFuIHRoYXQgd2hpY2ggd2hvIHdob20gd2hvc2UgdGhpcyB0aGVzZSB0aG9zZSBpdCBpdHMgdGhleSB0aGVtIHRoZWlyIHRoZWlycyBoZSBoaW0gaGlzIHNoZSBoZXIgaGVycyB3ZSB1cyBvdXIgb3VycyB5b3UgeW91ciB5b3VycyBpIG1lIG15IG1pbmUgaW4gb24gYXQgdG8gZnJvbSB3aXRoIHdpdGhvdXQgZm9yIG9mIGJ5IG92ZXIgdW5kZXIgYmV0d2VlbiB0aHJvdWdoIGludG8gYWJvdXQgaXMgYXJlIHdhcyB3ZXJlIGJlIGJlZW4gYmVpbmcgaGF2ZSBoYXMgaGFkIGRvIGRvZXMgZGlkIGNhbiBjb3VsZCB3aWxsIHdvdWxkIHNoYWxsIHNob3VsZCBtYXkgbWlnaHQgbXVzdCBub3Qgbm8gbm9yIGFsbCBhbnkgc29tZSBlYWNoIGV2ZXJ5IGJvdGggZWl0aGVyIG5laXRoZXIgb3RoZXIgYW5vdGhlciBzdWNoIG9ubHkgYWxzbyBqdXN0IHZlcnkgdG9vIG1vcmUgbW9zdCBsZXNzIGxlYXN0IGhlcmUgdGhlcmUgd2hlbiB3aGVyZSB3aHkgaG93IHdoYXQgd2hldGhlciB0aGVuIG5vdyBzdGlsbCB5ZXQgZXZlbiBtdWNoIG1hbnkgZmV3IGxpdHRsZScKCmRlZiBzaGFwZV90ZXh0KHRleHQpOgogICAgIyBQcmVzZXJ2ZSBjaGFyYWN0ZXIgY291bnQvY2FzZSwgcHVuY3R1YXRpb24gYW5kIHdoaXRlc3BhY2U7IHN1cHByZXNzIGxldHRlcnMvZGlnaXRzLgogICAgcmV0dXJuIHJlLnN1YihyJ1xkJywnIycscmUuc3ViKHInW15cV1xkX10nLGxhbWJkYSBtOidBJyBpZiBtWzBdLmlzdXBwZXIoKSBlbHNlICdhJyx0ZXh0KSkKCmRlZiBmdW5jdGlvbl90ZXh0KHRleHQpOgogICAgd29yZHM9c2V0KEZVTkNUSU9OX1dPUkRTX1RFWFQuc3BsaXQoKSkKICAgIHJldHVybiAnICcuam9pbih0Lmxvd2VyKCkgaWYgdC5sb3dlcigpIGluIHdvcmRzIGVsc2UgKCdDT05URU5UJyBpZiB0LmlzYWxudW0oKSBlbHNlIHQpCiAgICAgICAgZm9yIHQgaW4gcmUuZmluZGFsbChyJ1x3K3xbXlx3XHNdJyx0ZXh0KSkKClZFQ1RPUklaRVJTID0geydzaGFwZV9jaGFyJzogeydtaW5fZGYnOiAzLCAnbWF4X2RmJzogMC45OTUsICdzdWJsaW5lYXJfdGYnOiBUcnVlLCAnbm9ybSc6ICdsMicsICdhbmFseXplcic6ICdjaGFyJywgJ25ncmFtX3JhbmdlJzogWzMsIDVdLCAnbG93ZXJjYXNlJzogRmFsc2UsICdtYXhfZmVhdHVyZXMnOiAyMDAwMH0sICdmdW5jdGlvbl93b3JkJzogeydtaW5fZGYnOiAzLCAnbWF4X2RmJzogMC45OTUsICdzdWJsaW5lYXJfdGYnOiBUcnVlLCAnbm9ybSc6ICdsMicsICdhbmFseXplcic6ICd3b3JkJywgJ25ncmFtX3JhbmdlJzogWzEsIDNdLCAnbG93ZXJjYXNlJzogRmFsc2UsICd0b2tlbl9wYXR0ZXJuJzogJyg/dSlcXGJcXHcrXFxifFteXFx3XFxzXScsICdtYXhfZmVhdHVyZXMnOiAyMDAwMH19CgpkZWYgbWFza2VkX3N2ZF9ibG9ja3ModHJhaW5fdGV4dHMsIHF1ZXJ5X3RleHRzKToKICAgIHRyYWluX2Jsb2NrcywgcXVlcnlfYmxvY2tzLCBhdWRpdCA9IFtdLCBbXSwgW10KICAgIGZvciBuYW1lLCBjZmcgaW4gVkVDVE9SSVpFUlMuaXRlbXMoKToKICAgICAgICBtYXNrZXIgPSBzaGFwZV90ZXh0IGlmIG5hbWUgPT0gJ3NoYXBlX2NoYXInIGVsc2UgZnVuY3Rpb25fdGV4dAogICAgICAgIGZpdF92YWx1ZXMgPSBbbWFza2VyKHRleHQpIGZvciB0ZXh0IGluIHRyYWluX3RleHRzXQogICAgICAgIHF1ZXJ5X3ZhbHVlcyA9IFttYXNrZXIodGV4dCkgZm9yIHRleHQgaW4gcXVlcnlfdGV4dHNdCiAgICAgICAgb3B0aW9ucyA9IGRpY3QoY2ZnKQogICAgICAgIG9wdGlvbnNbJ25ncmFtX3JhbmdlJ10gPSB0dXBsZShvcHRpb25zWyduZ3JhbV9yYW5nZSddKQogICAgICAgIG9wdGlvbnNbJ2R0eXBlJ10gPSBucC5mbG9hdDMyCiAgICAgICAgdmVjdG9yaXplciA9IFRmaWRmVmVjdG9yaXplcigqKm9wdGlvbnMpCiAgICAgICAgc3BhcnNlX2ZpdCA9IHZlY3Rvcml6ZXIuZml0X3RyYW5zZm9ybShmaXRfdmFsdWVzKQogICAgICAgIGFzc2VydCBzcGFyc2VfZml0LnNoYXBlWzFdID4gMTYKICAgICAgICBzdmQgPSBUcnVuY2F0ZWRTVkQobl9jb21wb25lbnRzPTE2LCBuX2l0ZXI9NSwgcmFuZG9tX3N0YXRlPTIwMjYxMDE3KQogICAgICAgIHN2ZC5maXQoc3BhcnNlX2ZpdCkKICAgICAgICB0cmFpbl9ibG9jayA9IHN2ZC50cmFuc2Zvcm0odmVjdG9yaXplci50cmFuc2Zvcm0oZml0X3ZhbHVlcykpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHF1ZXJ5X2Jsb2NrID0gc3ZkLnRyYW5zZm9ybSh2ZWN0b3JpemVyLnRyYW5zZm9ybShxdWVyeV92YWx1ZXMpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBhc3NlcnQgdHJhaW5fYmxvY2suc2hhcGUgPT0gKGxlbih0cmFpbl90ZXh0cyksIDE2KQogICAgICAgIGFzc2VydCBxdWVyeV9ibG9jay5zaGFwZSA9PSAobGVuKHF1ZXJ5X3RleHRzKSwgMTYpCiAgICAgICAgYXNzZXJ0IG5wLmlzZmluaXRlKHRyYWluX2Jsb2NrKS5hbGwoKSBhbmQgbnAuaXNmaW5pdGUocXVlcnlfYmxvY2spLmFsbCgpCiAgICAgICAgdHJhaW5fYmxvY2tzLmFwcGVuZCh0cmFpbl9ibG9jaykKICAgICAgICBxdWVyeV9ibG9ja3MuYXBwZW5kKHF1ZXJ5X2Jsb2NrKQogICAgICAgIGF1ZGl0LmFwcGVuZChkaWN0KHZpZXc9bmFtZSwgdm9jYWJ1bGFyeV9zaXplPWxlbih2ZWN0b3JpemVyLnZvY2FidWxhcnlfKSwKICAgICAgICAgICAgZml0X3Jvd3M9bGVuKHRyYWluX3RleHRzKSwgcXVlcnlfcm93cz1sZW4ocXVlcnlfdGV4dHMpLCBjb21wb25lbnRzPTE2LAogICAgICAgICAgICBmaXRfc2NvcGU9J29mZmljaWFsX3RyYWluX29ubHknLAogICAgICAgICAgICBleHBsYWluZWRfdmFyaWFuY2VfcmF0aW9fc3VtPWZsb2F0KHN2ZC5leHBsYWluZWRfdmFyaWFuY2VfcmF0aW9fLnN1bSgpKSkpCiAgICByZXR1cm4gbnAuY29uY2F0ZW5hdGUodHJhaW5fYmxvY2tzLCBheGlzPTEpLCBucC5jb25jYXRlbmF0ZShxdWVyeV9ibG9ja3MsIGF4aXM9MSksIGF1ZGl0Cgo=\",\"tabular_worker.py\":\"IiIiRml4ZWQgdHJhaW4tb25seSBmZWF0dXJlIHByZXBhcmF0aW9uIG9yIG9uZSBpc29sYXRlZCB0YWJ1bGFyIGluZmVyZW5jZSBtZW1iZXIuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBpbXBvcnRsaWIubWV0YWRhdGEKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKYXNzZXJ0IG9zLm5hbWUhPSdudCcgYW5kIFBhdGgoJy9rYWdnbGUvaW5wdXQnKS5pc19kaXIoKQpmb3IgbmFtZSBpbiAoJ09NUF9OVU1fVEhSRUFEUycsJ09QRU5CTEFTX05VTV9USFJFQURTJywnTUtMX05VTV9USFJFQURTJyk6b3MuZW52aXJvbltuYW1lXT0nNCcKb3MuZW52aXJvblsnSEZfSFVCX09GRkxJTkUnXT0nMScKaW1wb3J0IG51bXB5IGFzIG5wCgpoZXJlPVBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKY29uZmlnPWpzb24ubG9hZHMoKGhlcmUvJ3J1bnRpbWVfY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKY2FjaGU9UGF0aChvcy5lbnZpcm9uWydOVExfVEFCVUxBUl9URU1QJ10pLnJlc29sdmUoKQphc3NlcnQgY2FjaGUuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQptb2RlPXN5cy5hcmd2WzFdCnN0YXJ0ZWQ9dGltZS5tb25vdG9uaWMoKQoKZGVmIHNoYShwYXRoKToKICAgIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIHN0cmVhbTpyZXR1cm4gaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sJ3NoYTI1NicpLmhleGRpZ2VzdCgpCgpkZWYgc3R5bGUyNCh0KToKICAgICMgT3JpZ2luYWwgcmVmZXJlbmNlIGZpcnN0IDIwIGRldGVybWluaXN0aWMgc2NhbGFycywgcGx1cyBmb3VyIGZpeGVkIHJhdGlvcy4KICAgIHdvcmRzID0gcmUuZmluZGFsbChyIltBLVphLXpcdTIwMTknXSsiLCB0KQogICAgbncsIG5jID0gbWF4KGxlbih3b3JkcyksIDEpLCBtYXgobGVuKHQpLCAxKQogICAgc2VudHMgPSBbeCBmb3IgeCBpbiByZS5zcGxpdChyJyg/PD1bLiE/XSlccysnLCB0LnN0cmlwKCkpIGlmIHhdCiAgICBzbCA9IG5wLmFycmF5KFtsZW4ocmUuZmluZGFsbChyIltBLVphLXpcdTIwMTknXSsiLCB4KSkgZm9yIHggaW4gc2VudHNdKSBpZiBzZW50cyBlbHNlIG5wLmFycmF5KFswXSkKICAgIHdsID0gbnAuYXJyYXkoW2xlbih4KSBmb3IgeCBpbiB3b3Jkc10pIGlmIHdvcmRzIGVsc2UgbnAuYXJyYXkoWzBdKQogICAgbHcgPSBbeC5sb3dlcigpIGZvciB4IGluIHdvcmRzXQogICAgcmV0dXJuIFtucC5sb2cxcChudyksIGxlbihzZW50cyksIHNsLm1lYW4oKSwgc2wuc3RkKCksIHNsLm1heCgpLCBzbC5taW4oKSwgd2wubWVhbigpLCB3bC5zdGQoKSwKICAgICAgICAod2wgPj0gOSkubWVhbigpLCBsZW4oc2V0KGx3KSkvbncsIHN1bShjLmlzdXBwZXIoKSBmb3IgYyBpbiB0KS9uYywgc3VtKGMuaXNkaWdpdCgpIGZvciBjIGluIHQpL25jLAogICAgICAgIHN1bShvcmQoYyk+MTI3IGZvciBjIGluIHQpL25jLCB0LmNvdW50KCcsICcpL253LCB0LmNvdW50KCcgYW5kICcpL253LCB0LmNvdW50KCcgYnV0ICcpL253LAogICAgICAgIHN1bSh4WzoxXS5pc3VwcGVyKCkgYW5kIHguc3BsaXQoKVswXS5sb3dlcigpIGluICgndGhpcycsJ3RoZXNlJywnaXQnLCd0aGV5JykgZm9yIHggaW4gc2VudHMpL21heChsZW4oc2VudHMpLDEpLAogICAgICAgIHN1bSh4LmVuZHN3aXRoKCdseScpIGZvciB4IGluIHdvcmRzKS9udywgc3VtKHguZW5kc3dpdGgoJ2luZycpIGZvciB4IGluIHdvcmRzKS9udywKICAgICAgICBzdW0oeC5lbmRzd2l0aCgndGlvbicpIGZvciB4IGluIHdvcmRzKS9udywKICAgICAgICBmbG9hdChhbnkoYyBpbiB0IGZvciBjIGluICgnXHUyMDE5JywnXHUyMDFjJywnXHUyMDFkJykpKSwgdC5jb3VudCgnXHUyMDE0JykvbncsCiAgICAgICAgdC5jb3VudCgnXG4nKS9uYywgc3VtKGJvb2woeC5zdHJpcCgpKSBmb3IgeCBpbiB0LnNwbGl0bGluZXMoKSkvbnddCgp3aXRoIG5wLmxvYWQoY2FjaGUvJ2FsaWdubWVudC5ucHonLGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgc2F2ZWQ6CiAgICBhbGlnbm1lbnQ9e2tleTpzYXZlZFtrZXldLmNvcHkoKSBmb3Iga2V5IGluIHNhdmVkLmZpbGVzfQppZiBtb2RlPT0ncHJlcGFyZSc6CiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCiAgICBmcm9tIHNrbGVhcm4uZGVjb21wb3NpdGlvbiBpbXBvcnQgUENBCiAgICBmcm9tIHRocmVhZHBvb2xjdGwgaW1wb3J0IHRocmVhZHBvb2xfbGltaXRzCiAgICB0aHJlYWRwb29sX2xpbWl0cyg0KQogICAgZGF0YT1uZXh0KHBhdGggZm9yIHBhdGggaW4gW1BhdGgoJy9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zL25hbWUtdGhhdC1sbG0nKSxQYXRoKCcva2FnZ2xlL2lucHV0L25hbWUtdGhhdC1sbG0nKV0gaWYgKHBhdGgvJ3RyYWluLmNzdicpLmlzX2ZpbGUoKSkKICAgIHRyYWluLHRlc3Q9cGQucmVhZF9jc3YoZGF0YS8ndHJhaW4uY3N2JykscGQucmVhZF9jc3YoZGF0YS8ndGVzdC5jc3YnKQogICAgZm9yIGZyYW1lLGtleSBpbiBbKHRyYWluLCd0cmFpbl9pZHMnKSwodGVzdCwndGVzdF9pZHMnKV06CiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoZnJhbWUuaWQudG9fbnVtcHkoZHR5cGU9c3RyKSxhbGlnbm1lbnRba2V5XSkKICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHRyYWluLm1vZGVsLnRvX251bXB5KGR0eXBlPXN0ciksYWxpZ25tZW50WydsYWJlbHMnXVthbGlnbm1lbnRbJ3knXV0pCiAgICB0cmFpbl9jb2x1bW5zLHRlc3RfY29sdW1ucyxhdWRpdD1bXSxbXSxbXQogICAgZm9yIGtleSx3aWR0aCBpbiBbKCdxM184Yl9jYXQ3KzIyJyw4MTkyKSwoJ3N0eWxlNGInLDUxMjApXToKICAgICAgICB0cmFpbl92aWV3PW5wLmxvYWQoY2FjaGUvKGtleSsnX3RyYWluLm5weScpLG1tYXBfbW9kZT0ncicsYWxsb3dfcGlja2xlPUZhbHNlKQogICAgICAgIHRlc3Rfdmlldz1ucC5sb2FkKGNhY2hlLyhrZXkrJ190ZXN0Lm5weScpLG1tYXBfbW9kZT0ncicsYWxsb3dfcGlja2xlPUZhbHNlKQogICAgICAgIGFzc2VydCB0cmFpbl92aWV3LnNoYXBlPT0obGVuKHRyYWluKSx3aWR0aCkgYW5kIHRlc3Rfdmlldy5zaGFwZT09KGxlbih0ZXN0KSx3aWR0aCkKICAgICAgICBwY2E9UENBKG5fY29tcG9uZW50cz02NCxzdmRfc29sdmVyPSdyYW5kb21pemVkJyxpdGVyYXRlZF9wb3dlcj0zLHJhbmRvbV9zdGF0ZT00MikKICAgICAgICBwY2EuZml0KHRyYWluX3ZpZXcpCiAgICAgICAgdHJhaW5fY29sdW1ucy5hcHBlbmQocGNhLnRyYW5zZm9ybSh0cmFpbl92aWV3KS5hc3R5cGUobnAuZmxvYXQzMikpCiAgICAgICAgdGVzdF9jb2x1bW5zLmFwcGVuZChwY2EudHJhbnNmb3JtKHRlc3RfdmlldykuYXN0eXBlKG5wLmZsb2F0MzIpKQogICAgICAgIGF1ZGl0LmFwcGVuZChkaWN0KHZpZXc9a2V5LGZpdF9yb3dzPWxlbih0cmFpbiksY29tcG9uZW50cz02NCxmaXRfc2NvcGU9J29mZmljaWFsX3RyYWluX29ubHknLAogICAgICAgICAgICBleHBsYWluZWRfdmFyaWFuY2U9ZmxvYXQocGNhLmV4cGxhaW5lZF92YXJpYW5jZV9yYXRpb18uc3VtKCkpKSkKICAgIGZvciBmcmFtZSxjb2x1bW5zIGluIFsodHJhaW4sdHJhaW5fY29sdW1ucyksKHRlc3QsdGVzdF9jb2x1bW5zKV06CiAgICAgICAgdGV4dHM9ZnJhbWUudGV4dC5maWxsbmEoJycpLmFzdHlwZShzdHIpCiAgICAgICAgY29sdW1ucy5hcHBlbmQobnAuYXNhcnJheShbc3R5bGUyNCh0ZXh0KSBmb3IgdGV4dCBpbiB0ZXh0c10sZHR5cGU9bnAuZmxvYXQzMikpCiAgICBhc3NlcnQgc2hhKGhlcmUvJ21hc2tlZF9mZWF0dXJlcy5weScpPT1jb25maWdbJ21hc2tlZF9mZWF0dXJlX21vZHVsZV9zaGEyNTYnXQogICAgYmFzZV90cmFpbixiYXNlX3Rlc3Q9bnAuY29uY2F0ZW5hdGUodHJhaW5fY29sdW1ucyxheGlzPTEpLG5wLmNvbmNhdGVuYXRlKHRlc3RfY29sdW1ucyxheGlzPTEpCiAgICBmcm9tIG1hc2tlZF9mZWF0dXJlcyBpbXBvcnQgbWFza2VkX3N2ZF9ibG9ja3MKICAgIHRleHRfdHJhaW4sdGV4dF90ZXN0LHRleHRfYXVkaXQ9bWFza2VkX3N2ZF9ibG9ja3MoCiAgICAgICAgdHJhaW4udGV4dC5maWxsbmEoJycpLmFzdHlwZShzdHIpLnRvbGlzdCgpLHRlc3QudGV4dC5maWxsbmEoJycpLmFzdHlwZShzdHIpLnRvbGlzdCgpKQogICAgdHJhaW5fY29sdW1ucy5hcHBlbmQodGV4dF90cmFpbikKICAgIHRlc3RfY29sdW1ucy5hcHBlbmQodGV4dF90ZXN0KQogICAgeF90cmFpbix4X3Rlc3Q9bnAuY29uY2F0ZW5hdGUodHJhaW5fY29sdW1ucyxheGlzPTEpLG5wLmNvbmNhdGVuYXRlKHRlc3RfY29sdW1ucyxheGlzPTEpCiAgICBhc3NlcnQgeF90cmFpbi5zaGFwZT09KGxlbih0cmFpbiksMTg0KSBhbmQgeF90ZXN0LnNoYXBlPT0obGVuKHRlc3QpLDE4NCkKICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHhfdHJhaW5bOiw6MTUyXSxiYXNlX3RyYWluKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoeF90ZXN0WzosOjE1Ml0sYmFzZV90ZXN0KQogICAgYXNzZXJ0IG5wLmlzZmluaXRlKHhfdHJhaW4pLmFsbCgpIGFuZCBucC5pc2Zpbml0ZSh4X3Rlc3QpLmFsbCgpCiAgICBucC5zYXZleihjYWNoZS8ndGFibGVzLm5weicseF90cmFpbj14X3RyYWluLHhfdGVzdD14X3Rlc3QsKiphbGlnbm1lbnQpCiAgICBQYXRoKCcva2FnZ2xlL3dvcmtpbmcvdGFidWxhcl9wcmVwYXJlX2F1ZGl0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdCgKICAgICAgICBzdGF0dXM9J0ZVTExfVFJBSU5fT05MWV9QQ0FfU1RZTEUyNF9BTkRfTUFTS0VEX1NWRF9DT01QTEVURScscGNhPWF1ZGl0LHRleHRfc3ZkPXRleHRfYXVkaXQsZmVhdHVyZXM9MTg0LAogICAgICAgIG1hc2tlZF9mZWF0dXJlX21vZHVsZV9zaGEyNTY9Y29uZmlnWydtYXNrZWRfZmVhdHVyZV9tb2R1bGVfc2hhMjU2J10sCiAgICAgICAgcXVlcnlfcm93cz1sZW4odGVzdCksZWxhcHNlZF9zZWNvbmRzPXRpbWUubW9ub3RvbmljKCktc3RhcnRlZCwKICAgICAgICBwYWNrYWdlcz17bmFtZTppbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihuYW1lKSBmb3IgbmFtZSBpbiAoJ251bXB5JywncGFuZGFzJywnc2Npa2l0LWxlYXJuJywnc2NpcHknKX0sCiAgICAgICAgdHJhaW5pbmdfY29udGV4dF9leHBvcnRlZD1GYWxzZSksaW5kZW50PTIpKQogICAgc3lzLmV4aXQoMCkKCmFzc2VydCBtb2RlPT0ndGFicGZuJwppbXBvcnQgdG9yY2gKYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkKYXNzZXJ0IHRvcmNoLmN1ZGEubWVtX2dldF9pbmZvKClbMF0+OCoxMDI0KiozCndpdGggbnAubG9hZChjYWNoZS8ndGFibGVzLm5weicsYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBzYXZlZDoKICAgIHhfdHJhaW4seF90ZXN0LHk9KHNhdmVkW2tleV0uY29weSgpIGZvciBrZXkgaW4gKCd4X3RyYWluJywneF90ZXN0JywneScpKQphc3NldD1QYXRoKG9zLmVudmlyb25bJ05UTF9UQUJVTEFSX0FTU0VUJ10pCnNldHRpbmdzPWNvbmZpZ1ttb2RlXQpjaGVja3BvaW50PWFzc2V0L3NldHRpbmdzWydjaGVja3BvaW50X2ZpbGVuYW1lJ10KYXNzZXJ0IGNoZWNrcG9pbnQuaXNfZmlsZSgpIGFuZCBzaGEoY2hlY2twb2ludCk9PXNldHRpbmdzWydjaGVja3BvaW50X3NoYTI1NiddCmFzc2VydCBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihtb2RlKT09c2V0dGluZ3NbJ3BhY2thZ2VfdmVyc2lvbiddCmFzc2VydCBzZXR0aW5nc1snbl9lc3RpbWF0b3JzJ109PTEyOCwnUHJlc2VydmUgdGhlIHNlbGVjdGVkMTI4LWVzdGltYXRvciBjb25maWd1cmF0aW9uJwpmcm9tIHRhYnBmbiBpbXBvcnQgVGFiUEZOQ2xhc3NpZmllcgpmcm9tIHRhYnBmbi5jb25zdGFudHMgaW1wb3J0IE1vZGVsVmVyc2lvbgpwYXJhbXM9ZGljdChuX2VzdGltYXRvcnM9MTI4LG1vZGVsX3BhdGg9c3RyKGNoZWNrcG9pbnQpLGRldmljZT0nY3VkYTowJyxmaXRfbW9kZT0nbG93X21lbW9yeScsCiAgICBtZW1vcnlfc2F2aW5nX21vZGU9J2F1dG8nLGJhbGFuY2VfcHJvYmFiaWxpdGllcz1UcnVlLHR1bmluZ19jb25maWc9Tm9uZSxyYW5kb21fc3RhdGU9NDIsbl9wcmVwcm9jZXNzaW5nX2pvYnM9MSkKbW9kZWw9VGFiUEZOQ2xhc3NpZmllci5jcmVhdGVfZGVmYXVsdF9mb3JfdmVyc2lvbihNb2RlbFZlcnNpb24uVjNfNSwqKnBhcmFtcykKdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCnRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTt0PXRpbWUubW9ub3RvbmljKCkKbW9kZWwuZml0KHhfdHJhaW4seSkKdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO2ZpdF9zZWNvbmRzPXRpbWUubW9ub3RvbmljKCktdAp0PXRpbWUubW9ub3RvbmljKCk7cmF3PW1vZGVsLnByZWRpY3RfcHJvYmEoeF90ZXN0KQp0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7cHJlZGljdF9zZWNvbmRzPXRpbWUubW9ub3RvbmljKCktdApjbGFzc2VzPW5wLmFzYXJyYXkobW9kZWwuY2xhc3Nlc18sZHR5cGU9aW50KQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChucC5zb3J0KGNsYXNzZXMpLG5wLmFyYW5nZSgxOCkpCnByb2JhYmlsaXRpZXM9bnAuemVyb3MoKGxlbih4X3Rlc3QpLDE4KSxkdHlwZT1ucC5mbG9hdDY0KQpwcm9iYWJpbGl0aWVzWzosY2xhc3Nlc109cmF3CmFzc2VydCBucC5pc2Zpbml0ZShwcm9iYWJpbGl0aWVzKS5hbGwoKSBhbmQgKHByb2JhYmlsaXRpZXM+PTApLmFsbCgpCm5wLnRlc3RpbmcuYXNzZXJ0X2FsbGNsb3NlKHByb2JhYmlsaXRpZXMuc3VtKDEpLDEscnRvbD0wLGF0b2w9MWUtNSkKbnAuc2F2ZXooY2FjaGUvKG1vZGUrJ19wcm9iYS5ucHonKSxwcm9iYT1wcm9iYWJpbGl0aWVzLGlkcz1hbGlnbm1lbnRbJ3Rlc3RfaWRzJ10sbGFiZWxzPWFsaWdubWVudFsnbGFiZWxzJ10pClBhdGgoJy9rYWdnbGUvd29ya2luZy8nK21vZGUrJ19pbmZlcmVuY2VfYXVkaXQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KAogICAgc3RhdHVzPSdDT01QTEVURScsbW9kZWw9bW9kZSxjaGVja3BvaW50X3NoYTI1Nj1zZXR0aW5nc1snY2hlY2twb2ludF9zaGEyNTYnXSxwYXJhbWV0ZXJzPXBhcmFtcywKICAgIHRyYWluX3Jvd3M9bGVuKHhfdHJhaW4pLHF1ZXJ5X3Jvd3M9bGVuKHhfdGVzdCksZmVhdHVyZXM9MTg0LGZpdF9zZWNvbmRzPWZpdF9zZWNvbmRzLAogICAgcHJlZGljdF9zZWNvbmRzPXByZWRpY3Rfc2Vjb25kcyxwZWFrX2N1ZGFfYWxsb2NhdGVkX2J5dGVzPWludCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkpLAogICAgcGVha19jdWRhX3Jlc2VydmVkX2J5dGVzPWludCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfcmVzZXJ2ZWQoKSkscHJvYmFiaWxpdHlfdHJhbnNmb3JtPSdub25lJywKICAgIHJvd19tYXNzX21heGFicz1mbG9hdChucC5tYXgobnAuYWJzKHByb2JhYmlsaXRpZXMuc3VtKDEpLTEpKSksCiAgICBwYWNrYWdlcz17bmFtZTppbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihuYW1lKSBmb3IgbmFtZSBpbiAobW9kZSwndG9yY2gnLCdudW1weScsJ3NjaWtpdC1sZWFybicpfSwKICAgIG9vZj1GYWxzZSxtb2RlbF9jb250ZXh0X2V4cG9ydGVkPUZhbHNlKSxpbmRlbnQ9MikpCg==\",\"run_baseline.py\":\"IiIiRXhhY3QgZnJvemVuIEI1NTYgd2l0aCBvbmUgcmV2ZXJzaWJsZSBvYnNlcnZhdGlvbiBob29rIGFuZCBwb3N0LWV4ZWN1dGlvbiBleHBvcnRzLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gY2FwdHVyZV9ydW50aW1lIGltcG9ydCB3cmFwX2ZpdF9oZWFkLCBjYXB0dXJlX2JhbGFuY2VkCgphc3NlcnQgb3MubmFtZSAhPSAnbnQnIGFuZCBQYXRoKCcva2FnZ2xlL2lucHV0JykuaXNfZGlyKCkKaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKY29uZmlnID0ganNvbi5sb2FkcygoaGVyZSAvICdydW50aW1lX2NvbmZpZy5qc29uJykucmVhZF90ZXh0KCkpCnJhdyA9IChoZXJlIC8gJ250bF9yZWZlcmVuY2UucHknKS5yZWFkX2J5dGVzKCkKYXNzZXJ0IGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgPT0gY29uZmlnWydyZWZlcmVuY2Vfc291cmNlX3NoYTI1NiddCnNvdXJjZSA9IHJhdy5kZWNvZGUoJ3V0Zi04JykKYXNzZXJ0IGNvbmZpZ1snYmFzZWxpbmVfYmFsYW5jZV9hbHBoYSddID09IC41CmFzc2VydCBjb25maWdbJ3JlZmVyZW5jZV9ub3RlYm9va19pZCddID09IDEzOTIwMDEwMCBhbmQgY29uZmlnWydyZWZlcmVuY2VfdmVyc2lvbiddID09IDIKYXNzZXJ0ICdCQUxBTkNFX0FMUEhBID0gMC41JyBpbiBzb3VyY2UgYW5kICJXID0geydzdHlsZWdyYW0nOiAxLjUxNiIgaW4gc291cmNlCmxvb3AgPSAnZm9yIGtleSwgbXQsIGhpZCwgbGF5ZXJzLCBDLCBzbW9rZV9pZCwgcG9vbCBpbiBMTE1fSEVBRFM6Jwpob29rID0gJ2ZpdF9oZWFkID0gX3RhYnVsYXJfd3JhcF9maXRfaGVhZChmaXRfaGVhZCwgdHJhaW4sIHRlc3QsIExBQkVMUylcbicgKyBsb29wCmFzc2VydCBzb3VyY2UuY291bnQobG9vcCkgPT0gMQptb2RpZmllZCA9IHNvdXJjZS5yZXBsYWNlKGxvb3AsIGhvb2spCmFzc2VydCBtb2RpZmllZC5yZXBsYWNlKGhvb2ssIGxvb3ApID09IHNvdXJjZQpuYW1lc3BhY2UgPSBkaWN0KF9fbmFtZV9fPSdfX21haW5fXycsIF9fZmlsZV9fPXN0cihoZXJlIC8gJ250bF9yZWZlcmVuY2UucHknKSwKICAgICAgICAgICAgICAgICBfdGFidWxhcl93cmFwX2ZpdF9oZWFkPXdyYXBfZml0X2hlYWQpCmV4ZWMoY29tcGlsZShtb2RpZmllZCwgJ0I1NTZfb2JzZXJ2YXRpb25fb25seScsICdleGVjJyksIG5hbWVzcGFjZSkKYXNzZXJ0IFBhdGgoJy9rYWdnbGUvd29ya2luZy9zdWJtaXNzaW9uLnBhcnF1ZXQnKS5pc19maWxlKCkKaW1wb3J0IG51bXB5IGFzIG5wCmNhY2hlID0gUGF0aChvcy5lbnZpcm9uWydOVExfVEFCVUxBUl9URU1QJ10pLnJlc29sdmUoKQphc3NlcnQgY2FjaGUuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQpQLCByYXdfbG9naXRzLCB0ZXN0LCBsYWJlbHMgPSAobmFtZXNwYWNlW2tleV0gZm9yIGtleSBpbiAoJ1AnLCAnTE9HSVQnLCAndGVzdCcsICdMQUJFTFMnKSkKZ3JvdXBzLCBhZGFwdGl2ZSwgcHJlZCA9IChuYW1lc3BhY2Vba2V5XSBmb3Iga2V5IGluICgnR19URScsICdBREFQVF9ST1dTJywgJ3ByZWQnKSkKYXNzZXJ0IFAuZHR5cGUgPT0gcmF3X2xvZ2l0cy5kdHlwZSA9PSBucC5mbG9hdDY0IGFuZCBQLnNoYXBlID09IHJhd19sb2dpdHMuc2hhcGUgPT0gKGxlbih0ZXN0KSwgMTgpCmFzc2VydCBncm91cHMuc2hhcGUgPT0gYWRhcHRpdmUuc2hhcGUgPT0gcHJlZC5zaGFwZSA9PSAobGVuKHRlc3QpLCkKYXNzZXJ0IG5wLmlzc3ViZHR5cGUoZ3JvdXBzLmR0eXBlLCBucC5pbnRlZ2VyKSBhbmQgYWRhcHRpdmUuZHR5cGUgPT0gYm9vbApzaXplcyA9IG5wLmJpbmNvdW50KGdyb3VwcylbZ3JvdXBzXQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChhZGFwdGl2ZSwgKHNpemVzID49IDIpICYgKHNpemVzIDw9IDE4KSkKIyBPcmlnaW5hbCBjb2RlIGhhcyBhbHJlYWR5IGZpbmlzaGVkLiBFeHRyYSBJL08gb2JzZXJ2ZXMgdmFsdWVzIHdpdGhvdXQgcmVydW5uaW5nIGl0cyBnYXRlIG9yIG1vZGVscy4KY2FwdHVyZV9iYWxhbmNlZChQLCB0ZXN0LCBsYWJlbHMsIG5hbWVzcGFjZVsnRENfVEUnXSwgbmFtZXNwYWNlWydMQ19URSddLCBuYW1lc3BhY2VbJ0NMQVVERV9JRFgnXSwKICAgICAgICAgICAgICAgICBuYW1lc3BhY2VbJ0NMQVVERV9HQVRFX0xBTSddLCBuYW1lc3BhY2VbJ0NMQVVERV9MT1JBX0xBTSddKQpucC5zYXZlel9jb21wcmVzc2VkKGNhY2hlIC8gJ2Jhc2VsaW5lX29ic2VydmF0aW9uLm5weicsIGlkcz10ZXN0LmlkLnRvX251bXB5KGR0eXBlPXN0ciksCiAgICBsYWJlbHM9bnAuYXNhcnJheShsYWJlbHMsIGR0eXBlPXN0ciksIHJhdz1yYXdfbG9naXRzLCBiYWxhbmNlZD1QLCBnYXRlZF9wcmVkaWN0aW9ucz1wcmVkLAogICAgcmVjb3ZlcmVkX2dyb3VwX2lkcz1ncm91cHMsIGFkYXB0aXZlX21hc2s9YWRhcHRpdmUsIHBvb2xfaWRzPW5wLnplcm9zKGxlbih0ZXN0KSwgZHR5cGU9bnAuaW50NjQpKQpQYXRoKCcva2FnZ2xlL3dvcmtpbmcvQjU1Nl9zb3VyY2VfcGFyaXR5Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdCgKICAgIHNvdXJjZV9ub3RlYm9va19pZD0xMzkyMDAxMDAsIHNvdXJjZV92ZXJzaW9uPTIsIHNvdXJjZV9zY3JpcHRfdmVyc2lvbl9pZD0zNTkzMTMzNDgsCiAgICBvcmlnaW5hbF9zb3VyY2Vfc2hhMjU2PWhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCksIGV4ZWN1dGVkX3NvdXJjZV9zaGEyNTY9aGFzaGxpYi5zaGEyNTYobW9kaWZpZWQuZW5jb2RlKCkpLmhleGRpZ2VzdCgpLAogICAgb2JzZXJ2YXRpb25faG9va19yZXZlcnNpYmxlPVRydWUsIG9yaWdpbmFsX1dfVFdJTl9hbmRfYWxsX3NjaWVudGlmaWNfYm9kaWVzX3VuY2hhbmdlZD1UcnVlLAogICAgb3JpZ2luYWxfYmFsYW5jZV9hbHBoYT0uNSwgb3JpZ2luYWxfc3R5bGVncmFtX2xhbWJkYT0xLCBvcmlnaW5hbF9nYXRlX2xhc3Q9VHJ1ZSwKICAgIGV4cG9ydGVkX3ZhbHVlcz1bJ3VuYmFsYW5jZWRfcmF3JywgJ2JhbGFuY2VkX1AnLCAnb3JpZ2luYWxfZ2F0ZWRfbGFiZWxzJywgJ2FjdHVhbF9yZWNvdmVyZWRfZ3JvdXBzJywgJ2FjdHVhbF9hZGFwdGl2ZV9tYXNrJ10sCiAgICBxdWVyeV9wb29sX3NlbWFudGljcz0nT25lIGFjdHVhbCBjb21wbGV0ZSB0ZXN0IENTViBwb29sOyBwb29sX2lkMDsgc291cmNlIEdfVEUgaXMgcmV1c2VkIHdpdGhvdXQgcmVjbHVzdGVyaW5nJywKICAgIG9ic2VydmF0aW9uX2FycmF5c19zYXZlZF91bmRlcl90ZW1wX29ubHk9VHJ1ZSwgZXh0cmFfbGFuZ3VhZ2VfbW9kZWxfcGFzc2VzPTApLCBpbmRlbnQ9MikpCg==\",\"protected_operator.py\":\"aW1wb3J0IG51bXB5IGFzIG5wCgpkZWYgcHJvdGVjdGVkKHEscCxiYXNlbGluZSxjbGF1ZGUsbm9uY2xhdWRlLHdlaWdodD0uMTApOgogICAgcHJvdGVjdGVkID0gbnAuaXNpbihiYXNlbGluZSwgY2xhdWRlKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwocHJvdGVjdGVkLCBucC5pc2luKHEuYXJnbWF4KC0xKSwgY2xhdWRlKSkKICAgIGNhbmRpZGF0ZSA9IHEuY29weSgpCiAgICByZXZlcnRlZCA9IG5wLnplcm9zKHEuc2hhcGVbOjJdLCBkdHlwZT1ib29sKQogICAgZm9yIGNvbnRleHQgaW4gcmFuZ2UocS5zaGFwZVswXSk6CiAgICAgICAgcm93cyA9IG5wLmZsYXRub256ZXJvKH5wcm90ZWN0ZWRbY29udGV4dF0pCiAgICAgICAgb3JpZ2luYWxfbm9uID0gcVtjb250ZXh0XVtucC5peF8ocm93cywgbm9uY2xhdWRlKV0KICAgICAgICBub25fbWFzcyA9IG9yaWdpbmFsX25vbi5zdW0oMSwga2VlcGRpbXM9VHJ1ZSkKICAgICAgICBwX25vbiA9IHBbbnAuaXhfKHJvd3MsIG5vbmNsYXVkZSldCiAgICAgICAgYXNzZXJ0IChub25fbWFzcyA+IDApLmFsbCgpIGFuZCAocF9ub24uc3VtKDEpID4gMCkuYWxsKCkKICAgICAgICBjb25kaXRpb25hbCA9IG5wLnBvd2VyKG9yaWdpbmFsX25vbiAvIG5vbl9tYXNzLCAxIC0gd2VpZ2h0KSAqIG5wLnBvd2VyKHBfbm9uIC8gcF9ub24uc3VtKDEsIGtlZXBkaW1zPVRydWUpLCB3ZWlnaHQpCiAgICAgICAgYXNzZXJ0IG5wLmlzZmluaXRlKGNvbmRpdGlvbmFsKS5hbGwoKSBhbmQgKGNvbmRpdGlvbmFsLnN1bSgxKSA+IDApLmFsbCgpCiAgICAgICAgY29uZGl0aW9uYWwgLz0gY29uZGl0aW9uYWwuc3VtKDEsIGtlZXBkaW1zPVRydWUpCiAgICAgICAgY2FuZGlkYXRlW2NvbnRleHRdW25wLml4Xyhyb3dzLCBub25jbGF1ZGUpXSA9IG5vbl9tYXNzICogY29uZGl0aW9uYWwKICAgICAgICBjcm9zc2VkID0gbnAuaXNpbihjYW5kaWRhdGVbY29udGV4dCwgcm93c10uYXJnbWF4KC0xKSwgY2xhdWRlKQogICAgICAgIHJldmVydGVkW2NvbnRleHQsIHJvd3NbY3Jvc3NlZF1dID0gVHJ1ZQogICAgICAgIGNhbmRpZGF0ZVtjb250ZXh0LCByb3dzW2Nyb3NzZWRdXSA9IHFbY29udGV4dCwgcm93c1tjcm9zc2VkXV0KICAgICMgUmV0YWluIG9yaWdpbmFsIGV4cGVydC1zZWxlY3RlZCBDbGF1ZGUgbGFiZWxzOyBvbmx5IGVsaWdpYmxlIG5vbi1DbGF1ZGUgYXJnbWF4IGxhYmVscyBtYXkgY2hhbmdlLgogICAgcHJlZGljdGlvbiA9IGJhc2VsaW5lLmNvcHkoKQogICAgcHJlZGljdGlvblt+cHJvdGVjdGVkXSA9IGNhbmRpZGF0ZS5hcmdtYXgoLTEpW35wcm90ZWN0ZWRdCiAgICBhc3NlcnQgY2FuZGlkYXRlLmR0eXBlID09IG5wLmZsb2F0NjQgYW5kIG5wLmlzZmluaXRlKGNhbmRpZGF0ZSkuYWxsKCkgYW5kIChjYW5kaWRhdGUgPj0gMCkuYWxsKCkKICAgIGFzc2VydCBjYW5kaWRhdGVbOiwgOiwgY2xhdWRlXS50b2J5dGVzKCkgPT0gcVs6LCA6LCBjbGF1ZGVdLnRvYnl0ZXMoKSwgIkNsYXVkZSBjb2x1bW5zIGNoYW5nZWQgYXQgdGhlIGJpdCBsZXZlbCIKICAgIGFzc2VydCBjYW5kaWRhdGVbcHJvdGVjdGVkXS50b2J5dGVzKCkgPT0gcVtwcm90ZWN0ZWRdLnRvYnl0ZXMoKSwgIlByb3RlY3RlZCB3aG9sZSByb3cgY2hhbmdlZCIKICAgIGFzc2VydCBjYW5kaWRhdGVbcmV2ZXJ0ZWRdLnRvYnl0ZXMoKSA9PSBxW3JldmVydGVkXS50b2J5dGVzKCksICJGYW1pbHkgZmxpcCB3YXMgbm90IGZ1bGx5IHJldmVydGVkIgogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwocHJlZGljdGlvbltwcm90ZWN0ZWRdLCBiYXNlbGluZVtwcm90ZWN0ZWRdKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwobnAuaXNpbihwcmVkaWN0aW9uLCBjbGF1ZGUpLCBwcm90ZWN0ZWQpCiAgICBucC50ZXN0aW5nLmFzc2VydF9hbGxjbG9zZShjYW5kaWRhdGVbOiwgOiwgbm9uY2xhdWRlXS5zdW0oLTEpLCBxWzosIDosIG5vbmNsYXVkZV0uc3VtKC0xKSwgcnRvbD0wLCBhdG9sPTFlLTE1KQogICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UoY2FuZGlkYXRlLnN1bSgtMSksIHEuc3VtKC0xKSwgcnRvbD0wLCBhdG9sPTFlLTE1KQogICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UoY2FuZGlkYXRlLnN1bSgtMSksIDEsIHJ0b2w9MCwgYXRvbD0yZS0xMikKICAgIHJldHVybiBjYW5kaWRhdGUscHJlZGljdGlvbixyZXZlcnRlZAo=\",\"QWEN35_LICENSE\":\"DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBBcGFjaGUgTGljZW5zZQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgVmVyc2lvbiAyLjAsIEphbnVhcnkgMjAwNA0KICAgICAgICAgICAgICAgICAgICAgICAgaHR0cDovL3d3dy5hcGFjaGUub3JnL2xpY2Vuc2VzLw0KDQogICBURVJNUyBBTkQgQ09ORElUSU9OUyBGT1IgVVNFLCBSRVBST0RVQ1RJT04sIEFORCBESVNUUklCVVRJT04NCg0KICAgMS4gRGVmaW5pdGlvbnMuDQoNCiAgICAgICJMaWNlbnNlIiBzaGFsbCBtZWFuIHRoZSB0ZXJtcyBhbmQgY29uZGl0aW9ucyBmb3IgdXNlLCByZXByb2R1Y3Rpb24sDQogICAgICBhbmQgZGlzdHJpYnV0aW9uIGFzIGRlZmluZWQgYnkgU2VjdGlvbnMgMSB0aHJvdWdoIDkgb2YgdGhpcyBkb2N1bWVudC4NCg0KICAgICAgIkxpY2Vuc29yIiBzaGFsbCBtZWFuIHRoZSBjb3B5cmlnaHQgb3duZXIgb3IgZW50aXR5IGF1dGhvcml6ZWQgYnkNCiAgICAgIHRoZSBjb3B5cmlnaHQgb3duZXIgdGhhdCBpcyBncmFudGluZyB0aGUgTGljZW5zZS4NCg0KICAgICAgIkxlZ2FsIEVudGl0eSIgc2hhbGwgbWVhbiB0aGUgdW5pb24gb2YgdGhlIGFjdGluZyBlbnRpdHkgYW5kIGFsbA0KICAgICAgb3RoZXIgZW50aXRpZXMgdGhhdCBjb250cm9sLCBhcmUgY29udHJvbGxlZCBieSwgb3IgYXJlIHVuZGVyIGNvbW1vbg0KICAgICAgY29udHJvbCB3aXRoIHRoYXQgZW50aXR5LiBGb3IgdGhlIHB1cnBvc2VzIG9mIHRoaXMgZGVmaW5pdGlvbiwNCiAgICAgICJjb250cm9sIiBtZWFucyAoaSkgdGhlIHBvd2VyLCBkaXJlY3Qgb3IgaW5kaXJlY3QsIHRvIGNhdXNlIHRoZQ0KICAgICAgZGlyZWN0aW9uIG9yIG1hbmFnZW1lbnQgb2Ygc3VjaCBlbnRpdHksIHdoZXRoZXIgYnkgY29udHJhY3Qgb3INCiAgICAgIG90aGVyd2lzZSwgb3IgKGlpKSBvd25lcnNoaXAgb2YgZmlmdHkgcGVyY2VudCAoNTAlKSBvciBtb3JlIG9mIHRoZQ0KICAgICAgb3V0c3RhbmRpbmcgc2hhcmVzLCBvciAoaWlpKSBiZW5lZmljaWFsIG93bmVyc2hpcCBvZiBzdWNoIGVudGl0eS4NCg0KICAgICAgIllvdSIgKG9yICJZb3VyIikgc2hhbGwgbWVhbiBhbiBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eQ0KICAgICAgZXhlcmNpc2luZyBwZXJtaXNzaW9ucyBncmFudGVkIGJ5IHRoaXMgTGljZW5zZS4NCg0KICAgICAgIlNvdXJjZSIgZm9ybSBzaGFsbCBtZWFuIHRoZSBwcmVmZXJyZWQgZm9ybSBmb3IgbWFraW5nIG1vZGlmaWNhdGlvbnMsDQogICAgICBpbmNsdWRpbmcgYnV0IG5vdCBsaW1pdGVkIHRvIHNvZnR3YXJlIHNvdXJjZSBjb2RlLCBkb2N1bWVudGF0aW9uDQogICAgICBzb3VyY2UsIGFuZCBjb25maWd1cmF0aW9uIGZpbGVzLg0KDQogICAgICAiT2JqZWN0IiBmb3JtIHNoYWxsIG1lYW4gYW55IGZvcm0gcmVzdWx0aW5nIGZyb20gbWVjaGFuaWNhbA0KICAgICAgdHJhbnNmb3JtYXRpb24gb3IgdHJhbnNsYXRpb24gb2YgYSBTb3VyY2UgZm9ybSwgaW5jbHVkaW5nIGJ1dA0KICAgICAgbm90IGxpbWl0ZWQgdG8gY29tcGlsZWQgb2JqZWN0IGNvZGUsIGdlbmVyYXRlZCBkb2N1bWVudGF0aW9uLA0KICAgICAgYW5kIGNvbnZlcnNpb25zIHRvIG90aGVyIG1lZGlhIHR5cGVzLg0KDQogICAgICAiV29yayIgc2hhbGwgbWVhbiB0aGUgd29yayBvZiBhdXRob3JzaGlwLCB3aGV0aGVyIGluIFNvdXJjZSBvcg0KICAgICAgT2JqZWN0IGZvcm0sIG1hZGUgYXZhaWxhYmxlIHVuZGVyIHRoZSBMaWNlbnNlLCBhcyBpbmRpY2F0ZWQgYnkgYQ0KICAgICAgY29weXJpZ2h0IG5vdGljZSB0aGF0IGlzIGluY2x1ZGVkIGluIG9yIGF0dGFjaGVkIHRvIHRoZSB3b3JrDQogICAgICAoYW4gZXhhbXBsZSBpcyBwcm92aWRlZCBpbiB0aGUgQXBwZW5kaXggYmVsb3cpLg0KDQogICAgICAiRGVyaXZhdGl2ZSBXb3JrcyIgc2hhbGwgbWVhbiBhbnkgd29yaywgd2hldGhlciBpbiBTb3VyY2Ugb3IgT2JqZWN0DQogICAgICBmb3JtLCB0aGF0IGlzIGJhc2VkIG9uIChvciBkZXJpdmVkIGZyb20pIHRoZSBXb3JrIGFuZCBmb3Igd2hpY2ggdGhlDQogICAgICBlZGl0b3JpYWwgcmV2aXNpb25zLCBhbm5vdGF0aW9ucywgZWxhYm9yYXRpb25zLCBvciBvdGhlciBtb2RpZmljYXRpb25zDQogICAgICByZXByZXNlbnQsIGFzIGEgd2hvbGUsIGFuIG9yaWdpbmFsIHdvcmsgb2YgYXV0aG9yc2hpcC4gRm9yIHRoZSBwdXJwb3Nlcw0KICAgICAgb2YgdGhpcyBMaWNlbnNlLCBEZXJpdmF0aXZlIFdvcmtzIHNoYWxsIG5vdCBpbmNsdWRlIHdvcmtzIHRoYXQgcmVtYWluDQogICAgICBzZXBhcmFibGUgZnJvbSwgb3IgbWVyZWx5IGxpbmsgKG9yIGJpbmQgYnkgbmFtZSkgdG8gdGhlIGludGVyZmFjZXMgb2YsDQogICAgICB0aGUgV29yayBhbmQgRGVyaXZhdGl2ZSBXb3JrcyB0aGVyZW9mLg0KDQogICAgICAiQ29udHJpYnV0aW9uIiBzaGFsbCBtZWFuIGFueSB3b3JrIG9mIGF1dGhvcnNoaXAsIGluY2x1ZGluZw0KICAgICAgdGhlIG9yaWdpbmFsIHZlcnNpb24gb2YgdGhlIFdvcmsgYW5kIGFueSBtb2RpZmljYXRpb25zIG9yIGFkZGl0aW9ucw0KICAgICAgdG8gdGhhdCBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiwgdGhhdCBpcyBpbnRlbnRpb25hbGx5DQogICAgICBzdWJtaXR0ZWQgdG8gTGljZW5zb3IgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yayBieSB0aGUgY29weXJpZ2h0IG93bmVyDQogICAgICBvciBieSBhbiBpbmRpdmlkdWFsIG9yIExlZ2FsIEVudGl0eSBhdXRob3JpemVkIHRvIHN1Ym1pdCBvbiBiZWhhbGYgb2YNCiAgICAgIHRoZSBjb3B5cmlnaHQgb3duZXIuIEZvciB0aGUgcHVycG9zZXMgb2YgdGhpcyBkZWZpbml0aW9uLCAic3VibWl0dGVkIg0KICAgICAgbWVhbnMgYW55IGZvcm0gb2YgZWxlY3Ryb25pYywgdmVyYmFsLCBvciB3cml0dGVuIGNvbW11bmljYXRpb24gc2VudA0KICAgICAgdG8gdGhlIExpY2Vuc29yIG9yIGl0cyByZXByZXNlbnRhdGl2ZXMsIGluY2x1ZGluZyBidXQgbm90IGxpbWl0ZWQgdG8NCiAgICAgIGNvbW11bmljYXRpb24gb24gZWxlY3Ryb25pYyBtYWlsaW5nIGxpc3RzLCBzb3VyY2UgY29kZSBjb250cm9sIHN5c3RlbXMsDQogICAgICBhbmQgaXNzdWUgdHJhY2tpbmcgc3lzdGVtcyB0aGF0IGFyZSBtYW5hZ2VkIGJ5LCBvciBvbiBiZWhhbGYgb2YsIHRoZQ0KICAgICAgTGljZW5zb3IgZm9yIHRoZSBwdXJwb3NlIG9mIGRpc2N1c3NpbmcgYW5kIGltcHJvdmluZyB0aGUgV29yaywgYnV0DQogICAgICBleGNsdWRpbmcgY29tbXVuaWNhdGlvbiB0aGF0IGlzIGNvbnNwaWN1b3VzbHkgbWFya2VkIG9yIG90aGVyd2lzZQ0KICAgICAgZGVzaWduYXRlZCBpbiB3cml0aW5nIGJ5IHRoZSBjb3B5cmlnaHQgb3duZXIgYXMgIk5vdCBhIENvbnRyaWJ1dGlvbi4iDQoNCiAgICAgICJDb250cmlidXRvciIgc2hhbGwgbWVhbiBMaWNlbnNvciBhbmQgYW55IGluZGl2aWR1YWwgb3IgTGVnYWwgRW50aXR5DQogICAgICBvbiBiZWhhbGYgb2Ygd2hvbSBhIENvbnRyaWJ1dGlvbiBoYXMgYmVlbiByZWNlaXZlZCBieSBMaWNlbnNvciBhbmQNCiAgICAgIHN1YnNlcXVlbnRseSBpbmNvcnBvcmF0ZWQgd2l0aGluIHRoZSBXb3JrLg0KDQogICAyLiBHcmFudCBvZiBDb3B5cmlnaHQgTGljZW5zZS4gU3ViamVjdCB0byB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YNCiAgICAgIHRoaXMgTGljZW5zZSwgZWFjaCBDb250cmlidXRvciBoZXJlYnkgZ3JhbnRzIHRvIFlvdSBhIHBlcnBldHVhbCwNCiAgICAgIHdvcmxkd2lkZSwgbm9uLWV4Y2x1c2l2ZSwgbm8tY2hhcmdlLCByb3lhbHR5LWZyZWUsIGlycmV2b2NhYmxlDQogICAgICBjb3B5cmlnaHQgbGljZW5zZSB0byByZXByb2R1Y2UsIHByZXBhcmUgRGVyaXZhdGl2ZSBXb3JrcyBvZiwNCiAgICAgIHB1YmxpY2x5IGRpc3BsYXksIHB1YmxpY2x5IHBlcmZvcm0sIHN1YmxpY2Vuc2UsIGFuZCBkaXN0cmlidXRlIHRoZQ0KICAgICAgV29yayBhbmQgc3VjaCBEZXJpdmF0aXZlIFdvcmtzIGluIFNvdXJjZSBvciBPYmplY3QgZm9ybS4NCg0KICAgMy4gR3JhbnQgb2YgUGF0ZW50IExpY2Vuc2UuIFN1YmplY3QgdG8gdGhlIHRlcm1zIGFuZCBjb25kaXRpb25zIG9mDQogICAgICB0aGlzIExpY2Vuc2UsIGVhY2ggQ29udHJpYnV0b3IgaGVyZWJ5IGdyYW50cyB0byBZb3UgYSBwZXJwZXR1YWwsDQogICAgICB3b3JsZHdpZGUsIG5vbi1leGNsdXNpdmUsIG5vLWNoYXJnZSwgcm95YWx0eS1mcmVlLCBpcnJldm9jYWJsZQ0KICAgICAgKGV4Y2VwdCBhcyBzdGF0ZWQgaW4gdGhpcyBzZWN0aW9uKSBwYXRlbnQgbGljZW5zZSB0byBtYWtlLCBoYXZlIG1hZGUsDQogICAgICB1c2UsIG9mZmVyIHRvIHNlbGwsIHNlbGwsIGltcG9ydCwgYW5kIG90aGVyd2lzZSB0cmFuc2ZlciB0aGUgV29yaywNCiAgICAgIHdoZXJlIHN1Y2ggbGljZW5zZSBhcHBsaWVzIG9ubHkgdG8gdGhvc2UgcGF0ZW50IGNsYWltcyBsaWNlbnNhYmxlDQogICAgICBieSBzdWNoIENvbnRyaWJ1dG9yIHRoYXQgYXJlIG5lY2Vzc2FyaWx5IGluZnJpbmdlZCBieSB0aGVpcg0KICAgICAgQ29udHJpYnV0aW9uKHMpIGFsb25lIG9yIGJ5IGNvbWJpbmF0aW9uIG9mIHRoZWlyIENvbnRyaWJ1dGlvbihzKQ0KICAgICAgd2l0aCB0aGUgV29yayB0byB3aGljaCBzdWNoIENvbnRyaWJ1dGlvbihzKSB3YXMgc3VibWl0dGVkLiBJZiBZb3UNCiAgICAgIGluc3RpdHV0ZSBwYXRlbnQgbGl0aWdhdGlvbiBhZ2FpbnN0IGFueSBlbnRpdHkgKGluY2x1ZGluZyBhDQogICAgICBjcm9zcy1jbGFpbSBvciBjb3VudGVyY2xhaW0gaW4gYSBsYXdzdWl0KSBhbGxlZ2luZyB0aGF0IHRoZSBXb3JrDQogICAgICBvciBhIENvbnRyaWJ1dGlvbiBpbmNvcnBvcmF0ZWQgd2l0aGluIHRoZSBXb3JrIGNvbnN0aXR1dGVzIGRpcmVjdA0KICAgICAgb3IgY29udHJpYnV0b3J5IHBhdGVudCBpbmZyaW5nZW1lbnQsIHRoZW4gYW55IHBhdGVudCBsaWNlbnNlcw0KICAgICAgZ3JhbnRlZCB0byBZb3UgdW5kZXIgdGhpcyBMaWNlbnNlIGZvciB0aGF0IFdvcmsgc2hhbGwgdGVybWluYXRlDQogICAgICBhcyBvZiB0aGUgZGF0ZSBzdWNoIGxpdGlnYXRpb24gaXMgZmlsZWQuDQoNCiAgIDQuIFJlZGlzdHJpYnV0aW9uLiBZb3UgbWF5IHJlcHJvZHVjZSBhbmQgZGlzdHJpYnV0ZSBjb3BpZXMgb2YgdGhlDQogICAgICBXb3JrIG9yIERlcml2YXRpdmUgV29ya3MgdGhlcmVvZiBpbiBhbnkgbWVkaXVtLCB3aXRoIG9yIHdpdGhvdXQNCiAgICAgIG1vZGlmaWNhdGlvbnMsIGFuZCBpbiBTb3VyY2Ugb3IgT2JqZWN0IGZvcm0sIHByb3ZpZGVkIHRoYXQgWW91DQogICAgICBtZWV0IHRoZSBmb2xsb3dpbmcgY29uZGl0aW9uczoNCg0KICAgICAgKGEpIFlvdSBtdXN0IGdpdmUgYW55IG90aGVyIHJlY2lwaWVudHMgb2YgdGhlIFdvcmsgb3INCiAgICAgICAgICBEZXJpdmF0aXZlIFdvcmtzIGEgY29weSBvZiB0aGlzIExpY2Vuc2U7IGFuZA0KDQogICAgICAoYikgWW91IG11c3QgY2F1c2UgYW55IG1vZGlmaWVkIGZpbGVzIHRvIGNhcnJ5IHByb21pbmVudCBub3RpY2VzDQogICAgICAgICAgc3RhdGluZyB0aGF0IFlvdSBjaGFuZ2VkIHRoZSBmaWxlczsgYW5kDQoNCiAgICAgIChjKSBZb3UgbXVzdCByZXRhaW4sIGluIHRoZSBTb3VyY2UgZm9ybSBvZiBhbnkgRGVyaXZhdGl2ZSBXb3Jrcw0KICAgICAgICAgIHRoYXQgWW91IGRpc3RyaWJ1dGUsIGFsbCBjb3B5cmlnaHQsIHBhdGVudCwgdHJhZGVtYXJrLCBhbmQNCiAgICAgICAgICBhdHRyaWJ1dGlvbiBub3RpY2VzIGZyb20gdGhlIFNvdXJjZSBmb3JtIG9mIHRoZSBXb3JrLA0KICAgICAgICAgIGV4Y2x1ZGluZyB0aG9zZSBub3RpY2VzIHRoYXQgZG8gbm90IHBlcnRhaW4gdG8gYW55IHBhcnQgb2YNCiAgICAgICAgICB0aGUgRGVyaXZhdGl2ZSBXb3JrczsgYW5kDQoNCiAgICAgIChkKSBJZiB0aGUgV29yayBpbmNsdWRlcyBhICJOT1RJQ0UiIHRleHQgZmlsZSBhcyBwYXJ0IG9mIGl0cw0KICAgICAgICAgIGRpc3RyaWJ1dGlvbiwgdGhlbiBhbnkgRGVyaXZhdGl2ZSBXb3JrcyB0aGF0IFlvdSBkaXN0cmlidXRlIG11c3QNCiAgICAgICAgICBpbmNsdWRlIGEgcmVhZGFibGUgY29weSBvZiB0aGUgYXR0cmlidXRpb24gbm90aWNlcyBjb250YWluZWQNCiAgICAgICAgICB3aXRoaW4gc3VjaCBOT1RJQ0UgZmlsZSwgZXhjbHVkaW5nIHRob3NlIG5vdGljZXMgdGhhdCBkbyBub3QNCiAgICAgICAgICBwZXJ0YWluIHRvIGFueSBwYXJ0IG9mIHRoZSBEZXJpdmF0aXZlIFdvcmtzLCBpbiBhdCBsZWFzdCBvbmUNCiAgICAgICAgICBvZiB0aGUgZm9sbG93aW5nIHBsYWNlczogd2l0aGluIGEgTk9USUNFIHRleHQgZmlsZSBkaXN0cmlidXRlZA0KICAgICAgICAgIGFzIHBhcnQgb2YgdGhlIERlcml2YXRpdmUgV29ya3M7IHdpdGhpbiB0aGUgU291cmNlIGZvcm0gb3INCiAgICAgICAgICBkb2N1bWVudGF0aW9uLCBpZiBwcm92aWRlZCBhbG9uZyB3aXRoIHRoZSBEZXJpdmF0aXZlIFdvcmtzOyBvciwNCiAgICAgICAgICB3aXRoaW4gYSBkaXNwbGF5IGdlbmVyYXRlZCBieSB0aGUgRGVyaXZhdGl2ZSBXb3JrcywgaWYgYW5kDQogICAgICAgICAgd2hlcmV2ZXIgc3VjaCB0aGlyZC1wYXJ0eSBub3RpY2VzIG5vcm1hbGx5IGFwcGVhci4gVGhlIGNvbnRlbnRzDQogICAgICAgICAgb2YgdGhlIE5PVElDRSBmaWxlIGFyZSBmb3IgaW5mb3JtYXRpb25hbCBwdXJwb3NlcyBvbmx5IGFuZA0KICAgICAgICAgIGRvIG5vdCBtb2RpZnkgdGhlIExpY2Vuc2UuIFlvdSBtYXkgYWRkIFlvdXIgb3duIGF0dHJpYnV0aW9uDQogICAgICAgICAgbm90aWNlcyB3aXRoaW4gRGVyaXZhdGl2ZSBXb3JrcyB0aGF0IFlvdSBkaXN0cmlidXRlLCBhbG9uZ3NpZGUNCiAgICAgICAgICBvciBhcyBhbiBhZGRlbmR1bSB0byB0aGUgTk9USUNFIHRleHQgZnJvbSB0aGUgV29yaywgcHJvdmlkZWQNCiAgICAgICAgICB0aGF0IHN1Y2ggYWRkaXRpb25hbCBhdHRyaWJ1dGlvbiBub3RpY2VzIGNhbm5vdCBiZSBjb25zdHJ1ZWQNCiAgICAgICAgICBhcyBtb2RpZnlpbmcgdGhlIExpY2Vuc2UuDQoNCiAgICAgIFlvdSBtYXkgYWRkIFlvdXIgb3duIGNvcHlyaWdodCBzdGF0ZW1lbnQgdG8gWW91ciBtb2RpZmljYXRpb25zIGFuZA0KICAgICAgbWF5IHByb3ZpZGUgYWRkaXRpb25hbCBvciBkaWZmZXJlbnQgbGljZW5zZSB0ZXJtcyBhbmQgY29uZGl0aW9ucw0KICAgICAgZm9yIHVzZSwgcmVwcm9kdWN0aW9uLCBvciBkaXN0cmlidXRpb24gb2YgWW91ciBtb2RpZmljYXRpb25zLCBvcg0KICAgICAgZm9yIGFueSBzdWNoIERlcml2YXRpdmUgV29ya3MgYXMgYSB3aG9sZSwgcHJvdmlkZWQgWW91ciB1c2UsDQogICAgICByZXByb2R1Y3Rpb24sIGFuZCBkaXN0cmlidXRpb24gb2YgdGhlIFdvcmsgb3RoZXJ3aXNlIGNvbXBsaWVzIHdpdGgNCiAgICAgIHRoZSBjb25kaXRpb25zIHN0YXRlZCBpbiB0aGlzIExpY2Vuc2UuDQoNCiAgIDUuIFN1Ym1pc3Npb24gb2YgQ29udHJpYnV0aW9ucy4gVW5sZXNzIFlvdSBleHBsaWNpdGx5IHN0YXRlIG90aGVyd2lzZSwNCiAgICAgIGFueSBDb250cmlidXRpb24gaW50ZW50aW9uYWxseSBzdWJtaXR0ZWQgZm9yIGluY2x1c2lvbiBpbiB0aGUgV29yaw0KICAgICAgYnkgWW91IHRvIHRoZSBMaWNlbnNvciBzaGFsbCBiZSB1bmRlciB0aGUgdGVybXMgYW5kIGNvbmRpdGlvbnMgb2YNCiAgICAgIHRoaXMgTGljZW5zZSwgd2l0aG91dCBhbnkgYWRkaXRpb25hbCB0ZXJtcyBvciBjb25kaXRpb25zLg0KICAgICAgTm90d2l0aHN0YW5kaW5nIHRoZSBhYm92ZSwgbm90aGluZyBoZXJlaW4gc2hhbGwgc3VwZXJzZWRlIG9yIG1vZGlmeQ0KICAgICAgdGhlIHRlcm1zIG9mIGFueSBzZXBhcmF0ZSBsaWNlbnNlIGFncmVlbWVudCB5b3UgbWF5IGhhdmUgZXhlY3V0ZWQNCiAgICAgIHdpdGggTGljZW5zb3IgcmVnYXJkaW5nIHN1Y2ggQ29udHJpYnV0aW9ucy4NCg0KICAgNi4gVHJhZGVtYXJrcy4gVGhpcyBMaWNlbnNlIGRvZXMgbm90IGdyYW50IHBlcm1pc3Npb24gdG8gdXNlIHRoZSB0cmFkZQ0KICAgICAgbmFtZXMsIHRyYWRlbWFya3MsIHNlcnZpY2UgbWFya3MsIG9yIHByb2R1Y3QgbmFtZXMgb2YgdGhlIExpY2Vuc29yLA0KICAgICAgZXhjZXB0IGFzIHJlcXVpcmVkIGZvciByZWFzb25hYmxlIGFuZCBjdXN0b21hcnkgdXNlIGluIGRlc2NyaWJpbmcgdGhlDQogICAgICBvcmlnaW4gb2YgdGhlIFdvcmsgYW5kIHJlcHJvZHVjaW5nIHRoZSBjb250ZW50IG9mIHRoZSBOT1RJQ0UgZmlsZS4NCg0KICAgNy4gRGlzY2xhaW1lciBvZiBXYXJyYW50eS4gVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yDQogICAgICBhZ3JlZWQgdG8gaW4gd3JpdGluZywgTGljZW5zb3IgcHJvdmlkZXMgdGhlIFdvcmsgKGFuZCBlYWNoDQogICAgICBDb250cmlidXRvciBwcm92aWRlcyBpdHMgQ29udHJpYnV0aW9ucykgb24gYW4gIkFTIElTIiBCQVNJUywNCiAgICAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvcg0KICAgICAgaW1wbGllZCwgaW5jbHVkaW5nLCB3aXRob3V0IGxpbWl0YXRpb24sIGFueSB3YXJyYW50aWVzIG9yIGNvbmRpdGlvbnMNCiAgICAgIG9mIFRJVExFLCBOT04tSU5GUklOR0VNRU5ULCBNRVJDSEFOVEFCSUxJVFksIG9yIEZJVE5FU1MgRk9SIEENCiAgICAgIFBBUlRJQ1VMQVIgUFVSUE9TRS4gWW91IGFyZSBzb2xlbHkgcmVzcG9uc2libGUgZm9yIGRldGVybWluaW5nIHRoZQ0KICAgICAgYXBwcm9wcmlhdGVuZXNzIG9mIHVzaW5nIG9yIHJlZGlzdHJpYnV0aW5nIHRoZSBXb3JrIGFuZCBhc3N1bWUgYW55DQogICAgICByaXNrcyBhc3NvY2lhdGVkIHdpdGggWW91ciBleGVyY2lzZSBvZiBwZXJtaXNzaW9ucyB1bmRlciB0aGlzIExpY2Vuc2UuDQoNCiAgIDguIExpbWl0YXRpb24gb2YgTGlhYmlsaXR5LiBJbiBubyBldmVudCBhbmQgdW5kZXIgbm8gbGVnYWwgdGhlb3J5LA0KICAgICAgd2hldGhlciBpbiB0b3J0IChpbmNsdWRpbmcgbmVnbGlnZW5jZSksIGNvbnRyYWN0LCBvciBvdGhlcndpc2UsDQogICAgICB1bmxlc3MgcmVxdWlyZWQgYnkgYXBwbGljYWJsZSBsYXcgKHN1Y2ggYXMgZGVsaWJlcmF0ZSBhbmQgZ3Jvc3NseQ0KICAgICAgbmVnbGlnZW50IGFjdHMpIG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzaGFsbCBhbnkgQ29udHJpYnV0b3IgYmUNCiAgICAgIGxpYWJsZSB0byBZb3UgZm9yIGRhbWFnZXMsIGluY2x1ZGluZyBhbnkgZGlyZWN0LCBpbmRpcmVjdCwgc3BlY2lhbCwNCiAgICAgIGluY2lkZW50YWwsIG9yIGNvbnNlcXVlbnRpYWwgZGFtYWdlcyBvZiBhbnkgY2hhcmFjdGVyIGFyaXNpbmcgYXMgYQ0KICAgICAgcmVzdWx0IG9mIHRoaXMgTGljZW5zZSBvciBvdXQgb2YgdGhlIHVzZSBvciBpbmFiaWxpdHkgdG8gdXNlIHRoZQ0KICAgICAgV29yayAoaW5jbHVkaW5nIGJ1dCBub3QgbGltaXRlZCB0byBkYW1hZ2VzIGZvciBsb3NzIG9mIGdvb2R3aWxsLA0KICAgICAgd29yayBzdG9wcGFnZSwgY29tcHV0ZXIgZmFpbHVyZSBvciBtYWxmdW5jdGlvbiwgb3IgYW55IGFuZCBhbGwNCiAgICAgIG90aGVyIGNvbW1lcmNpYWwgZGFtYWdlcyBvciBsb3NzZXMpLCBldmVuIGlmIHN1Y2ggQ29udHJpYnV0b3INCiAgICAgIGhhcyBiZWVuIGFkdmlzZWQgb2YgdGhlIHBvc3NpYmlsaXR5IG9mIHN1Y2ggZGFtYWdlcy4NCg0KICAgOS4gQWNjZXB0aW5nIFdhcnJhbnR5IG9yIEFkZGl0aW9uYWwgTGlhYmlsaXR5LiBXaGlsZSByZWRpc3RyaWJ1dGluZw0KICAgICAgdGhlIFdvcmsgb3IgRGVyaXZhdGl2ZSBXb3JrcyB0aGVyZW9mLCBZb3UgbWF5IGNob29zZSB0byBvZmZlciwNCiAgICAgIGFuZCBjaGFyZ2UgYSBmZWUgZm9yLCBhY2NlcHRhbmNlIG9mIHN1cHBvcnQsIHdhcnJhbnR5LCBpbmRlbW5pdHksDQogICAgICBvciBvdGhlciBsaWFiaWxpdHkgb2JsaWdhdGlvbnMgYW5kL29yIHJpZ2h0cyBjb25zaXN0ZW50IHdpdGggdGhpcw0KICAgICAgTGljZW5zZS4gSG93ZXZlciwgaW4gYWNjZXB0aW5nIHN1Y2ggb2JsaWdhdGlvbnMsIFlvdSBtYXkgYWN0IG9ubHkNCiAgICAgIG9uIFlvdXIgb3duIGJlaGFsZiBhbmQgb24gWW91ciBzb2xlIHJlc3BvbnNpYmlsaXR5LCBub3Qgb24gYmVoYWxmDQogICAgICBvZiBhbnkgb3RoZXIgQ29udHJpYnV0b3IsIGFuZCBvbmx5IGlmIFlvdSBhZ3JlZSB0byBpbmRlbW5pZnksDQogICAgICBkZWZlbmQsIGFuZCBob2xkIGVhY2ggQ29udHJpYnV0b3IgaGFybWxlc3MgZm9yIGFueSBsaWFiaWxpdHkNCiAgICAgIGluY3VycmVkIGJ5LCBvciBjbGFpbXMgYXNzZXJ0ZWQgYWdhaW5zdCwgc3VjaCBDb250cmlidXRvciBieSByZWFzb24NCiAgICAgIG9mIHlvdXIgYWNjZXB0aW5nIGFueSBzdWNoIHdhcnJhbnR5IG9yIGFkZGl0aW9uYWwgbGlhYmlsaXR5Lg0KDQogICBFTkQgT0YgVEVSTVMgQU5EIENPTkRJVElPTlMNCg0KICAgQVBQRU5ESVg6IEhvdyB0byBhcHBseSB0aGUgQXBhY2hlIExpY2Vuc2UgdG8geW91ciB3b3JrLg0KDQogICAgICBUbyBhcHBseSB0aGUgQXBhY2hlIExpY2Vuc2UgdG8geW91ciB3b3JrLCBhdHRhY2ggdGhlIGZvbGxvd2luZw0KICAgICAgYm9pbGVycGxhdGUgbm90aWNlLCB3aXRoIHRoZSBmaWVsZHMgZW5jbG9zZWQgYnkgYnJhY2tldHMgIltdIg0KICAgICAgcmVwbGFjZWQgd2l0aCB5b3VyIG93biBpZGVudGlmeWluZyBpbmZvcm1hdGlvbi4gKERvbid0IGluY2x1ZGUNCiAgICAgIHRoZSBicmFja2V0cyEpICBUaGUgdGV4dCBzaG91bGQgYmUgZW5jbG9zZWQgaW4gdGhlIGFwcHJvcHJpYXRlDQogICAgICBjb21tZW50IHN5bnRheCBmb3IgdGhlIGZpbGUgZm9ybWF0LiBXZSBhbHNvIHJlY29tbWVuZCB0aGF0IGENCiAgICAgIGZpbGUgb3IgY2xhc3MgbmFtZSBhbmQgZGVzY3JpcHRpb24gb2YgcHVycG9zZSBiZSBpbmNsdWRlZCBvbiB0aGUNCiAgICAgIHNhbWUgInByaW50ZWQgcGFnZSIgYXMgdGhlIGNvcHlyaWdodCBub3RpY2UgZm9yIGVhc2llcg0KICAgICAgaWRlbnRpZmljYXRpb24gd2l0aGluIHRoaXJkLXBhcnR5IGFyY2hpdmVzLg0KDQogICBDb3B5cmlnaHQgMjAyNiBBbGliYWJhIENsb3VkDQoNCiAgIExpY2Vuc2VkIHVuZGVyIHRoZSBBcGFjaGUgTGljZW5zZSwgVmVyc2lvbiAyLjAgKHRoZSAiTGljZW5zZSIpOw0KICAgeW91IG1heSBub3QgdXNlIHRoaXMgZmlsZSBleGNlcHQgaW4gY29tcGxpYW5jZSB3aXRoIHRoZSBMaWNlbnNlLg0KICAgWW91IG1heSBvYnRhaW4gYSBjb3B5IG9mIHRoZSBMaWNlbnNlIGF0DQoNCiAgICAgICBodHRwOi8vd3d3LmFwYWNoZS5vcmcvbGljZW5zZXMvTElDRU5TRS0yLjANCg0KICAgVW5sZXNzIHJlcXVpcmVkIGJ5IGFwcGxpY2FibGUgbGF3IG9yIGFncmVlZCB0byBpbiB3cml0aW5nLCBzb2Z0d2FyZQ0KICAgZGlzdHJpYnV0ZWQgdW5kZXIgdGhlIExpY2Vuc2UgaXMgZGlzdHJpYnV0ZWQgb24gYW4gIkFTIElTIiBCQVNJUywNCiAgIFdJVEhPVVQgV0FSUkFOVElFUyBPUiBDT05ESVRJT05TIE9GIEFOWSBLSU5ELCBlaXRoZXIgZXhwcmVzcyBvciBpbXBsaWVkLg0KICAgU2VlIHRoZSBMaWNlbnNlIGZvciB0aGUgc3BlY2lmaWMgbGFuZ3VhZ2UgZ292ZXJuaW5nIHBlcm1pc3Npb25zIGFuZA0KICAgbGltaXRhdGlvbnMgdW5kZXIgdGhlIExpY2Vuc2Uu\",\"QWEN35_NOTICE.md\":\"UXdlbjMuNS05Qgo9PT09PT09PT09PQoKQ29weXJpZ2h0IDIwMjYgQWxpYmFiYSBDbG91ZC4gTGljZW5zZWQgdW5kZXIgQXBhY2hlIExpY2Vuc2UgMi4wOyBzZWUgUVdFTjM1X0xJQ0VOU0UuCgpPZmZpY2lhbCByZXBvc2l0b3J5OiBodHRwczovL2h1Z2dpbmdmYWNlLmNvL1F3ZW4vUXdlbjMuNS05QgpMaWNlbnNlIHNvdXJjZTogaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9Rd2VuL1F3ZW4zLjUtOUIvYmxvYi9jMjAyMjM2MjM1NzYyZTFjODcxYWQwY2NiNjBjOGVlNWJhMzM3YjlhL0xJQ0VOU0UKClRoZSBleGlzdGluZyBLYWdnbGUgaW5wdXQgYmJ1Y3hpL3F3ZW4zNS05Yi1zYWZldGVuc29ycy8xIHN1cHBsaWVzIHRoZSBtb2RlbC4gSXRzIGNvbmZpZyBhbmQgdGVuc29yIGluZGV4IG1hdGNoIHRoZSBvZmZpY2lhbCByZXBvc2l0b3J5IGF0IHRoZSByZXZpc2lvbiBhYm92ZSBieXRlLWZvci1ieXRlOyBhbGwgZm91ciB3ZWlnaHQgc2hhcmQgbmFtZXMgYW5kIGJ5dGUgbGVuZ3RocyBhZ3JlZS4gVGhpcyBhdHRyaWJ1dGlvbiByZWNvcmRzIHRoZSB1cHN0cmVhbSBsaWNlbnNlIGFuZCBkb2VzIG5vdCBjaGFuZ2UgdGhlIGlucHV0IHdlaWdodHMuCg==\",\"component.py\":\"IiIiRGlzYWJsZWQtYnktZGVmYXVsdCBwb3J0YWJsZSBTbW9sIGZpbmFsIHBsYWluIGNvbXBvbmVudDsgbm8gdHJhaW5pbmcsIHRhcmdldHMsIG9yIG9wdGltaXplci4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKZGVmIHNoYShwYXRoKToKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCJyYiIpIGFzIHN0cmVhbToKICAgICAgICByZXR1cm4gaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sICJzaGEyNTYiKS5oZXhkaWdlc3QoKQoKCmRlZiB2YWxpZGF0ZV9jb250cmFjdChzcGVjLCBhcnRpZmFjdF9yb290LCBiYXNlX3Jvb3QsIGRpYWdub3N0aWNfb25seT1GYWxzZSk6CiAgICAiIiJTdGRsaWItb25seSBmdWxsLWZpbmFsIGFkbWlzc2lvbiBhbmQgaW1tdXRhYmxlIHBvcnRhYmxlIGZpbGUgY2hlY2tzLiIiIgogICAgYXNzZXJ0IHNwZWNbImFjdHVhbF9maW5hbF9iaW5kaW5nIl0sICJTb3VyY2Ugb25seTogYWN0dWFsIGZ1bGwtZmluYWwgUUEvYmluZGluZyBtaXNzaW5nIgogICAgYmluZGluZyA9IHNwZWNbImFjdHVhbF9maW5hbF9iaW5kaW5nIl0KICAgIGFzc2VydCBiaW5kaW5nWyJhY3R1YWxfZnVsbF9RQV9wYXNzZWQiXSBpcyBUcnVlCiAgICBpZiBkaWFnbm9zdGljX29ubHk6CiAgICAgICAgYXNzZXJ0IHNwZWNbImRpYWdub3N0aWNfb25seV9hdXRob3JpemVkIl0gaXMgVHJ1ZSBhbmQgc3BlY1siaW5mZXJlbmNlX2F1dGhvcml6ZWQiXSBpcyBGYWxzZQogICAgICAgIGFzc2VydCBiaW5kaW5nWyJwYWlyZWRfZ2Fpbl9hZG1pdHRlZF9ieV9yb290Il0gaXMgRmFsc2UsICJEaWFnbm9zdGljIG11c3Qgbm90IGNsYWltIGNhbmRpZGF0ZSBhZG1pc3Npb24iCiAgICBlbHNlOgogICAgICAgIGFzc2VydCBzcGVjWyJpbmZlcmVuY2VfYXV0aG9yaXplZCJdIGlzIFRydWUgYW5kIGJpbmRpbmdbInBhaXJlZF9nYWluX2FkbWl0dGVkX2J5X3Jvb3QiXSBpcyBUcnVlCiAgICBhc3NlcnQgYmluZGluZ1sibm90ZWJvb2tfaWQiXSA9PSAxMzkzNDc0NzAgYW5kIGJpbmRpbmdbInZlcnNpb24iXSA9PSAxCiAgICBhc3NlcnQgYmluZGluZ1sic2NyaXB0X3ZlcnNpb25faWQiXSA+IDAKICAgIGFzc2VydCBiaW5kaW5nWyJ3b3JrZXJfc2hhMjU2Il0gPT0gc3BlY1sidHJhaW5pbmdfd29ya2VyX3NoYTI1NiJdCiAgICBhc3NlcnQgYmluZGluZ1siY2hlY2twb2ludF9yZWxhdGl2ZSJdID09ICJmaW5hbC9maW5pc2hlZF9jaGVja3BvaW50IgogICAgYXJ0aWZhY3Rfcm9vdCwgYmFzZV9yb290ID0gUGF0aChhcnRpZmFjdF9yb290KSwgUGF0aChiYXNlX3Jvb3QpCiAgICBjb21wbGV0ZV9wYXRoID0gYXJ0aWZhY3Rfcm9vdCAvICJmaW5hbC9DT01QTEVURS5qc29uIgogICAgcmVwb3J0X3BhdGggPSBhcnRpZmFjdF9yb290IC8gImZpbmFsL2NvbnRpbnVhdGlvbi1yZXBvcnQuanNvbiIKICAgIGFzc2VydCBzaGEoY29tcGxldGVfcGF0aCkgPT0gYmluZGluZ1siY29tcGxldGVfc2hhMjU2Il0gYW5kIHNoYShyZXBvcnRfcGF0aCkgPT0gYmluZGluZ1sicmVwb3J0X3NoYTI1NiJdCiAgICBjb21wbGV0ZSwgcmVwb3J0ID0ganNvbi5sb2Fkcyhjb21wbGV0ZV9wYXRoLnJlYWRfdGV4dCgpKSwganNvbi5sb2FkcyhyZXBvcnRfcGF0aC5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBjb21wbGV0ZVsicmVwb3J0X3NoYTI1NiJdID09IGJpbmRpbmdbInJlcG9ydF9zaGEyNTYiXQogICAgYXNzZXJ0IGNvbXBsZXRlWyJzdGF0dXMiXSA9PSByZXBvcnRbInN0YXR1cyJdID09ICJDT01QTEVURV9TTU9MX0VYQUNUX0NPTlRJTlVBVElPTiIKICAgIGFzc2VydCBjb21wbGV0ZVsiam9iIl0gPT0gcmVwb3J0WyJqb2IiXSA9PSAiZmluYWwiIGFuZCBjb21wbGV0ZVsic3VjY2Vzc2Z1bF91cGRhdGVzIl0gPT0gcmVwb3J0WyJzdWNjZXNzZnVsX3VwZGF0ZXMiXSA9PSA0NTAKICAgIGFzc2VydCByZXBvcnRbImZ1bGxfcGxhbm5lZF91cGRhdGVzIl0gPT0gNDUwIGFuZCByZXBvcnRbInJlc3VtZWRfc3VjY2Vzc2Z1bF91cGRhdGVzIl0gPT0gMjAKICAgIGFzc2VydCByZXBvcnRbIm5ld19zdWNjZXNzZnVsX3VwZGF0ZXMiXSA9PSA0MzAgYW5kIHJlcG9ydFsiY29tcGxldGVkX2Vwb2NocyJdID09IHJlcG9ydFsiZnVsbF9wbGFubmVkX2Vwb2NocyJdID09IDMKICAgIGFzc2VydCByZXBvcnRbImZpbmFsX2lzX25vdF9PT0YiXSBhbmQgcmVwb3J0WyJmaXRfcm93cyJdID09IGxpc3QocmFuZ2UoMjY5OSkpIGFuZCByZXBvcnRbInZhbGlkX3Jvd3MiXSA9PSBbXQogICAgYXNzZXJ0IGxlbihyZXBvcnRbImZpdF9pZHMiXSkgPT0gbGVuKHNldChyZXBvcnRbImZpdF9pZHMiXSkpID09IDI2OTkgYW5kIHJlcG9ydFsidmFsaWRfaWRzIl0gPT0gW10KICAgIGFzc2VydCByZXBvcnRbImNsYXNzZXMiXSA9PSBzcGVjWyJjbGFzc2VzIl0gYW5kIHJlcG9ydFsidHJhaW5fc2hhMjU2Il0gPT0gc3BlY1sidHJhaW5fc2hhMjU2Il0KICAgIGFzc2VydCByZXBvcnRbIndvcmtlcl9zaGEyNTYiXSA9PSBzcGVjWyJ0cmFpbmluZ193b3JrZXJfc2hhMjU2Il0KICAgIGFzc2VydCByZXBvcnRbImJhc2VfcmV2aXNpb24iXSA9PSBzcGVjWyJiYXNlX3JldmlzaW9uIl0gYW5kIHJlcG9ydFsiYmFzZV9tYW5pZmVzdF9zaGEyNTYiXSA9PSBzcGVjWyJiYXNlX21hbmlmZXN0X3NoYTI1NiJdCiAgICBhc3NlcnQgcmVwb3J0WyJ2ZXJzaW9ucyJdID09IHNwZWNbImNvcmVfdmVyc2lvbnMiXSBhbmQgcmVwb3J0WyJwcmV0cmFpbmVkX2Jhc2Vfb25seSJdIGFuZCBub3QgcmVwb3J0WyJxdWFudGl6YXRpb24iXQogICAgYXNzZXJ0IHJlcG9ydFsic3RyaWN0X3JlbG9hZF9yZXBsYXlfcGFzc2VkIl0gYW5kIHJlcG9ydFsiaW5mZXJlbmNlX2FydGlmYWN0X3JlYWR5Il0gYW5kIG5vdCByZXBvcnRbInZhbGlkYXRpb25fc2NvcmVkIl0KICAgIGFzc2VydCBjb21wbGV0ZVsiaW5mZXJlbmNlX2FydGlmYWN0c19yZWFkeSJdIGFuZCByZXBvcnRbImNoZWNrcG9pbnQiXSA9PSAiZmluaXNoZWRfY2hlY2twb2ludCIKICAgIGFzc2VydCByZXBvcnRbInF1ZXJ5X2FydGlmYWN0Il0gaXMgTm9uZSBhbmQgY29tcGxldGVbInF1ZXJ5X2FydGlmYWN0Il0gaXMgTm9uZQogICAgYXNzZXJ0IG1heChyZXBvcnRbInByb2JhYmlsaXR5X3JlcGxheV9tYXhhYnMiXS52YWx1ZXMoKSkgPD0gMWUtNgogICAgYXNzZXJ0IFtyZWNvcmRbInVwZGF0ZSJdIGZvciByZWNvcmQgaW4gcmVwb3J0WyJ1cGRhdGVzIl1dID09IGxpc3QocmFuZ2UoMSwgNDUxKSkKICAgIGFzc2VydCBhbGwocmVjb3JkWyJmaW5pdGVfZ3JhZGllbnRzIl0gZm9yIHJlY29yZCBpbiByZXBvcnRbInVwZGF0ZXMiXSkKICAgIGNoZWNrcG9pbnQgPSBhcnRpZmFjdF9yb290IC8gYmluZGluZ1siY2hlY2twb2ludF9yZWxhdGl2ZSJdCiAgICByZXF1aXJlZCA9IHsiYWRhcHRlci9hZGFwdGVyX21vZGVsLnNhZmV0ZW5zb3JzIiwgImFkYXB0ZXIvYWRhcHRlcl9jb25maWcuanNvbiIsICJoZWFkcy5zYWZldGVuc29ycyIsCiAgICAgICAgICAgICAgICAicmVhZG91dHMubnB6IiwgInJlcGxheV9hbmNob3JzLm5weiJ9CiAgICBhc3NlcnQgcmVxdWlyZWQgPD0gc2V0KGJpbmRpbmdbInBvcnRhYmxlX2ZpbGVzX3NoYTI1NiJdKQogICAgZm9yIHJlbGF0aXZlLCBkaWdlc3QgaW4gYmluZGluZ1sicG9ydGFibGVfZmlsZXNfc2hhMjU2Il0uaXRlbXMoKToKICAgICAgICBwYXRoID0gY2hlY2twb2ludCAvIHJlbGF0aXZlCiAgICAgICAgYXNzZXJ0IHBhdGgucmVzb2x2ZSgpLmlzX3JlbGF0aXZlX3RvKGNoZWNrcG9pbnQucmVzb2x2ZSgpKQogICAgICAgIGFzc2VydCByZWxhdGl2ZSAhPSAiY29udGludWF0aW9uX3N0YXRlLnB0IiwgIk9wdGltaXplciBzdGF0ZSBpcyB1bm5lY2Vzc2FyeSBmb3IgaW5mZXJlbmNlIgogICAgICAgIGFzc2VydCBkaWdlc3QgPT0gY29tcGxldGVbImNoZWNrcG9pbnRfZmlsZXNfc2hhMjU2Il1bcmVsYXRpdmVdIGFuZCBzaGEocGF0aCkgPT0gZGlnZXN0CiAgICBjb25maWcgPSBqc29uLmxvYWRzKChjaGVja3BvaW50IC8gImFkYXB0ZXIvYWRhcHRlcl9jb25maWcuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgYXNzZXJ0IGNvbmZpZ1siciJdID09IDE2IGFuZCBjb25maWdbImxvcmFfYWxwaGEiXSA9PSAzMiBhbmQgY29uZmlnWyJsb3JhX2Ryb3BvdXQiXSA9PSAuMDUKICAgIGFzc2VydCBjb25maWdbImJpYXMiXSA9PSAibm9uZSIgYW5kIGNvbmZpZ1sicGVmdF90eXBlIl0gPT0gIkxPUkEiCiAgICBzdGFnZV9wYXRoID0gYmFzZV9yb290IC8gInNtb2xsbTNfc3RhZ2luZ19jb21wbGV0ZS5qc29uIgogICAgYXNzZXJ0IHNoYShzdGFnZV9wYXRoKSA9PSBzcGVjWyJiYXNlX21hbmlmZXN0X3NoYTI1NiJdCiAgICBzdGFnZSA9IGpzb24ubG9hZHMoc3RhZ2VfcGF0aC5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBzdGFnZVsic3RhdHVzIl0gPT0gIkNPTVBMRVRFX1BJTk5FRF9QVUJMSUNfQkFTRV9PTkxZIiBhbmQgc3RhZ2VbInJldmlzaW9uIl0gPT0gc3BlY1siYmFzZV9yZXZpc2lvbiJdCiAgICBhc3NlcnQgc3RhZ2VbImxpY2Vuc2UiXSA9PSAiQXBhY2hlLTIuMCIgYW5kIG5vdCBzdGFnZVsiY29tcGV0aXRpb25fZGF0YSJdIGFuZCBub3Qgc3RhZ2VbIm1vZGVsX2luaXRpYWxpemVkIl0KICAgIGJhc2VfZGlyID0gYmFzZV9yb290IC8gInNtb2xsbTMtYmFzZSIKICAgIGFzc2VydCBsZW4oc3RhZ2VbImZpbGVzIl0pID09IDkKICAgIGZvciByZWNvcmQgaW4gc3RhZ2VbImZpbGVzIl06CiAgICAgICAgYXNzZXJ0IHNoYShiYXNlX2RpciAvIHJlY29yZFsibmFtZSJdKSA9PSByZWNvcmRbInNoYTI1NiJdCiAgICAgICAgYXNzZXJ0IChiYXNlX2RpciAvIHJlY29yZFsibmFtZSJdKS5zdGF0KCkuc3Rfc2l6ZSA9PSBzcGVjWyJiYXNlX2ZpbGVzIl1bcmVjb3JkWyJuYW1lIl1dWyJzaXplIl0KICAgICAgICBwaW5uZWQgPSBzcGVjWyJiYXNlX2ZpbGVzIl1bcmVjb3JkWyJuYW1lIl1dLmdldCgibGZzX3NoYTI1NiIpCiAgICAgICAgYXNzZXJ0IG5vdCBwaW5uZWQgb3IgcGlubmVkID09IHJlY29yZFsic2hhMjU2Il0KICAgIHJldHVybiBjaGVja3BvaW50LCBiYXNlX2RpciwgcmVwb3J0CgoKY2xhc3MgU21vbEZpbmFsUGxhaW46CiAgICBkZWYgX19pbml0X18oc2VsZiwgc3BlY19wYXRoLCBhcnRpZmFjdF9yb290LCBiYXNlX3Jvb3QsIGFuY2hvcl90cmFpbl9jc3YsIGRpYWdub3N0aWNfb25seT1GYWxzZSk6CiAgICAgICAgc2VsZi5zcGVjID0ganNvbi5sb2FkcyhQYXRoKHNwZWNfcGF0aCkucmVhZF90ZXh0KCkpCiAgICAgICAgZm9yIG5hbWUsIGRpZ2VzdCBpbiBzZWxmLnNwZWNbInNvdXJjZV9oYXNoZXMiXS5pdGVtcygpOgogICAgICAgICAgICBhc3NlcnQgc2hhKFBhdGgoX19maWxlX18pLnBhcmVudCAvIG5hbWUpID09IGRpZ2VzdAogICAgICAgIGNoZWNrcG9pbnQsIGJhc2VfZGlyLCByZXBvcnQgPSB2YWxpZGF0ZV9jb250cmFjdChzZWxmLnNwZWMsIGFydGlmYWN0X3Jvb3QsIGJhc2Vfcm9vdCwgZGlhZ25vc3RpY19vbmx5PWRpYWdub3N0aWNfb25seSkKICAgICAgICBhc3NlcnQgb3MubmFtZSAhPSAibnQiIGFuZCBQYXRoKCIva2FnZ2xlL2lucHV0IikuaXNfZGlyKCksICJSZW1vdGUtb25seSBjb21wb25lbnQiCiAgICAgICAgYXNzZXJ0IHNoYShhbmNob3JfdHJhaW5fY3N2KSA9PSBzZWxmLnNwZWNbInRyYWluX3NoYTI1NiJdCiAgICAgICAgb3MuZW52aXJvblsiSEZfSFVCX09GRkxJTkUiXSA9ICIxIgogICAgICAgIG9zLmVudmlyb25bIlRSQU5TRk9STUVSU19PRkZMSU5FIl0gPSAiMSIKICAgICAgICBvcy5lbnZpcm9uWyJUT0tFTklaRVJTX1BBUkFMTEVMSVNNIl0gPSAiZmFsc2UiCiAgICAgICAgaW1wb3J0IGltcG9ydGxpYi51dGlsCiAgICAgICAgZmluZGVyID0gaW1wb3J0bGliLnV0aWwuZmluZF9zcGVjCiAgICAgICAgaW1wb3J0bGliLnV0aWwuZmluZF9zcGVjID0gbGFtYmRhIG5hbWUsICphLCAqKmt3OiBOb25lIGlmIG5hbWUuc3BsaXQoIi4iKVswXSA9PSAidG9yY2hhbyIgZWxzZSBmaW5kZXIobmFtZSwgKmEsICoqa3cpCiAgICAgICAgIyBLZWVwIHRoZSBzdWNjZXNzZnVsIEZVTEwgd29ya2VyJ3MgdG9yY2hhbyBtYXNrIGFjdGl2ZSB0aHJvdWdoIFBFRlQgZGlzcGF0Y2guCiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSB0b3JjaCBpbXBvcnQgbm4KICAgICAgICBmcm9tIHBlZnQgaW1wb3J0IFBlZnRNb2RlbAogICAgICAgIGltcG9ydCBwZWZ0LmltcG9ydF91dGlscyBhcyBwZWZ0X2ltcG9ydHMKICAgICAgICBwZWZ0X2ltcG9ydHMuaXNfdG9yY2hhb19hdmFpbGFibGUgPSBsYW1iZGEgKmEsICoqa3c6IEZhbHNlCiAgICAgICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgICAgIGZyb20gc2FmZXRlbnNvcnMudG9yY2ggaW1wb3J0IGxvYWRfZmlsZQogICAgICAgIGZyb20gc2NpcHkuc3BlY2lhbCBpbXBvcnQgbG9nX3NvZnRtYXgsIHNvZnRtYXgKICAgICAgICBmcm9tIG93bmVkX3BhdGggaW1wb3J0IGxvYWRfb3duZWRfYmFzZSwgbWFrZV9yZXByZXNlbnRhdGlvbiwgdG9rZW5pemVfb3duZWQKCiAgICAgICAgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApID09ICg3LCA1KQogICAgICAgIGFzc2VydCB7bmFtZTogaW1wb3J0bGliLm1ldGFkYXRhLnZlcnNpb24obmFtZSkgZm9yIG5hbWUgaW4gc2VsZi5zcGVjWyJjb3JlX3ZlcnNpb25zIl19ID09IHNlbGYuc3BlY1siY29yZV92ZXJzaW9ucyJdCiAgICAgICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpCiAgICAgICAgbG9hZGluZyA9IFtdCiAgICAgICAgYmFzZSA9IGxvYWRfb3duZWRfYmFzZShiYXNlX2RpciwgbG9hZGluZywgbGFtYmRhOiBOb25lKQogICAgICAgIHNlbGYubW9kZWwgPSBQZWZ0TW9kZWwuZnJvbV9wcmV0cmFpbmVkKGJhc2UsIGNoZWNrcG9pbnQgLyAiYWRhcHRlciIsIGlzX3RyYWluYWJsZT1GYWxzZSwgbG9jYWxfZmlsZXNfb25seT1UcnVlKS5ldmFsKCkKICAgICAgICBhc3NlcnQgbm90IGFueShwYXJhbWV0ZXIucmVxdWlyZXNfZ3JhZCBmb3IgcGFyYW1ldGVyIGluIHNlbGYubW9kZWwucGFyYW1ldGVycygpKQogICAgICAgIGFzc2VydCBhbGwocGFyYW1ldGVyLmR0eXBlID09IHRvcmNoLmZsb2F0MzIgZm9yIG5hbWUsIHBhcmFtZXRlciBpbiBzZWxmLm1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKSBpZiAibG9yYV8iIGluIG5hbWUpCiAgICAgICAgc2VsZi5sYXllcnMgPSBbNiwgMTIsIDE4LCAyNCwgMzAsIDM2XQogICAgICAgIHNlbGYubm9ybXMgPSBubi5Nb2R1bGVMaXN0KFtubi5MYXllck5vcm0oMjA0OCkgZm9yIF8gaW4gc2VsZi5sYXllcnNdKS5jdWRhKCkuZmxvYXQoKQogICAgICAgIHNhdmVkID0gbG9hZF9maWxlKGNoZWNrcG9pbnQgLyAiaGVhZHMuc2FmZXRlbnNvcnMiKQogICAgICAgIGFzc2VydCBhbGwodmFsdWUuZHR5cGUgPT0gdG9yY2guZmxvYXQzMiBhbmQgYm9vbCh0b3JjaC5pc2Zpbml0ZSh2YWx1ZSkuYWxsKCkpIGZvciB2YWx1ZSBpbiBzYXZlZC52YWx1ZXMoKSkKICAgICAgICBzZWxmLm5vcm1zLmxvYWRfc3RhdGVfZGljdCh7a2V5LnJlbW92ZXByZWZpeCgibm9ybXMuIik6IHZhbHVlIGZvciBrZXksIHZhbHVlIGluIHNhdmVkLml0ZW1zKCkgaWYga2V5LnN0YXJ0c3dpdGgoIm5vcm1zLiIpfSwgc3RyaWN0PVRydWUpCiAgICAgICAgc2VsZi5ub3Jtcy5ldmFsKCkKICAgICAgICBzZWxmLm5vcm1zLnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIHdpdGggbnAubG9hZChjaGVja3BvaW50IC8gInJlYWRvdXRzLm5weiIsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgcmVhZG91dHM6CiAgICAgICAgICAgIHNlbGYucmVhZG91dHMgPSB7a2V5OiByZWFkb3V0c1trZXldLmNvcHkoKSBmb3Iga2V5IGluIHJlYWRvdXRzLmZpbGVzfQogICAgICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHNlbGYucmVhZG91dHNbImNsYXNzZXMiXSwgbnAuYXNhcnJheShzZWxmLnNwZWNbImNsYXNzZXMiXSwgZHR5cGU9c3RyKSkKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChzZWxmLnJlYWRvdXRzWyJwb29sX2xheWVycyJdLCBzZWxmLmxheWVycykKICAgICAgICBhc3NlcnQgc2VsZi5yZWFkb3V0c1sicGxhaW5fY29lZiJdLnNoYXBlID09IHNlbGYucmVhZG91dHNbImNlbnRlcl9jb2VmIl0uc2hhcGUgPT0gKDE4LCAxMjI4OCkKICAgICAgICBhc3NlcnQgc2VsZi5yZWFkb3V0c1sicGxhaW5faW50ZXJjZXB0Il0uc2hhcGUgPT0gc2VsZi5yZWFkb3V0c1siY2VudGVyX2ludGVyY2VwdCJdLnNoYXBlID09ICgxOCwpCiAgICAgICAgZm9yIHJvdXRlLCBwcmVmaXggaW4gKCgicGxhaW4iLCAicGxhaW4iKSwgKCJjZW50ZXIiLCAiY2VudGVyZWQiKSk6CiAgICAgICAgICAgIGZvciBrZXksIHN0YXRlX2tleSBpbiAoKCJjb2VmIiwgIndlaWdodCIpLCAoImludGVyY2VwdCIsICJiaWFzIikpOgogICAgICAgICAgICAgICAgYXJyYXkgPSBzZWxmLnJlYWRvdXRzW3JvdXRlICsgIl8iICsga2V5XQogICAgICAgICAgICAgICAgYXNzZXJ0IGFycmF5LmR0eXBlID09IG5wLmZsb2F0MzIgYW5kIG5wLmlzZmluaXRlKGFycmF5KS5hbGwoKQogICAgICAgICAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoYXJyYXksIHNhdmVkW3ByZWZpeCArICIuIiArIHN0YXRlX2tleV0ubnVtcHkoKSkKICAgICAgICAjIFJlYWQgb25seSBhbmNob3IgSURzL3RleHQsIG5ldmVyIGEgdGFyZ2V0L21vZGVsIGNvbHVtbi4gQjMgYW5kIHBhZGRpbmcgbWF0Y2ggdGhlIHNhdmVkIHJlcGxheS4KICAgICAgICBhbmNob3JfZGF0YSA9IHBkLnJlYWRfY3N2KGFuY2hvcl90cmFpbl9jc3YsIHVzZWNvbHM9WyJpZCIsICJ0ZXh0Il0sIGR0eXBlPXN0cikKICAgICAgICBhc3NlcnQgbGVuKGFuY2hvcl9kYXRhKSA9PSAyNjk5IGFuZCBhbmNob3JfZGF0YS5pZC50b2xpc3QoKSA9PSByZXBvcnRbImZpdF9pZHMiXQogICAgICAgIHNlbGYuX2RpYWdub3N0aWNfb25seSA9IGRpYWdub3N0aWNfb25seQogICAgICAgIHNlbGYuX2RpYWdub3N0aWNfaWRzID0gYW5jaG9yX2RhdGEuaWQuaWxvY1s6NV0udG9saXN0KCkKICAgICAgICBzZWxmLl9kaWFnbm9zdGljX3RleHRzID0gYW5jaG9yX2RhdGEudGV4dC5pbG9jWzo1XS50b2xpc3QoKQogICAgICAgIHNlbGYudG9rZW5pemVyLCB0b2tlbnMgPSB0b2tlbml6ZV9vd25lZChiYXNlX2RpciwgYW5jaG9yX2RhdGEudGV4dC5pbG9jWzozXS50b2xpc3QoKSkKICAgICAgICBzZWxmLl90b3JjaCwgc2VsZi5fbnAgPSB0b3JjaCwgbnAKICAgICAgICBzZWxmLl9tYWtlX3JlcHJlc2VudGF0aW9uLCBzZWxmLl90b2tlbml6ZV9vd25lZCA9IG1ha2VfcmVwcmVzZW50YXRpb24sIHRva2VuaXplX293bmVkCiAgICAgICAgc2VsZi5fbG9nX3NvZnRtYXggPSBsb2dfc29mdG1heAogICAgICAgIHdpdGggbnAubG9hZChjaGVja3BvaW50IC8gInJlcGxheV9hbmNob3JzLm5weiIsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgYW5jaG9yczoKICAgICAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoYW5jaG9yc1sicm93cyJdLCBbMCwgMSwgMl0pCiAgICAgICAgICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKGFuY2hvcnNbImlkcyJdLCBhbmNob3JfZGF0YS5pZC5pbG9jWzozXS50b19udW1weShkdHlwZT1zdHIpKQogICAgICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChhbmNob3JzWyJjbGFzc2VzIl0sIHNlbGYucmVhZG91dHNbImNsYXNzZXMiXSkKICAgICAgICAgICAgZXhwZWN0ZWQsIHJlY29yZGVkX3JlbG9hZGVkID0gYW5jaG9yc1siZmVhdHVyZXMiXS5jb3B5KCksIGFuY2hvcnNbInJlbG9hZGVkX2ZlYXR1cmVzIl0uY29weSgpCiAgICAgICAgZnJlc2ggPSBzZWxmLl9mZWF0dXJlcyhhbmNob3JfZGF0YS50ZXh0Lmlsb2NbOjNdLnRvbGlzdCgpLCBzb3J0X2J5X2xlbmd0aD1GYWxzZSkKICAgICAgICBhc3NlcnQgZXhwZWN0ZWQuZHR5cGUgPT0gZnJlc2guZHR5cGUgPT0gbnAuZmxvYXQzMiBhbmQgbnAuaXNmaW5pdGUoZXhwZWN0ZWQpLmFsbCgpCiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UoZnJlc2gsIGV4cGVjdGVkLCBydG9sPTFlLTUsIGF0b2w9MWUtNSkKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hbGxjbG9zZShmcmVzaCwgcmVjb3JkZWRfcmVsb2FkZWQsIHJ0b2w9MWUtNSwgYXRvbD0xZS01KQogICAgICAgIHBhcml0eSA9IHt9CiAgICAgICAgZm9yIHJvdXRlIGluICgicGxhaW4iLCAiY2VudGVyIik6CiAgICAgICAgICAgIGEgPSBleHBlY3RlZCBpZiByb3V0ZSA9PSAicGxhaW4iIGVsc2UgZXhwZWN0ZWQgLSBleHBlY3RlZC5tZWFuKDApCiAgICAgICAgICAgIGIgPSBmcmVzaCBpZiByb3V0ZSA9PSAicGxhaW4iIGVsc2UgZnJlc2ggLSBmcmVzaC5tZWFuKDApCiAgICAgICAgICAgIGNvZWYsIGludGVyY2VwdCA9IHNlbGYucmVhZG91dHNbcm91dGUgKyAiX2NvZWYiXSwgc2VsZi5yZWFkb3V0c1tyb3V0ZSArICJfaW50ZXJjZXB0Il0KICAgICAgICAgICAgYmVmb3JlID0gc29mdG1heChsb2dfc29mdG1heChhIEAgY29lZi5UICsgaW50ZXJjZXB0LCBheGlzPTEpLCBheGlzPTEpCiAgICAgICAgICAgIGFmdGVyID0gc29mdG1heChsb2dfc29mdG1heChiIEAgY29lZi5UICsgaW50ZXJjZXB0LCBheGlzPTEpLCBheGlzPTEpCiAgICAgICAgICAgIHBhcml0eVtyb3V0ZV0gPSBmbG9hdChucC5tYXgobnAuYWJzKGJlZm9yZSAtIGFmdGVyKSkpCiAgICAgICAgICAgIGFzc2VydCBwYXJpdHlbcm91dGVdIDw9IDFlLTYKICAgICAgICBzZWxmLmFuY2hvcl9yZXBsYXkgPSB7ImZlYXR1cmVfbWF4YWJzIjogZmxvYXQobnAubWF4KG5wLmFicyhmcmVzaCAtIGV4cGVjdGVkKSkpLCAicHJvYmFiaWxpdHlfbWF4YWJzIjogcGFyaXR5LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZmVhdHVyZV9ydG9sIjogMWUtNSwgImZlYXR1cmVfYXRvbCI6IDFlLTUsICJwcm9iYWJpbGl0eV9hdG9sIjogMWUtNn0KCiAgICBkZWYgX2ZlYXR1cmVzKHNlbGYsIHRleHRzLCBzb3J0X2J5X2xlbmd0aD1UcnVlKToKICAgICAgICB0b3JjaCwgbnAgPSBzZWxmLl90b3JjaCwgc2VsZi5fbnAKICAgICAgICB0b2tlbml6ZXIsIHRva2VucyA9IHNlbGYuX3Rva2VuaXplX293bmVkKE5vbmUsIHRleHRzLCB0b2tlbml6ZXI9c2VsZi50b2tlbml6ZXIpCiAgICAgICAgcmVwcmVzZW50YXRpb24gPSBzZWxmLl9tYWtlX3JlcHJlc2VudGF0aW9uKHNlbGYubW9kZWwsIHRva2VuaXplciwgdG9rZW5zLCBzZWxmLmxheWVycywgc2VsZi5ub3JtcykKICAgICAgICBmZWF0dXJlcyA9IG5wLmVtcHR5KChsZW4odG9rZW5zKSwgMTIyODgpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIG9yZGVyID0gbnAuYXJnc29ydChucC5hc2FycmF5KFtsZW4ocm93KSBmb3Igcm93IGluIHRva2Vuc10pLCBraW5kPSJzdGFibGUiKSBpZiBzb3J0X2J5X2xlbmd0aCBlbHNlIG5wLmFyYW5nZShsZW4odG9rZW5zKSkKICAgICAgICB3aXRoIHRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgIGZvciBzdGFydCBpbiByYW5nZSgwLCBsZW4ob3JkZXIpLCAzKToKICAgICAgICAgICAgICAgIHBvc2l0aW9ucyA9IG9yZGVyW3N0YXJ0OnN0YXJ0ICsgM10KICAgICAgICAgICAgICAgIGZlYXR1cmVzW3Bvc2l0aW9uc10gPSByZXByZXNlbnRhdGlvbihwb3NpdGlvbnMpWzBdLmNwdSgpLm51bXB5KCkKICAgICAgICBhc3NlcnQgbnAuaXNmaW5pdGUoZmVhdHVyZXMpLmFsbCgpCiAgICAgICAgcmV0dXJuIGZlYXR1cmVzCgogICAgZGVmIHByZWRpY3RfcGxhaW5fbG9ncChzZWxmLCBpZHMsIHRleHRzKToKICAgICAgICBucCA9IHNlbGYuX25wCiAgICAgICAgaWRzID0gbnAuYXNhcnJheShpZHMsIGR0eXBlPXN0cikKICAgICAgICBhc3NlcnQgaWRzLm5kaW0gPT0gMSBhbmQgbGVuKGlkcykgPT0gbGVuKHRleHRzKSA+IDAgYW5kIGxlbihucC51bmlxdWUoaWRzKSkgPT0gbGVuKGlkcykKICAgICAgICBpZiBzZWxmLl9kaWFnbm9zdGljX29ubHk6CiAgICAgICAgICAgIGFzc2VydCBsZW4oaWRzKSBpbiAoMywgNSkgYW5kIGlkcy50b2xpc3QoKSA9PSBzZWxmLl9kaWFnbm9zdGljX2lkc1s6bGVuKGlkcyldCiAgICAgICAgICAgIGFzc2VydCBsaXN0KHRleHRzKSA9PSBzZWxmLl9kaWFnbm9zdGljX3RleHRzWzpsZW4oaWRzKV0sICJPbmx5IGV4YWN0IG93bmVkMy81LXJvdyBkaWFnbm9zdGljIGZpeHR1cmVzIGFkbWl0dGVkIgogICAgICAgIGZlYXR1cmVzID0gc2VsZi5fZmVhdHVyZXModGV4dHMpCiAgICAgICAgbG9ncCA9IHNlbGYuX2xvZ19zb2Z0bWF4KGZlYXR1cmVzIEAgc2VsZi5yZWFkb3V0c1sicGxhaW5fY29lZiJdLlQgKyBzZWxmLnJlYWRvdXRzWyJwbGFpbl9pbnRlcmNlcHQiXSwgYXhpcz0xKQogICAgICAgIGFzc2VydCBsb2dwLnNoYXBlID09IChsZW4oaWRzKSwgMTgpIGFuZCBsb2dwLmR0eXBlID09IG5wLmZsb2F0MzIgYW5kIG5wLmlzZmluaXRlKGxvZ3ApLmFsbCgpCiAgICAgICAgcmV0dXJuIHsiaWRzIjogaWRzLCAiY2xhc3NlcyI6IHNlbGYucmVhZG91dHNbImNsYXNzZXMiXS5jb3B5KCksICJwbGFpbl9sb2dwIjogbG9ncH0K\",\"owned_path.py\":\"IiIiTWluaW1hbCBieXRlLXByZXNlcnZlZCBzY2llbnRpZmljIGZ1bmN0aW9ucyBleHRyYWN0ZWQgZnJvbSB0aGUgcmV2aWV3ZWQgb3duZWQgZnVsbC1maXQgcGF0aC4iIiIKCgpkZWYgbG9hZF9vd25lZF9iYXNlKGJhc2VfZGlyLCBsb2FkaW5nLCBkZWFkbGluZSk6CiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwKICAgIGRlZiBsb2FkX2Jhc2UoKToKICAgICAgICBkZWFkbGluZSgpCiAgICAgICAgYmFzZSwgaW5mbyA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoYmFzZV9kaXIsIGR0eXBlPXRvcmNoLmZsb2F0MTYsIGRldmljZV9tYXA9eyIiOiAwfSwKICAgICAgICAgICAgYXR0bl9pbXBsZW1lbnRhdGlvbj0ic2RwYSIsIHRydXN0X3JlbW90ZV9jb2RlPUZhbHNlLCBsb2NhbF9maWxlc19vbmx5PVRydWUsIG91dHB1dF9sb2FkaW5nX2luZm89VHJ1ZSkKICAgICAgICBhc3NlcnQgbm90IGluZm8uZ2V0KCJtaXNzaW5nX2tleXMiKSBhbmQgbm90IGluZm8uZ2V0KCJtaXNtYXRjaGVkX2tleXMiKSBhbmQgbm90IGluZm8uZ2V0KCJlcnJvcl9tc2dzIiksIGluZm8KICAgICAgICBhc3NlcnQgc2V0KGluZm8uZ2V0KCJ1bmV4cGVjdGVkX2tleXMiLCBbXSkpIDw9IHsibG1faGVhZC53ZWlnaHQifSwgaW5mbwogICAgICAgIGFzc2VydCBsZW4oYmFzZS5sYXllcnMpID09IDM2IGFuZCBub3QgaGFzYXR0cihiYXNlLCAibG1faGVhZCIpCiAgICAgICAgYXNzZXJ0IGFsbChwYXJhbWV0ZXIuZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGFuZCBwYXJhbWV0ZXIuZGV2aWNlLmluZGV4ID09IDAgZm9yIHBhcmFtZXRlciBpbiBiYXNlLnBhcmFtZXRlcnMoKSkKICAgICAgICBiYXNlLnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIGxvYWRpbmcuYXBwZW5kKGluZm8pCiAgICAgICAgcmV0dXJuIGJhc2UKICAgIHJldHVybiBsb2FkX2Jhc2UoKQoKCmRlZiBtYWtlX3JlcHJlc2VudGF0aW9uKG1vZGVsLCB0b2tlbml6ZXIsIHRva2VucywgbGF5ZXJzLCBub3Jtcyk6CiAgICBpbXBvcnQgdG9yY2gKICAgIGRlZiByZXByZXNlbnRhdGlvbihyb3dzKToKICAgICAgICBiYXRjaCA9IHRva2VuaXplci5wYWQoeyJpbnB1dF9pZHMiOiBbdG9rZW5zW2ldIGZvciBpIGluIHJvd3NdfSwgcmV0dXJuX3RlbnNvcnM9InB0IikudG8oImN1ZGE6MCIpCiAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgiY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICBoaWRkZW4gPSBtb2RlbCgqKmJhdGNoLCBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlLCB1c2VfY2FjaGU9RmFsc2UpLmhpZGRlbl9zdGF0ZXMKICAgICAgICBhc3NlcnQgbGVuKGhpZGRlbikgPT0gMzcKICAgICAgICBwaWVjZXMgPSBbXQogICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoImN1ZGEiLCBlbmFibGVkPUZhbHNlKToKICAgICAgICAgICAgZm9yIGxheWVyLCBub3JtIGluIHppcChsYXllcnMsIG5vcm1zKToKICAgICAgICAgICAgICAgIG1hc2sgPSBiYXRjaC5hdHRlbnRpb25fbWFzay51bnNxdWVlemUoLTEpLmZsb2F0KCkKICAgICAgICAgICAgICAgIG1hc2tbOiwgMF0gPSAwCiAgICAgICAgICAgICAgICBhc3NlcnQgYm9vbCgobWFzay5zdW0oMSkgPiAwKS5hbGwoKSkKICAgICAgICAgICAgICAgIG1lYW4gPSAoaGlkZGVuW2xheWVyXS5mbG9hdCgpICogbWFzaykuc3VtKDEpIC8gbWFzay5zdW0oMSkKICAgICAgICAgICAgICAgIHBpZWNlcy5hcHBlbmQobm9ybShtZWFuKSkKICAgICAgICAgICAgeiA9IHRvcmNoLmNhdChwaWVjZXMsIC0xKQogICAgICAgIGFzc2VydCBib29sKHRvcmNoLmlzZmluaXRlKHopLmFsbCgpKQogICAgICAgIHJldHVybiB6LCBpbnQoYmF0Y2guYXR0ZW50aW9uX21hc2suc3VtKCkpLCBpbnQoYmF0Y2guaW5wdXRfaWRzLm51bWVsKCkpCiAgICByZXR1cm4gcmVwcmVzZW50YXRpb24KCgpkZWYgdG9rZW5pemVfb3duZWQoYmFzZV9kaXIsIHRleHRzLCB0b2tlbml6ZXI9Tm9uZSk6CiAgICBpZiB0b2tlbml6ZXIgaXMgTm9uZToKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b1Rva2VuaXplcgogICAgICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGJhc2VfZGlyLCBsb2NhbF9maWxlc19vbmx5PVRydWUsIHRydXN0X3JlbW90ZV9jb2RlPUZhbHNlKQogICAgYXNzZXJ0IHRva2VuaXplci5ib3NfdG9rZW5faWQgaXMgTm9uZSBhbmQgdG9rZW5pemVyLmVvc190b2tlbl9pZCA9PSAxMjgwMDEKICAgIHRva2VuaXplci5wYWRkaW5nX3NpZGUgPSAicmlnaHQiCiAgICB0b2tlbml6ZXIucGFkX3Rva2VuX2lkID0gMTI4MDA0CiAgICB0b2tlbnMgPSB0b2tlbml6ZXIodGV4dHMsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSwgdHJ1bmNhdGlvbj1UcnVlLCBtYXhfbGVuZ3RoPTQ0NylbImlucHV0X2lkcyJdCiAgICB0b2tlbnMgPSBbcm93ICsgWzEyODAwMV0gZm9yIHJvdyBpbiB0b2tlbnNdCiAgICBhc3NlcnQgYWxsKDIgPD0gbGVuKHJvdykgPD0gNDQ4IGZvciByb3cgaW4gdG9rZW5zKQogICAgcmV0dXJuIHRva2VuaXplciwgdG9rZW5z\",\"run_component.py\":\"IiIiUmVtb3RlLW9ubHkgcGxhaW4tY29tcG9uZW50IGFkYXB0ZXI7IG5vIHN1Ym1pc3Npb24sIHRyYWluaW5nLCBvciBhdXRvbWF0aWMgYWRtaXNzaW9uLiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gY29tcG9uZW50IGltcG9ydCBTbW9sRmluYWxQbGFpbiwgdmFsaWRhdGVfY29udHJhY3QKCnBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zcGVjIiwgZGVmYXVsdD0iaW5mZXJlbmNlLXNwZWMuanNvbiIpCnBhcnNlci5hZGRfYXJndW1lbnQoIi0tYXJ0aWZhY3Qtcm9vdCIsIHJlcXVpcmVkPVRydWUpCnBhcnNlci5hZGRfYXJndW1lbnQoIi0tYmFzZS1yb290IiwgcmVxdWlyZWQ9VHJ1ZSkKcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1hbmNob3ItdHJhaW4tY3N2IiwgcmVxdWlyZWQ9VHJ1ZSkKcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10ZXN0LWNzdiIpCnBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0cHV0IiwgZGVmYXVsdD0iL2thZ2dsZS93b3JraW5nL3Ntb2xsbTNfcGxhaW5fbG9ncC5ucHoiKQpwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXZhbGlkYXRlLW9ubHkiLCBhY3Rpb249InN0b3JlX3RydWUiKQphcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQpzcGVjID0ganNvbi5sb2FkcyhQYXRoKGFyZ3Muc3BlYykucmVhZF90ZXh0KCkpCnZhbGlkYXRlX2NvbnRyYWN0KHNwZWMsIGFyZ3MuYXJ0aWZhY3Rfcm9vdCwgYXJncy5iYXNlX3Jvb3QpCmlmIGFyZ3MudmFsaWRhdGVfb25seToKICAgIHByaW50KGpzb24uZHVtcHMoeyJzdGF0dXMiOiAiUEFTU19GVUxMX0ZJTkFMX0ZJTEVTX05PX01PREVMX0lOSVQiLCAiVG9yY2hfaW1wb3J0ZWQiOiBGYWxzZX0pLCBmbHVzaD1UcnVlKQplbHNlOgogICAgYXNzZXJ0IGFyZ3MudGVzdF9jc3YKICAgIG1vZGVsID0gU21vbEZpbmFsUGxhaW4oYXJncy5zcGVjLCBhcmdzLmFydGlmYWN0X3Jvb3QsIGFyZ3MuYmFzZV9yb290LCBhcmdzLmFuY2hvcl90cmFpbl9jc3YpCiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKICAgIHRlc3QgPSBwZC5yZWFkX2NzdihhcmdzLnRlc3RfY3N2LCB1c2Vjb2xzPVsiaWQiLCAidGV4dCJdLCBkdHlwZT1zdHIpCiAgICBhc3NlcnQgdGVzdC5ub3RuYSgpLmFsbCgpLmFsbCgpIGFuZCB0ZXN0LmlkLmlzX3VuaXF1ZQogICAgcmVzdWx0ID0gbW9kZWwucHJlZGljdF9wbGFpbl9sb2dwKHRlc3QuaWQudG9fbnVtcHkoZHR5cGU9c3RyKSwgdGVzdC50ZXh0LnRvbGlzdCgpKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwocmVzdWx0WyJpZHMiXSwgdGVzdC5pZC50b19udW1weShkdHlwZT1zdHIpKQogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChhcmdzLm91dHB1dCwgKipyZXN1bHQpCiAgICBQYXRoKGFyZ3Mub3V0cHV0KS53aXRoX3N1ZmZpeCgiLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyJzdGF0dXMiOiAiUExBSU5fQ09NUE9ORU5UX09OTFkiLAogICAgICAgICJyb3dzIjogbGVuKHRlc3QpLCAiYW5jaG9yX3JlcGxheSI6IG1vZGVsLmFuY2hvcl9yZXBsYXksCiAgICAgICAgInRyYWluaW5nX29yX3N1Ym1pc3Npb25fcGVyZm9ybWVkIjogRmFsc2UsICJjZW50ZXJpbmdfb3JfY2FsaWJyYXRpb25fYXBwbGllZCI6IEZhbHNlfSwgaW5kZW50PTIpKQo=\",\"center_readout.py\":\"aW1wb3J0IG51bXB5IGFzIG5wDQoNCmRlZiBzb3VyY2VfbG9nc29mdG1heCh2YWx1ZSk6CiAgICB2YWx1ZSA9IHZhbHVlIC0gdmFsdWUubWF4KDEsIGtlZXBkaW1zPVRydWUpCiAgICByZXR1cm4gdmFsdWUgLSBucC5sb2cobnAuZXhwKHZhbHVlKS5zdW0oMSwga2VlcGRpbXM9VHJ1ZSkpCgoKZGVmIGFkZF9sb2dwcm9iKHApOgogICAgYXNzZXJ0IG5wLmlzZmluaXRlKHApLmFsbCgpIGFuZCAocCA+PSAwKS5hbGwoKQogICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2UocC5zdW0oLTEpLCAxLCBydG9sPTAsIGF0b2w9MmUtNSkKICAgIHJldHVybiBucC5sb2cobnAuY2xpcChwLCAxZS05LCAxKSkKCgpkZWYgbG9yYV9yZWFkb3V0cyhaLCBybywgcG9vbHMsIHJlY292ZXJlZCwgYWRhcHRpdmUpOgogICAgIiIiUHJvZHVjdGlvbiBGUDMyIGFmZmluZSBhbmQgcG9vbC1sb2NhbCBxdWVyeSBjZW50ZXJpbmcsIG5ldmVyIGdsb2JhbCBjZW50ZXJpbmcuIiIiCiAgICBhc3NlcnQgWi5kdHlwZSA9PSBucC5mbG9hdDMyIGFuZCBaLnNoYXBlID09IChsZW4oWiksIDEyMjg4KQogICAgYXNzZXJ0IHJvWydwbGFpbl9jb2VmJ10uZHR5cGUgPT0gcm9bJ2NlbnRlcl9jb2VmJ10uZHR5cGUgPT0gbnAuZmxvYXQzMgogICAgbGcgPSBaIEAgcm9bJ3BsYWluX2NvZWYnXS5UICsgcm9bJ3BsYWluX2ludGVyY2VwdCddCiAgICBsZyAtPSBsZy5tYXgoMSwga2VlcGRpbXM9VHJ1ZSkKICAgIHAgPSBucC5leHAobGcpCiAgICBwbGFpbiA9IGFkZF9sb2dwcm9iKHAgLyBwLnN1bSgxLCBrZWVwZGltcz1UcnVlKSkKICAgIGNlbnRlcmVkID0gbnAuemVyb3MoKGxlbihwb29scyksIGxlbihaKSwgMTgpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgZm9yIGNvbnRleHQgaW4gcmFuZ2UobGVuKHBvb2xzKSk6CiAgICAgICAgZm9yIHBvb2wgaW4gbnAudW5pcXVlKHBvb2xzW2NvbnRleHRdKToKICAgICAgICAgICAgbG9jYWwgPSBucC5mbGF0bm9uemVybygocG9vbHNbY29udGV4dF0gPT0gcG9vbCkgJiBhZGFwdGl2ZVtjb250ZXh0XSkKICAgICAgICAgICAgaWYgbm90IGxlbihsb2NhbCk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBYID0gbnAuYXNhcnJheShaW2xvY2FsXSwgZHR5cGU9bnAuZmxvYXQzMikuY29weSgpCiAgICAgICAgICAgIGdyb3VwcyA9IHJlY292ZXJlZFtjb250ZXh0LCBsb2NhbF0KICAgICAgICAgICAgZm9yIGdyb3VwIGluIG5wLnVuaXF1ZShncm91cHMpOgogICAgICAgICAgICAgICAgaSA9IGdyb3VwcyA9PSBncm91cAogICAgICAgICAgICAgICAgWFtpXSAtPSBYW2ldLm1lYW4oMCkKICAgICAgICAgICAgbGMgPSBYIEAgcm9bJ2NlbnRlcl9jb2VmJ10uVCArIHJvWydjZW50ZXJfaW50ZXJjZXB0J10KICAgICAgICAgICAgY2VudGVyZWRbY29udGV4dCwgbG9jYWxdID0gc291cmNlX2xvZ3NvZnRtYXgobGMpCiAgICBMQyA9IE5vbmUKICAgIGlmICdjbGF1ZGVfY29lZicgaW4gcm86CiAgICAgICAgTEMgPSBzb3VyY2VfbG9nc29mdG1heChaIEAgcm9bJ2NsYXVkZV9jb2VmJ10uVCArIHJvWydjbGF1ZGVfaW50ZXJjZXB0J10pCiAgICByZXR1cm4gcGxhaW4sIGNlbnRlcmVkLCBMQwoKCg==\",\"source_balance.py\":\"aW1wb3J0IG51bXB5IGFzIG5wCksgPSAxOApCQUxBTkNFX0FMUEhBID0gLjUKCmRlZiBzb3VyY2VfYmFsYW5jZShMT0dJVCk6CiAgICBQID0gbnAuZXhwKExPR0lUIC0gTE9HSVQubWF4KDEsIGtlZXBkaW1zPVRydWUpKQogICAgUCAvPSBQLnN1bSgxLCBrZWVwZGltcz1UcnVlKQogICAgaWYgQkFMQU5DRV9BTFBIQSA+IDAgYW5kIGxlbihQKSA+PSAyOgogICAgICAgIHRhcmdldCA9ICgxIC0gQkFMQU5DRV9BTFBIQSkgKiBQLnN1bSgwKSArIEJBTEFOQ0VfQUxQSEEgKiBucC5mdWxsKEssIGxlbihQKSAvIEspCiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoNTApOgogICAgICAgICAgICBQICo9ICh0YXJnZXQgLyAoUC5zdW0oMCkgKyAxZS0xMikpW05vbmUsIDpdCiAgICAgICAgICAgIFAgLz0gUC5zdW0oMSwga2VlcGRpbXM9VHJ1ZSkgKyAxZS0xMgogICAgICAgIHByaW50KCdiYWxhbmNpbmcgY2hhbmdlZCcsIGludCgoUC5hcmdtYXgoMSkgIT0gTE9HSVQuYXJnbWF4KDEpKS5zdW0oKSksICdvZicsIGxlbihQKSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBQCg==\",\"compute_center.py\":\"IiIiT3B0aW9uYWwgb3duZWQgZmluYWwgU21vbCBmZWF0dXJlIHBhc3MgYW5kIGV4YWN0IHBvb2wtbG9jYWwgQ0VOVEVSIHJlYWRvdXQuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHRpbWUKCmhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmZyb20gYWRtaXNzaW9uX3ByZWZsaWdodCBpbXBvcnQgdmFsaWRhdGUKY29uZmlnID0gdmFsaWRhdGUoaGVyZSkKYXNzZXJ0IGNvbmZpZ1snY2FuZGlkYXRlX2FkbWl0dGVkJ10gYW5kIGNvbmZpZ1snc2VsZWN0ZWRfYnlfcm9vdCddCmFzc2VydCBjb25maWdbJ2Fzc2V0c19wdWJsaWNfYXVkaXRlZCddIGFuZCBjb25maWdbJ2RpYWdub3N0aWNfYWN0dWFsX1BBU1NfYm91bmQnXQphc3NlcnQgY29uZmlnWydyYXdfZXh0ZW5zaW9uJ10gPT0gJ3Ntb2xfY2VudGVyX2FkZCcgYW5kIGNvbmZpZ1snY29lZmZpY2llbnQnXSA9PSAuMDMKYXNzZXJ0IG9zLm5hbWUgIT0gJ250JyBhbmQgUGF0aCgnL2thZ2dsZS9pbnB1dCcpLmlzX2RpcigpCgoKZGVmIHNoYShwYXRoKToKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICByZXR1cm4gaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sICdzaGEyNTYnKS5oZXhkaWdlc3QoKQoKCmZvciBuYW1lLCBkaWdlc3QgaW4gY29uZmlnWydzb3VyY2Vfc2hhMjU2J10uaXRlbXMoKToKICAgIGFzc2VydCBzaGEoaGVyZSAvIG5hbWUpID09IGRpZ2VzdCwgbmFtZQpjYWNoZSA9IFBhdGgob3MuZW52aXJvblsnTlRMX1RBQlVMQVJfVEVNUCddKS5yZXNvbHZlKCkKYXNzZXJ0IGNhY2hlLmlzX3JlbGF0aXZlX3RvKFBhdGgoJy9rYWdnbGUvdGVtcCcpLnJlc29sdmUoKSkKYXNzZXJ0IG5vdCAoY2FjaGUgLyAnc21vbF9jZW50ZXJfYWRkLm5weicpLmV4aXN0cygpLCAnTm8gYXV0b21hdGljIHJldXNlL3JldHJ5JwpzcGVjID0ganNvbi5sb2FkcygoaGVyZSAvICdzbW9sLXNwZWMuanNvbicpLnJlYWRfdGV4dCgpKQphc3NlcnQgc3BlY1snaW5mZXJlbmNlX2F1dGhvcml6ZWQnXSBhbmQgbm90IHNwZWNbJ2RpYWdub3N0aWNfb25seV9hdXRob3JpemVkJ10KYXNzZXJ0IHNwZWNbJ2FjdHVhbF9maW5hbF9iaW5kaW5nJ11bJ3BhaXJlZF9nYWluX2FkbWl0dGVkX2J5X3Jvb3QnXQphc3NldHMgPSBbcCBmb3IgcCBpbiBQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ2Fzc2V0X21hbmlmZXN0Lmpzb24nKQogICAgICAgICAgaWYgc2hhKHApID09IGNvbmZpZ1snY29tcGFjdF9hc3NldF9tYW5pZmVzdF9zaGEyNTYnXV0Kc3RhZ2VzID0gW3AgZm9yIHAgaW4gUGF0aCgnL2thZ2dsZS9pbnB1dCcpLnJnbG9iKCdzbW9sbG0zX3N0YWdpbmdfY29tcGxldGUuanNvbicpCiAgICAgICAgICBpZiBzaGEocCkgPT0gc3BlY1snYmFzZV9tYW5pZmVzdF9zaGEyNTYnXV0KdHJhaW5zID0gW3AgZm9yIHAgaW4gUGF0aCgnL2thZ2dsZS9pbnB1dCcpLnJnbG9iKCd0cmFpbi5jc3YnKSBpZiBzaGEocCkgPT0gc3BlY1sndHJhaW5fc2hhMjU2J11dCnRlc3RzID0gbGlzdChQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoJ3Rlc3QuY3N2JykpCmFzc2VydCBsZW4oYXNzZXRzKSA9PSBsZW4oc3RhZ2VzKSA9PSBsZW4odHJhaW5zKSA9PSBsZW4odGVzdHMpID09IDEKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gY29tcG9uZW50IGltcG9ydCBTbW9sRmluYWxQbGFpbgpmcm9tIGNlbnRlcl9yZWFkb3V0IGltcG9ydCBsb3JhX3JlYWRvdXRzCgp3aXRoIG5wLmxvYWQoY2FjaGUgLyAnYmFzZWxpbmVfb2JzZXJ2YXRpb24ubnB6JywgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyB2YWx1ZXM6CiAgICBvYnNlcnZlZCA9IHtrZXk6IHZhbHVlc1trZXldLmNvcHkoKSBmb3Iga2V5IGluIHZhbHVlcy5maWxlc30KcXVlcnkgPSBwZC5yZWFkX2Nzdih0ZXN0c1swXSwgdXNlY29scz1bJ2lkJywgJ3RleHQnXSwgZHR5cGU9c3RyKQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChxdWVyeS5pZC50b19udW1weShkdHlwZT1zdHIpLCBvYnNlcnZlZFsnaWRzJ10pCmFzc2VydCBxdWVyeS5pZC5pc191bmlxdWUgYW5kIGxlbihxdWVyeSkgPiAwCm1vZGVsID0gU21vbEZpbmFsUGxhaW4oaGVyZSAvICdzbW9sLXNwZWMuanNvbicsIGFzc2V0c1swXS5wYXJlbnQsIHN0YWdlc1swXS5wYXJlbnQsIHRyYWluc1swXSkKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwobW9kZWwucmVhZG91dHNbJ2NsYXNzZXMnXSwgb2JzZXJ2ZWRbJ2xhYmVscyddKQppbXBvcnQgdG9yY2gKYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgPT0gMSBhbmQgb3MuZW52aXJvblsnQ1VEQV9WSVNJQkxFX0RFVklDRVMnXS5zdGFydHN3aXRoKCdHUFUtJykKcXVlcnlfc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKYmVnaW4sIGVuZCA9IHRvcmNoLmN1ZGEuRXZlbnQoZW5hYmxlX3RpbWluZz1UcnVlKSwgdG9yY2guY3VkYS5FdmVudChlbmFibGVfdGltaW5nPVRydWUpCmJlZ2luLnJlY29yZCgpCiMgT25lIHF1ZXJ5IGZvcndhcmQgcGFzczsgX2ZlYXR1cmVzIHJlc3RvcmVzIHRoZSBhY3R1YWwgQ1NWIG9yZGVyIGFmdGVyIHN0YWJsZSBsZW5ndGggc29ydGluZy4KWiA9IG1vZGVsLl9mZWF0dXJlcyhxdWVyeS50ZXh0LnRvbGlzdCgpKQplbmQucmVjb3JkKCkKdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCnF1ZXJ5X2N1ZGFfbXMgPSBmbG9hdChiZWdpbi5lbGFwc2VkX3RpbWUoZW5kKSkKYXNzZXJ0IHF1ZXJ5X2N1ZGFfbXMgPiAwCmFzc2VydCBaLmR0eXBlID09IG5wLmZsb2F0MzIgYW5kIFouc2hhcGUgPT0gKGxlbihxdWVyeSksIDEyMjg4KQpwb29scyA9IG9ic2VydmVkWydwb29sX2lkcyddW05vbmVdCmdyb3VwcyA9IG9ic2VydmVkWydyZWNvdmVyZWRfZ3JvdXBfaWRzJ11bTm9uZV0KYWRhcHRpdmUgPSBvYnNlcnZlZFsnYWRhcHRpdmVfbWFzayddW05vbmVdCmFzc2VydCBwb29scy5zaGFwZSA9PSBncm91cHMuc2hhcGUgPT0gYWRhcHRpdmUuc2hhcGUgPT0gKDEsIGxlbihxdWVyeSkpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHBvb2xzLCBucC56ZXJvc19saWtlKHBvb2xzKSkKIyBUaGUgYzY2YzUgYm9keSBjZW50ZXJzIHNlcGFyYXRlbHkgd2l0aGluIGVhY2ggYWN0dWFsIHJlY292ZXJlZCBhZGFwdGl2ZSBncm91cC9wb29sLgojIEl0cyB1bnVzZWQgcGxhaW4gcmV0dXJuIGlzIG5ldmVyIGFkZGVkIHRvIHRoZSBlbnNlbWJsZSBhbmQgdHJpZ2dlcnMgbm8gZXh0cmEgbW9kZWwgcGFzcy4KXywgY2VudGVyZWQsIF8gPSBsb3JhX3JlYWRvdXRzKFosIG1vZGVsLnJlYWRvdXRzLCBwb29scywgZ3JvdXBzLCBhZGFwdGl2ZSkKU0MgPSBjZW50ZXJlZFswXQphc3NlcnQgU0MuZHR5cGUgPT0gbnAuZmxvYXQzMiBhbmQgU0Muc2hhcGUgPT0gKGxlbihxdWVyeSksIDE4KSBhbmQgbnAuaXNmaW5pdGUoU0MpLmFsbCgpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKFNDW35hZGFwdGl2ZVswXV0sIDApCm5wLnNhdmV6X2NvbXByZXNzZWQoY2FjaGUgLyAnc21vbF9jZW50ZXJfYWRkLm5weicsIGlkcz1vYnNlcnZlZFsnaWRzJ10sIGxhYmVscz1vYnNlcnZlZFsnbGFiZWxzJ10sCiAgICAgICAgICAgICAgICAgICAgU0M9U0MsIHBvb2xfaWRzPXBvb2xzWzBdLCByZWNvdmVyZWRfZ3JvdXBfaWRzPWdyb3Vwc1swXSwgYWRhcHRpdmVfbWFzaz1hZGFwdGl2ZVswXSkKUGF0aCgnL2thZ2dsZS93b3JraW5nL3Ntb2xfY2VudGVyX2luZmVyZW5jZV9hdWRpdC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3QoCiAgICBzdGF0dXM9J09XTkVEX0ZJTkFMX0NFTlRFUl9GRUFUVVJFX1BBU1NfQ09NUExFVEUnLCBxdWVyeV9mZWF0dXJlX3Bhc3Nlcz0xLCBxdWVyeV9yb3dzPWxlbihxdWVyeSksCiAgICByYXdfZmVhdHVyZV9zaGFwZT1saXN0KFouc2hhcGUpLCByYXdfZmVhdHVyZV9kdHlwZT1zdHIoWi5kdHlwZSksIGFuY2hvcl9yZXBsYXk9bW9kZWwuYW5jaG9yX3JlcGxheSwKICAgIENVREFfVklTSUJMRV9ERVZJQ0VTPW9zLmVudmlyb25bJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ10sIFBJRD1vcy5nZXRwaWQoKSwKICAgIGxvZ2ljYWxfZ3B1X25hbWU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCksIHF1ZXJ5X2N1ZGFfZXZlbnRfbXM9cXVlcnlfY3VkYV9tcywKICAgIHF1ZXJ5X3dhbGxfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXF1ZXJ5X3N0YXJ0ZWQsCiAgICBjZW50ZXJfaGVscGVyX3NoYTI1Nj1zaGEoaGVyZSAvICdjZW50ZXJfcmVhZG91dC5weScpLCBncm91cHNfcmVjbHVzdGVyZWQ9RmFsc2UsCiAgICBjZW50ZXJzX3dpdGhpbl9hY3R1YWxfcG9vbHNfYW5kX3JlY292ZXJlZF9ncm91cHM9VHJ1ZSwgemVyb19vdXRzaWRlX2FkYXB0aXZlPVRydWUsCiAgICB1c2VzX3BsYWluX3JlYWRvdXQ9RmFsc2UsIHRyYWluaW5nPUZhbHNlLCB0YXJnZXRfY29sdW1uX3JlYWQ9RmFsc2UpLCBpbmRlbnQ9MikpCg==\",\"finish_center_protected.py\":\"IiIiT3B0aW9uYWwgcmF3Ky4wMypTQywgb3JpZ2luYWwgYWxwaGEuNSBiYWxhbmNlL2dhdGUsIHRoZW4gc2hhcmVkIHByb3RlY3RlZCBQRk4uMTAuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBjbGF1ZGVfZ2F0ZSBpbXBvcnQgZ2F0ZWRfcHJlZGljdCwgbG9hZF9nYXRlX3N0YXRlCmZyb20gcHJvdGVjdGVkX29wZXJhdG9yIGltcG9ydCBwcm90ZWN0ZWQKZnJvbSBzb3VyY2VfYmFsYW5jZSBpbXBvcnQgc291cmNlX2JhbGFuY2UKCmhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmZyb20gYWRtaXNzaW9uX3ByZWZsaWdodCBpbXBvcnQgdmFsaWRhdGUKY29uZmlnID0gdmFsaWRhdGUoaGVyZSkKYXNzZXJ0IGNvbmZpZ1snY2FuZGlkYXRlX2FkbWl0dGVkJ10gYW5kIGNvbmZpZ1snc2VsZWN0ZWRfYnlfcm9vdCddCmFzc2VydCBjb25maWdbJ2Fzc2V0c19wdWJsaWNfYXVkaXRlZCddIGFuZCBjb25maWdbJ2RpYWdub3N0aWNfYWN0dWFsX1BBU1NfYm91bmQnXQphc3NlcnQgY29uZmlnWydyYXdfZXh0ZW5zaW9uJ10gPT0gJ3Ntb2xfY2VudGVyX2FkZCcgYW5kIGNvbmZpZ1snY29lZmZpY2llbnQnXSA9PSAuMDMKYXNzZXJ0IGNvbmZpZ1snYWxwaGEnXSA9PSAuNSBhbmQgY29uZmlnWydwcm90ZWN0ZWRfd2VpZ2h0J10gPT0gLjEwIGFuZCBjb25maWdbJ3Bvc3QnXSA9PSAwCmZvciBuYW1lLCBkaWdlc3QgaW4gY29uZmlnWydzb3VyY2Vfc2hhMjU2J10uaXRlbXMoKToKICAgIGFzc2VydCBoYXNobGliLnNoYTI1NigoaGVyZSAvIG5hbWUpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkgPT0gZGlnZXN0LCBuYW1lCmNhY2hlID0gUGF0aChvcy5lbnZpcm9uWydOVExfVEFCVUxBUl9URU1QJ10pLnJlc29sdmUoKQphc3NlcnQgY2FjaGUuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQoKCmRlZiBsb2FkKG5hbWUpOgogICAgd2l0aCBucC5sb2FkKGNhY2hlIC8gbmFtZSwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyB2YWx1ZXM6CiAgICAgICAgcmV0dXJuIHtrZXk6IHZhbHVlc1trZXldLmNvcHkoKSBmb3Iga2V5IGluIHZhbHVlcy5maWxlc30KCgpvYnNlcnZlZCwgc21vbCwgcGZuID0gKGxvYWQobmFtZSkgZm9yIG5hbWUgaW4KICAgICgnYmFzZWxpbmVfb2JzZXJ2YXRpb24ubnB6JywgJ3Ntb2xfY2VudGVyX2FkZC5ucHonLCAndGFicGZuX3Byb2JhLm5weicpKQpyYXcsIFAsIGlkcywgbGFiZWxzID0gKG9ic2VydmVkW2tleV0gZm9yIGtleSBpbiAoJ3JhdycsICdiYWxhbmNlZCcsICdpZHMnLCAnbGFiZWxzJykpCmFzc2VydCByYXcuZHR5cGUgPT0gUC5kdHlwZSA9PSBucC5mbG9hdDY0IGFuZCByYXcuc2hhcGUgPT0gUC5zaGFwZSA9PSAobGVuKGlkcyksIDE4KQpmb3IgdmFsdWVzIGluIChzbW9sLCBwZm4pOgogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwodmFsdWVzWydpZHMnXSwgaWRzKQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwodmFsdWVzWydsYWJlbHMnXSwgbGFiZWxzKQpmb3Iga2V5IGluICgncG9vbF9pZHMnLCAncmVjb3ZlcmVkX2dyb3VwX2lkcycsICdhZGFwdGl2ZV9tYXNrJyk6CiAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChzbW9sW2tleV0sIG9ic2VydmVkW2tleV0pCiMgVGhlIGV4dHJhY3RlZCBvcmlnaW5hbCBzb3VyY2UgYmFsYW5jZSBtdXN0IHJlcHJvZHVjZSB0aGUgb2JzZXJ2ZWQgYmFzZWxpbmUgYml0LWZvci1iaXQuCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHNvdXJjZV9iYWxhbmNlKHJhdyksIFApCmdhdGUgPSBsb2FkX2dhdGVfc3RhdGUoY2FjaGUsIFAsIGlkcywgbGFiZWxzKQpiYXNlbGluZSA9IGdhdGVkX3ByZWRpY3QoUCwgbGFiZWxzLCAqKmdhdGUpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKGJhc2VsaW5lLCBvYnNlcnZlZFsnZ2F0ZWRfcHJlZGljdGlvbnMnXSkKY29udHJvbCA9IHBkLnJlYWRfcGFycXVldCgnL2thZ2dsZS93b3JraW5nL0I1NTZfZ2F0ZWRfY29udHJvbC5wYXJxdWV0JykKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoY29udHJvbC5pZC50b19udW1weShkdHlwZT1zdHIpLCBpZHMpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKGNvbnRyb2wubW9kZWwudG9fbnVtcHkoZHR5cGU9c3RyKSwgbGFiZWxzW2Jhc2VsaW5lXSkKU0MgPSBzbW9sWydTQyddCmFzc2VydCBTQy5kdHlwZSA9PSBucC5mbG9hdDMyIGFuZCBTQy5zaGFwZSA9PSByYXcuc2hhcGUgYW5kIG5wLmlzZmluaXRlKFNDKS5hbGwoKQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChTQ1t+b2JzZXJ2ZWRbJ2FkYXB0aXZlX21hc2snXV0sIDApCiMgU2FtZSBGUDMyIG11bHRpcGxpY2F0aW9uIGFuZCBwcm9tb3Rpb24gb3JkZXIgYXMgdGhlIGFjdHVhbCBjNjZjNSBjb25kaXRpb25hbCBDUFUgYXJtLgpleHRlbmRlZF9yYXcgPSByYXcgKyAuMDMgKiBTQwpxID0gc291cmNlX2JhbGFuY2UoZXh0ZW5kZWRfcmF3KQpleHRlbmRlZF9sYWJlbHMgPSBnYXRlZF9wcmVkaWN0KHEsIGxhYmVscywgKipnYXRlKQpjbGF1ZGUgPSBnYXRlWydDTEFVREVfSURYJ10Kbm9uY2xhdWRlID0gbnAuZmxhdG5vbnplcm8ofm5wLmNoYXIuc3RhcnRzd2l0aChsYWJlbHMsICdjbGF1ZGUnKSkKYXNzZXJ0IGxlbihjbGF1ZGUpID09IDUgYW5kIGxlbihub25jbGF1ZGUpID09IDEzCnRhYnVsYXIgPSBwZm5bJ3Byb2JhJ10KYXNzZXJ0IHRhYnVsYXIuZHR5cGUgPT0gbnAuZmxvYXQ2NCBhbmQgdGFidWxhci5zaGFwZSA9PSBxLnNoYXBlIGFuZCBucC5pc2Zpbml0ZSh0YWJ1bGFyKS5hbGwoKQpjYW5kaWRhdGUsIHByZWRpY3Rpb24sIHJldmVydGVkID0gcHJvdGVjdGVkKHFbTm9uZV0sIHRhYnVsYXIsIGV4dGVuZGVkX2xhYmVsc1tOb25lXSwgY2xhdWRlLCBub25jbGF1ZGUsIHdlaWdodD0uMTApCnN1Ym1pc3Npb24gPSBwZC5EYXRhRnJhbWUoZGljdChpZD1pZHMsIG1vZGVsPWxhYmVsc1twcmVkaWN0aW9uWzBdXSkpCmFzc2VydCBzdWJtaXNzaW9uLmlkLmlzX3VuaXF1ZSBhbmQgbGVuKHN1Ym1pc3Npb24pID09IGxlbihjb250cm9sKSBhbmQgc3VibWlzc2lvbi5tb2RlbC5pc2luKGxhYmVscykuYWxsKCkKc3VibWlzc2lvbi50b19wYXJxdWV0KCcva2FnZ2xlL3dvcmtpbmcvc3VibWlzc2lvbi5wYXJxdWV0JywgaW5kZXg9RmFsc2UpClBhdGgoJy9rYWdnbGUvd29ya2luZy9wcm90ZWN0ZWRfcmVzaWR1YWxfYXVkaXQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KAogICAgc3RhdHVzPSdTTU9MX0NFTlRFUl9BRERfUFJPVEVDVEVEX1BGTjEwX1BSRURJQ1RJT05TX0NPTVBMRVRFJywgYmFzZWxpbmVfYmFsYW5jZV9hbmRfZ2F0ZV9yZXBsYXlfYml0ZXhhY3Q9VHJ1ZSwKICAgIGZvcm11bGE9J29yaWdpbmFsX3JhdyArIC4wMypTQyAtPiBvcmlnaW5hbF9hbHBoYS41X2JhbGFuY2UgLT4gb3JpZ2luYWxfQ2xhdWRlX2dhdGUgLT4gcHJvdGVjdGVkX1BGTi4xMCcsCiAgICBzb3VyY2Vfbm90ZWJvb2tfaWQ9MTM5MjAwMTAwLCBzb3VyY2VfdmVyc2lvbj0yLCBjb2VmZmljaWVudD0uMDMsIGFscGhhPS41LCBwb3N0PTAsCiAgICByZXNpZHVhbF93ZWlnaHQ9LjEwLCBxdWVyeV9yb3dzPWxlbihpZHMpLCBleHRyYV9nbG9iYWxfbm9ybWFsaXphdGlvbj1GYWxzZSwKICAgIHByb3RlY3RlZF9mYW1pbHlfbWVtYmVyc2hpcF9yZWxhdGl2ZV90bz0nU21vbENFTlRFUiBleHRlbmRlZCBvcmlnaW5hbCBnYXRlIGRlY2lzaW9ucycsCiAgICByZXZlcnRlZF9mYW1pbHlfZmxpcHM9aW50KHJldmVydGVkLnN1bSgpKSwgdHJhaW5pbmc9RmFsc2UpLCBpbmRlbnQ9MikpCg==\",\"admission_preflight.py\":\"IiIiU3RkbGliLW9ubHkgaW1tdXRhYmxlIGRpYWdub3N0aWMgZXZpZGVuY2UgYW5kIGV4cGxpY2l0IGNhbmRpZGF0ZSBhZG1pc3Npb24gZ3VhcmRzLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgdmFsaWRhdGUoaGVyZSk6CiAgICBoZXJlID0gUGF0aChoZXJlKQogICAgY29uZmlnID0ganNvbi5sb2FkcygoaGVyZSAvICdvcHRpb25hbC1jb25maWcuanNvbicpLnJlYWRfdGV4dCgpKQogICAgYXNzZXJ0IGNvbmZpZ1snY2FuZGlkYXRlX2FkbWl0dGVkJ10gYW5kIGNvbmZpZ1snc2VsZWN0ZWRfYnlfcm9vdCddCiAgICBhc3NlcnQgY29uZmlnWydhc3NldHNfcHVibGljX2F1ZGl0ZWQnXSBhbmQgY29uZmlnWydkaWFnbm9zdGljX2FjdHVhbF9QQVNTX2JvdW5kJ10KICAgIGFzc2VydCBjb25maWdbJ3Jvb3RfZGVjaXNpb25fYmluZGluZyddIGFuZCBjb25maWdbJ3Jvb3RfZGVjaXNpb25fYmluZGluZyddWydleHBsaWNpdF9jYW5kaWRhdGVfYWRtaXNzaW9uJ10KICAgIGJpbmRpbmcgPSBjb25maWdbJ2RpYWdub3N0aWNfUEFTU19iaW5kaW5nJ10KICAgIGFzc2VydCBiaW5kaW5nWydub3RlYm9va19pZCddID09IDEzOTM1MzQ1NSBhbmQgYmluZGluZ1sndmVyc2lvbiddID09IDMgYW5kIGJpbmRpbmdbJ3NjcmlwdF92ZXJzaW9uX2lkJ10gPT0gMzU5NTgxOTg4CiAgICByYXcgPSAoaGVyZSAvICdhY3R1YWwtZGlhZ25vc3RpYy1hdWRpdC5qc29uJykucmVhZF9ieXRlcygpCiAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKSA9PSBiaW5kaW5nWydhdWRpdF9zaGEyNTYnXQogICAgYXVkaXQgPSBqc29uLmxvYWRzKHJhdykKICAgIGFzc2VydCBhdWRpdFsnbm90ZWJvb2tfaWQnXSA9PSBiaW5kaW5nWydub3RlYm9va19pZCddIGFuZCBhdWRpdFsndmVyc2lvbiddID09IDMKICAgIGFzc2VydCBhdWRpdFsnY29tcG9uZW50X3NoYTI1NiddID09IGNvbmZpZ1snc291cmNlX3NoYTI1NiddWydjb21wb25lbnQucHknXQogICAgYXNzZXJ0IGF1ZGl0Wydib3RoX3BoeXNpY2FsX1Q0X2ZvcndhcmRfQ1VEQV9ldmVudHNfcHJvdmVuJ10gYW5kIGF1ZGl0Wyd0cmFpbmluZ191cGRhdGVzJ10gPT0gMAogICAgYXNzZXJ0IG5vdCBhdWRpdFsnY2FuZGlkYXRlX29yX3Byb2R1Y3Rpb25fYWRtaXR0ZWQnXQogICAgZm9yIHJvd3MgaW4gKCczJywgJzUnKToKICAgICAgICByZXBvcnQgPSBhdWRpdFsncmVwb3J0cyddW3Jvd3NdCiAgICAgICAgYXNzZXJ0IHJlcG9ydFsnc3RhdHVzJ10gPT0gJ1BBU1NfSU5GRVJFTkNFX0RJQUdOT1NUSUNfT05MWScKICAgICAgICBhc3NlcnQgcmVwb3J0Wydjb21wb25lbnRfc2hhMjU2J10gPT0gY29uZmlnWydzb3VyY2Vfc2hhMjU2J11bJ2NvbXBvbmVudC5weSddCiAgICAgICAgYXNzZXJ0IHJlcG9ydFsnb3duZWRfcGF0aF9zaGEyNTYnXSA9PSBjb25maWdbJ3NvdXJjZV9zaGEyNTYnXVsnb3duZWRfcGF0aC5weSddCiAgICAgICAgYXNzZXJ0IHJlcG9ydFsnb2JzZXJ2ZWRfcXVlcnlfYmF0Y2hfc2l6ZXMnXSA9PSAoWzNdIGlmIHJvd3MgPT0gJzMnIGVsc2UgWzMsIDJdKQogICAgICAgIGFzc2VydCByZXBvcnRbJ3ByZWRpY3Rpb25fY3VkYV9ldmVudF9tcyddID4gMCBhbmQgcmVwb3J0WydmaW5pdGUnXSBhbmQgcmVwb3J0WydpZF9vcmRlcl9wcmVzZXJ2ZWQnXQogICAgICAgIHJlcGxheSA9IHJlcG9ydFsnYW5jaG9yX3JlcGxheSddCiAgICAgICAgYXNzZXJ0IHJlcGxheVsnZmVhdHVyZV9ydG9sJ10gPT0gcmVwbGF5WydmZWF0dXJlX2F0b2wnXSA9PSAxZS01IGFuZCByZXBsYXlbJ3Byb2JhYmlsaXR5X2F0b2wnXSA9PSAxZS02CiAgICAgICAgYXNzZXJ0IHJlcGxheVsnZmVhdHVyZV9tYXhhYnMnXSA9PSAwIGFuZCByZXBsYXlbJ3Byb2JhYmlsaXR5X21heGFicyddID09IHsncGxhaW4nOiAwLCAnY2VudGVyJzogMH0KICAgIHJldHVybiBjb25maWcK\",\"run_parallel.py\":\"IiIiT3B0aW9uYWwgVVVJRC1pc29sYXRlZCBHUFUwIFBGTjEyOCBhbmQgR1BVMSBvd25lZCBTbW9sOyBubyBzaGFyZWQtR1BVIGxvYWRpbmcuIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNpZ25hbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgpoZXJlID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApvcHRpb25hbCA9IGpzb24ubG9hZHMoKGhlcmUgLyAnb3B0aW9uYWwtY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKY29uZmlnID0ganNvbi5sb2FkcygoaGVyZSAvICdydW50aW1lX2NvbmZpZy5qc29uJykucmVhZF90ZXh0KCkpCmFzc2VydCBvcHRpb25hbFsnY2FuZGlkYXRlX2FkbWl0dGVkJ10gYW5kIG9wdGlvbmFsWydzZWxlY3RlZF9ieV9yb290J10gYW5kIGNvbmZpZ1snY2FuZGlkYXRlX2FkbWl0dGVkJ10KYXNzZXJ0IG9wdGlvbmFsWydhc3NldHNfcHVibGljX2F1ZGl0ZWQnXSBhbmQgb3B0aW9uYWxbJ2RpYWdub3N0aWNfYWN0dWFsX1BBU1NfYm91bmQnXQphc3NlcnQgY29uZmlnWydyYXdfZXh0ZW5zaW9uJ10gPT0gb3B0aW9uYWxbJ3Jhd19leHRlbnNpb24nXSA9PSAnc21vbF9jZW50ZXJfYWRkJwphc3NlcnQgY29uZmlnWyd0YWJwZm4nXVsnbl9lc3RpbWF0b3JzJ10gPT0gMTI4CmFzc2VydCBvcy5uYW1lICE9ICdudCcgYW5kIFBhdGgoJy9rYWdnbGUvaW5wdXQnKS5pc19kaXIoKQptYXBwaW5nID0gW10KZm9yIGxpbmUgaW4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoWydudmlkaWEtc21pJywgJy0tcXVlcnktZ3B1PWluZGV4LHV1aWQsbmFtZScsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSwgdGV4dD1UcnVlKS5zdHJpcCgpLnNwbGl0bGluZXMoKToKICAgIGluZGV4LCB1dWlkLCBuYW1lID0gW3ZhbHVlLnN0cmlwKCkgZm9yIHZhbHVlIGluIGxpbmUuc3BsaXQoJywnLCAyKV0KICAgIG1hcHBpbmcuYXBwZW5kKGRpY3QocGh5c2ljYWxfaW5kZXg9aW50KGluZGV4KSwgdXVpZD11dWlkLCBuYW1lPW5hbWUpKQptYXBwaW5nLnNvcnQoa2V5PWxhbWJkYSBncHU6IGdwdVsncGh5c2ljYWxfaW5kZXgnXSkKYXNzZXJ0IFtncHVbJ3BoeXNpY2FsX2luZGV4J10gZm9yIGdwdSBpbiBtYXBwaW5nXSA9PSBbMCwgMV0KYXNzZXJ0IGFsbCgnVDQnIGluIGdwdVsnbmFtZSddIGFuZCBncHVbJ3V1aWQnXS5zdGFydHN3aXRoKCdHUFUtJykgZm9yIGdwdSBpbiBtYXBwaW5nKQphc3NlcnQgbWFwcGluZ1swXVsndXVpZCddICE9IG1hcHBpbmdbMV1bJ3V1aWQnXQpvdXQgPSBQYXRoKCcva2FnZ2xlL3dvcmtpbmcvc21vbF9wZm5fcGFyYWxsZWwnKQpvdXQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCmNoaWxkcmVuLCBsb2dzLCByZWNvcmRzID0gW10sIFtdLCBbXQpzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQp0cnk6CiAgICBmb3IgaW5kZXgsIHJvdXRlIGluIGVudW1lcmF0ZSgoJ1BGTjEyOCcsICdTbW9sQ0VOVEVSJykpOgogICAgICAgIGVudiA9IG9zLmVudmlyb24uY29weSgpCiAgICAgICAgZW52WydDVURBX1ZJU0lCTEVfREVWSUNFUyddID0gbWFwcGluZ1tpbmRleF1bJ3V1aWQnXQogICAgICAgIGVudlsnSEZfSFVCX09GRkxJTkUnXSA9IGVudlsnVFJBTlNGT1JNRVJTX09GRkxJTkUnXSA9ICcxJwogICAgICAgICMgUHJlc2VydmUgdGhlIGFscmVhZHkgaW5zdGFsbGVkIFBGTi1vbmx5IGFkZC1vbiBlbnZpcm9ubWVudDsgU21vbCB1c2VzIHRoZSBvcmlnaW5hbCBydW50aW1lLgogICAgICAgIGlmIHJvdXRlID09ICdQRk4xMjgnOgogICAgICAgICAgICBlbnZbJ1BZVEhPTlBBVEgnXSA9IG9zLmVudmlyb25bJ05UTF9UQUJVTEFSX1NJVEUnXQogICAgICAgIGNvbW1hbmQgPSAoW3N5cy5leGVjdXRhYmxlLCAnLXUnLCBzdHIoaGVyZSAvICd0YWJ1bGFyX3dvcmtlci5weScpLCAndGFicGZuJ10gaWYgaW5kZXggPT0gMAogICAgICAgICAgICAgICAgICAgZWxzZSBbc3lzLmV4ZWN1dGFibGUsICctdScsIHN0cihoZXJlIC8gJ2NvbXB1dGVfY2VudGVyLnB5JyldKQogICAgICAgIGxvZyA9IChvdXQgLyAocm91dGUgKyAnLmxvZycpKS5vcGVuKCd3JywgYnVmZmVyaW5nPTEpCiAgICAgICAgbG9ncy5hcHBlbmQobG9nKQogICAgICAgIGNoaWxkID0gc3VicHJvY2Vzcy5Qb3Blbihjb21tYW5kLCBlbnY9ZW52LCBzdGRvdXQ9bG9nLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgY2hpbGRyZW4uYXBwZW5kKGNoaWxkKQogICAgICAgIHJlY29yZHMuYXBwZW5kKGRpY3Qocm91dGU9cm91dGUsIFBJRD1jaGlsZC5waWQsIEdQVT1tYXBwaW5nW2luZGV4XSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0ZWRfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQsIGZpbmlzaGVkX3NlY29uZHM9Tm9uZSkpCiAgICB0aWNrLCBvdmVybGFwX3NhbXBsZXMgPSAwLCAwCiAgICBvYnNlcnZlZF9jb21wdXRlX1BJRHMgPSBzZXQoKQogICAgd2hpbGUgVHJ1ZToKICAgICAgICBlbGFwc2VkID0gdGltZS5tb25vdG9uaWMoKS1zdGFydGVkCiAgICAgICAgY29kZXMgPSBbY2hpbGQucG9sbCgpIGZvciBjaGlsZCBpbiBjaGlsZHJlbl0KICAgICAgICBmb3IgaSwgY29kZSBpbiBlbnVtZXJhdGUoY29kZXMpOgogICAgICAgICAgICBpZiBjb2RlIGlzIG5vdCBOb25lIGFuZCByZWNvcmRzW2ldWydmaW5pc2hlZF9zZWNvbmRzJ10gaXMgTm9uZToKICAgICAgICAgICAgICAgIHJlY29yZHNbaV1bJ2ZpbmlzaGVkX3NlY29uZHMnXSA9IGVsYXBzZWQKICAgICAgICAgICAgaWYgY29kZSBpcyBOb25lOgogICAgICAgICAgICAgICAgbGltaXQgPSBjb25maWdbJ21vZGVsX3RpbWVvdXRfc2Vjb25kcyddIGlmIGkgPT0gMCBlbHNlIG9wdGlvbmFsWydzbW9sX3RpbWVvdXRfc2Vjb25kcyddCiAgICAgICAgICAgICAgICBhc3NlcnQgZWxhcHNlZCA8IGxpbWl0LCAocmVjb3Jkc1tpXVsncm91dGUnXSwgJ3RpbWVvdXQ7IG5vIGF1dG9tYXRpYyByZXRyeScpCiAgICAgICAgaWYgYW55KGNvZGUgaXMgbm90IE5vbmUgYW5kIGNvZGUgIT0gMCBmb3IgY29kZSBpbiBjb2Rlcyk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmJ1BhcmFsbGVsIHdvcmtlciBmYWlsZWR7Y29kZXN9OyBubyBmaW5hbGl6YXRpb24gb3IgYXV0b21hdGljIHJldHJ5JykKICAgICAgICBzYW1wbGUgPSBzdWJwcm9jZXNzLmNoZWNrX291dHB1dChbJ252aWRpYS1zbWknLCAnLS1xdWVyeS1ncHU9aW5kZXgsdXVpZCx1dGlsaXphdGlvbi5ncHUsbWVtb3J5LnVzZWQnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLCB0ZXh0PVRydWUsIHRpbWVvdXQ9MTUpCiAgICAgICAgYXBwcyA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWNvbXB1dGUtYXBwcz1waWQsZ3B1X3V1aWQsdXNlZF9tZW1vcnknLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSwgdGV4dD1UcnVlLCB0aW1lb3V0PTE1KQogICAgICAgIGFjdGl2ZSA9IHt9CiAgICAgICAgZm9yIGxpbmUgaW4gc2FtcGxlLnN0cmlwKCkuc3BsaXRsaW5lcygpOgogICAgICAgICAgICBpbmRleCwgdXVpZCwgdXRpbGl6YXRpb24sIG1lbW9yeSA9IFt2YWx1ZS5zdHJpcCgpIGZvciB2YWx1ZSBpbiBsaW5lLnNwbGl0KCcsJyldCiAgICAgICAgICAgIGFjdGl2ZVt1dWlkXSA9IGZsb2F0KHV0aWxpemF0aW9uKSA+IDAKICAgICAgICBtYXRjaGVkID0gW10KICAgICAgICBmb3IgcmVjb3JkIGluIHJlY29yZHM6CiAgICAgICAgICAgIGV4YWN0ID0gYW55KGZpZWxkc1swXS5zdHJpcCgpID09IHN0cihyZWNvcmRbJ1BJRCddKSBhbmQgZmllbGRzWzFdLnN0cmlwKCkgPT0gcmVjb3JkWydHUFUnXVsndXVpZCddCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBmaWVsZHMgaW4gKGxpbmUuc3BsaXQoJywnKSBmb3IgbGluZSBpbiBhcHBzLnN0cmlwKCkuc3BsaXRsaW5lcygpKSBpZiBsZW4oZmllbGRzKSA+PSAyKQogICAgICAgICAgICBpZiBleGFjdDoKICAgICAgICAgICAgICAgIG9ic2VydmVkX2NvbXB1dGVfUElEcy5hZGQocmVjb3JkWydQSUQnXSkKICAgICAgICAgICAgbWF0Y2hlZC5hcHBlbmQoZXhhY3QpCiAgICAgICAgYm90aF9jb21wdXRlID0gYWxsKGNvZGUgaXMgTm9uZSBmb3IgY29kZSBpbiBjb2RlcykgYW5kIGFsbChtYXRjaGVkKSBhbmQgYWxsKGFjdGl2ZS5nZXQoZ3B1Wyd1dWlkJ10sIEZhbHNlKSBmb3IgZ3B1IGluIG1hcHBpbmcpCiAgICAgICAgb3ZlcmxhcF9zYW1wbGVzICs9IGludChib3RoX2NvbXB1dGUpCiAgICAgICAgd2l0aCAob3V0IC8gJ2dwdS10ZWxlbWV0cnkuanNvbmwnKS5vcGVuKCdhJykgYXMgaGFuZGxlOgogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhkaWN0KGVsYXBzZWRfc2Vjb25kcz1lbGFwc2VkLCBHUFU9c2FtcGxlLCBjb21wdXRlX2FwcHM9YXBwcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB3b3JrZXJfY29kZXM9Y29kZXMsIGJvdGhfd29ya2Vyc19tYXRjaGVkX2FjdGl2ZT1ib3RoX2NvbXB1dGUpKSsnXG4nKQogICAgICAgIGlmIHRpY2sgJSA2ID09IDA6CiAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMoZGljdChzdGFnZT0nUEZOX1NNT0xfUFJPR1JFU1MnLCBlbGFwc2VkX3NlY29uZHM9ZWxhcHNlZCwgd29ya2Vycz1yZWNvcmRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuX2NvZGVzPWNvZGVzLCBkZXZpY2VzPXNhbXBsZSkpLCBmbHVzaD1UcnVlKQogICAgICAgIGlmIGFsbChjb2RlID09IDAgZm9yIGNvZGUgaW4gY29kZXMpOgogICAgICAgICAgICBicmVhawogICAgICAgIHRpY2sgKz0gMQogICAgICAgIHRpbWUuc2xlZXAoNSkKICAgIGFzc2VydCBhbGwocmVjb3JkWydQSUQnXSBpbiBvYnNlcnZlZF9jb21wdXRlX1BJRHMgZm9yIHJlY29yZCBpbiByZWNvcmRzKSwgJ0FjdHVhbCBwZXItd29ya2VyIEdQVSBtYXBwaW5nIG5vdCBvYnNlcnZlZCcKICAgIHJlcG9ydCA9IGpzb24ubG9hZHMoUGF0aCgnL2thZ2dsZS93b3JraW5nL3Ntb2xfY2VudGVyX2luZmVyZW5jZV9hdWRpdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICBhc3NlcnQgcmVwb3J0WydzdGF0dXMnXSA9PSAnT1dORURfRklOQUxfQ0VOVEVSX0ZFQVRVUkVfUEFTU19DT01QTEVURScgYW5kIHJlcG9ydFsncXVlcnlfY3VkYV9ldmVudF9tcyddID4gMAogICAgYXNzZXJ0IHJlcG9ydFsnQ1VEQV9WSVNJQkxFX0RFVklDRVMnXSA9PSBtYXBwaW5nWzFdWyd1dWlkJ10gYW5kIHJlcG9ydFsnUElEJ10gPT0gY2hpbGRyZW5bMV0ucGlkCiAgICAob3V0IC8gJ0NPTVBMRVRFLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChzdGF0dXM9J0JPVEhfT1BUSU9OQUxfQ09NUE9ORU5UU19DT01QTEVURScsIHdvcmtlcnM9cmVjb3JkcywKICAgICAgICBvdmVybGFwX2FjdGl2ZV9zYW1wbGVzPW92ZXJsYXBfc2FtcGxlcywgYWN0dWFsX2NvbXB1dGVfb3ZlcmxhcF9vYnNlcnZlZD1vdmVybGFwX3NhbXBsZXMgPiAwLAogICAgICAgIG1pc3Npbmdfc2FtcGxlX2NhdmVhdD1Ob25lIGlmIG92ZXJsYXBfc2FtcGxlcyBlbHNlICdObyBvdmVybGFwcGluZyBhY3RpdmUgc2FtcGxlIGNhcHR1cmVkOyBubyBjbGFpbSBvZiBtZWFzdXJlZCBjb21wdXRlIG92ZXJsYXAuJywKICAgICAgICB3YWxsX3NlY29uZHM9dGltZS5tb25vdG9uaWMoKS1zdGFydGVkLCBhdXRvbWF0aWNfcmV0cnk9RmFsc2UpLCBpbmRlbnQ9MikpCmZpbmFsbHk6CiAgICBmb3IgY2hpbGQgaW4gY2hpbGRyZW46CiAgICAgICAgaWYgY2hpbGQucG9sbCgpIGlzIE5vbmU6CiAgICAgICAgICAgIG9zLmtpbGxwZyhjaGlsZC5waWQsIHNpZ25hbC5TSUdURVJNKQogICAgZm9yIGNoaWxkIGluIGNoaWxkcmVuOgogICAgICAgIHRyeToKICAgICAgICAgICAgY2hpbGQud2FpdCh0aW1lb3V0PTEwKQogICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgICAgICBvcy5raWxscGcoY2hpbGQucGlkLCBzaWduYWwuU0lHS0lMTCkKICAgICAgICAgICAgY2hpbGQud2FpdCgpCiAgICBmb3IgbG9nIGluIGxvZ3M6CiAgICAgICAgbG9nLmNsb3NlKCkK\",\"run_candidate_optional.py\":\"IiIiT2ZmbGluZSBzZXF1ZW50aWFsIGJhc2VsaW5lLCBmaXhlZCB0YWJ1bGFyIG1lbWJlcnMgYW5kIENQVSBmaW5hbGl6YXRpb24uIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgphc3NlcnQgb3MubmFtZSE9J250JyBhbmQgUGF0aCgnL2thZ2dsZS9pbnB1dCcpLmlzX2RpcigpLCdLYWdnbGUgZXhlY3V0aW9uIG9ubHknCmhlcmU9UGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApjb25maWc9anNvbi5sb2FkcygoaGVyZS8ncnVudGltZV9jb25maWcuanNvbicpLnJlYWRfdGV4dCgpKQphc3NlcnQgY29uZmlnWydyZWZlcmVuY2Vfc2NvcmVkX3NjcmlwdF92ZXJzaW9uX2lkJ109PTM1OTMxMzM0OAphc3NlcnQgY29uZmlnWydjYW5kaWRhdGVfYWRtaXR0ZWQnXSwgJ0NhbmRpZGF0ZSBhZG1pc3Npb24gcmVxdWlyZWQnCmFzc2VydCBjb25maWdbJ3Byb3RlY3RlZF93ZWlnaHQnXT09LjEwIGFuZCBjb25maWdbJ3Jhd19leHRlbnNpb24nXSA9PSAnc21vbF9jZW50ZXJfYWRkJwphc3NlcnQgbm90IGNvbmZpZ1snc2Vjb25kX2NvbHVtbl9iYWxhbmNlJ10gYW5kIGNvbmZpZ1sncHJvYmFiaWxpdHlfdHJhbnNmb3JtJ109PSdub25lJwptYXRjaGVzPWxpc3QoUGF0aCgnL2thZ2dsZS9pbnB1dCcpLnJnbG9iKGNvbmZpZ1sndGFicGZuJ11bJ2NoZWNrcG9pbnRfZmlsZW5hbWUnXSkpCmFzc2VydCBsZW4obWF0Y2hlcyk9PTEsJ0F0dGFjaCB0aGUgZXhhY3QgcmV2aWV3ZWQgb2ZmbGluZSB0YWJ1bGFyIGFzc2V0IHBhY2thZ2UnCmFzc2V0PW1hdGNoZXNbMF0ucGFyZW50Cm1hbmlmZXN0PWpzb24ubG9hZHMoKGFzc2V0LydydW50aW1lX21hbmlmZXN0Lmpzb24nKS5yZWFkX3RleHQoKSkKYXNzZXJ0IG1hbmlmZXN0WydzdGF0dXMnXT09J0NQVV9TRVRVUF9DT01QTEVURV9OT19NT0RFTF9FWEVDVVRJT04nCmFzc2VydCBtYW5pZmVzdFsnZ2xvYmFsX2Vudmlyb25tZW50X3VuY2hhbmdlZCddIGFuZCBub3QgbWFuaWZlc3RbJ21vZGVsX2ZpdCddIGFuZCBub3QgbWFuaWZlc3RbJ21vZGVsX3ByZWRpY3QnXQphc3NlcnQgbWFuaWZlc3RbJ2NoZWNrcG9pbnRfc2hhMjU2J109PWNvbmZpZ1sndGFicGZuJ11bJ2NoZWNrcG9pbnRfc2hhMjU2J10KYXNzZXJ0IG1hdGNoZXNbMF0uc3RhdCgpLnN0X3NpemU9PWNvbmZpZ1sndGFicGZuJ11bJ2NoZWNrcG9pbnRfYnl0ZXMnXQp3aXRoIG1hdGNoZXNbMF0ub3BlbigncmInKSBhcyBzdHJlYW06CiAgICBhc3NlcnQgaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sJ3NoYTI1NicpLmhleGRpZ2VzdCgpPT1jb25maWdbJ3RhYnBmbiddWydjaGVja3BvaW50X3NoYTI1NiddCmxvY2s9YXNzZXQvJ3J1bnRpbWUtYWRkb24tbG9jay50eHQnCndoZWVscz1hc3NldAphc3NlcnQgbG9jay5pc19maWxlKCkgYW5kIGFueSh3aGVlbHMuZ2xvYignKi53aGwnKSksJ1Jldmlld2VkIG9mZmxpbmUgYWRkb24gbG9jay93aGVlbHMgYXJlIHJlcXVpcmVkJwpmb3IgaXRlbSBpbiBtYW5pZmVzdFsnc2VsZWN0ZWRfYWRkb25zJ10udmFsdWVzKCk6CiAgICB3aXRoICh3aGVlbHMvaXRlbVsnd2hlZWwnXSkub3BlbigncmInKSBhcyBzdHJlYW06CiAgICAgICAgYXNzZXJ0IGhhc2hsaWIuZmlsZV9kaWdlc3Qoc3RyZWFtLCdzaGEyNTYnKS5oZXhkaWdlc3QoKT09aXRlbVsnc2hhMjU2J10KZm9yIGxpbmUgaW4gbG9jay5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCk6CiAgICBpZiAnPT0nIGluIGxpbmUgYW5kIG5vdCBsaW5lLmxzdHJpcCgpLnN0YXJ0c3dpdGgoJyMnKToKICAgICAgICBwYWNrYWdlPXJlLnNwbGl0KHInPT0nLGxpbmUuc3RyaXAoKSxtYXhzcGxpdD0xKVswXS5sb3dlcigpLnJlcGxhY2UoJ18nLCctJykKICAgICAgICBhc3NlcnQgcGFja2FnZSBub3QgaW4gKCd0b3JjaCcsJ3RyYW5zZm9ybWVycycsJ3BlZnQnLCdhY2NlbGVyYXRlJywnc2FmZXRlbnNvcnMnKQogICAgICAgIGFzc2VydCBub3QgcGFja2FnZS5zdGFydHN3aXRoKCdudmlkaWEtJyksJ1JldXNlIHRoZSBvcmlnaW5hbCBDVURBL1RvcmNoIHJ1bnRpbWUnClBhdGgoJy9rYWdnbGUvdGVtcCcpLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQp0ZW1wPVBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9J250bC10YWJ1bGFyLScsZGlyPScva2FnZ2xlL3RlbXAnKSkKc2l0ZT10ZW1wLydzaXRlJwpiYXNlX2Vudj1vcy5lbnZpcm9uLmNvcHkoKQpiYXNlX2VudlsnTlRMX1RBQlVMQVJfVEVNUCddPXN0cih0ZW1wKQpiYXNlX2VudlsnTlRMX1RBQlVMQVJfQVNTRVQnXT1zdHIoYXNzZXQpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwnLW0nLCdwaXAnLCdpbnN0YWxsJywnLS1uby1pbmRleCcsJy0tZmluZC1saW5rcycsc3RyKHdoZWVscyksCiAgICAnLS1uby1kZXBzJywnLS1yZXF1aXJlLWhhc2hlcycsJy0tdGFyZ2V0JyxzdHIoc2l0ZSksJy1yJyxzdHIobG9jayldLAogICAgY2hlY2s9VHJ1ZSx0aW1lb3V0PWNvbmZpZ1sncGFja2FnZV9pbnN0YWxsX3RpbWVvdXRfc2Vjb25kcyddKQp0YWJ1bGFyX2Vudj1iYXNlX2Vudi5jb3B5KCkKdGFidWxhcl9lbnZbJ1BZVEhPTlBBVEgnXT1zdHIoc2l0ZSkKdGFidWxhcl9lbnZbJ0hGX0hVQl9PRkZMSU5FJ109JzEnCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwnLWMnLAogICAgImltcG9ydCBqc29uLCBpbXBvcnRsaWIubWV0YWRhdGEgYXMgbTsgZnJvbSBwYXRobGliIGltcG9ydCBQYXRoOyBpbXBvcnQgbnVtcHksIHBhbmRhcywgc2NpcHksIHNrbGVhcm4sIHRvcmNoOyBmcm9tIHRhYnBmbiBpbXBvcnQgVGFiUEZOQ2xhc3NpZmllcjsgZnJvbSB0YWJwZm4uY29uc3RhbnRzIGltcG9ydCBNb2RlbFZlcnNpb247IGFjdHVhbD1kaWN0KG51bXB5PW51bXB5Ll9fdmVyc2lvbl9fLCBwYW5kYXM9cGFuZGFzLl9fdmVyc2lvbl9fLCBzY2lweT1zY2lweS5fX3ZlcnNpb25fXywgc2tsZWFybj1za2xlYXJuLl9fdmVyc2lvbl9fLCB0b3JjaD10b3JjaC5fX3ZlcnNpb25fXywgdGFicGZuPW0udmVyc2lvbigndGFicGZuJykpOyBleHBlY3RlZD0iK3JlcHIoY29uZmlnWyd2ZXJpZmllZF9ydW50aW1lX2ltcG9ydHMnXSkrIjsgYXNzZXJ0IGFjdHVhbD09ZXhwZWN0ZWQsIChhY3R1YWwsZXhwZWN0ZWQpOyBhc3NlcnQgTW9kZWxWZXJzaW9uLlYzXzU7IFBhdGgoJy9rYWdnbGUvd29ya2luZy90YWJ1bGFyX2Vudmlyb25tZW50X2F1ZGl0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChzdGF0dXM9J1ZFUklGSUVEX09GRkxJTkVfUlVOVElNRV9JTVBPUlRTJyxpbXBvcnRzPWFjdHVhbCxkaWFnbm9zdGljX3NjcmlwdF92ZXJzaW9uX2lkPTM1OTE2MjM2OCksaW5kZW50PTIpKTsgcHJpbnQoJ09mZmxpbmUgVGFiUEZOIGltcG9ydHMgcGFzc2VkJywgZmx1c2g9VHJ1ZSkiXSwKICAgIGVudj10YWJ1bGFyX2VudixjaGVjaz1UcnVlLHRpbWVvdXQ9OTApCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSxzdHIoaGVyZS8ncnVuX2Jhc2VsaW5lLnB5JyldLGN3ZD0nL2thZ2dsZS93b3JraW5nJyxlbnY9YmFzZV9lbnYsCiAgICBjaGVjaz1UcnVlLHRpbWVvdXQ9Y29uZmlnWydiYXNlbGluZV90aW1lb3V0X3NlY29uZHMnXSkKYmFzZWxpbmU9UGF0aCgnL2thZ2dsZS93b3JraW5nL3N1Ym1pc3Npb24ucGFycXVldCcpCmFzc2VydCBiYXNlbGluZS5pc19maWxlKCkKY29udHJvbD1QYXRoKCcva2FnZ2xlL3dvcmtpbmcvQjU1Nl9nYXRlZF9jb250cm9sLnBhcnF1ZXQnKQphc3NlcnQgbm90IGNvbnRyb2wuZXhpc3RzKCkKYmFzZWxpbmUucmVuYW1lKGNvbnRyb2wpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSxzdHIoaGVyZS8ndGFidWxhcl93b3JrZXIucHknKSwncHJlcGFyZSddLGVudj10YWJ1bGFyX2VudiwKICAgIGNoZWNrPVRydWUsdGltZW91dD1jb25maWdbJ3ByZXBhcmVfdGltZW91dF9zZWNvbmRzJ10pCmJhc2VfZW52WydOVExfVEFCVUxBUl9TSVRFJ109c3RyKHNpdGUpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwnLXUnLHN0cihoZXJlLydydW5fcGFyYWxsZWwucHknKV0sZW52PWJhc2VfZW52LGNoZWNrPVRydWUsCiAgICB0aW1lb3V0PW1heChjb25maWdbJ21vZGVsX3RpbWVvdXRfc2Vjb25kcyddLDM2MDApKzEyMCkKc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLHN0cihoZXJlLydmaW5pc2hfY2VudGVyX3Byb3RlY3RlZC5weScpXSxlbnY9YmFzZV9lbnYsY2hlY2s9VHJ1ZSx0aW1lb3V0PTEyMCkKYXNzZXJ0IFBhdGgoJy9rYWdnbGUvd29ya2luZy9zdWJtaXNzaW9uLnBhcnF1ZXQnKS5pc19maWxlKCkKYXNzZXJ0IGpzb24ubG9hZHMoUGF0aCgnL2thZ2dsZS93b3JraW5nL3Byb3RlY3RlZF9yZXNpZHVhbF9hdWRpdC5qc29uJykucmVhZF90ZXh0KCkpWydzdGF0dXMnXT09J1NNT0xfQ0VOVEVSX0FERF9QUk9URUNURURfUEZOMTBfUFJFRElDVElPTlNfQ09NUExFVEUn\",\"actual-diagnostic-audit.json\":\"ew0KICAibm90ZWJvb2tfc2hhMjU2IjogIjhiOTU1NzNlMGZjNjFlNjFlZTRhZTk2ZjQ0MzExMmNjZmY5YTJjMTBkZjQ5MjllYzY2OWQwZTliM2M5OGI3N2QiLA0KICAidGVsZW1ldHJ5X3NhbXBsZXMiOiA0OCwNCiAgInRhcmdldHNfbG9hZGVkIjogZmFsc2UsDQogICJjb21wYWN0X21hbmlmZXN0X3NoYTI1NiI6ICIxNzcyNzAzZmFhZWU1ZDJiMzBlNzJmZTM3ZDA4MDJjMGMyMGVhYjIzMzJlNGQ2N2EwNjFjZGE2OTEyZmI2MTFhIiwNCiAgImxvY2FsX21vZGVsX2V4ZWN1dGlvbiI6IGZhbHNlLA0KICAib3B0aW1pemVyX2xvYWRlZCI6IGZhbHNlLA0KICAibm90ZWJvb2tfaWQiOiAxMzkzNTM0NTUsDQogICJ2ZXJzaW9uIjogMywNCiAgInJlZiI6ICJsaWdodG5pbmd2MDgvcHJpdmF0ZS1zbW9sbG0zLWZpbmFsLWluZmVyZW5jZS1kaWFnbm9zdGljIiwNCiAgInRyYWluaW5nX3VwZGF0ZXMiOiAwLA0KICAiY29tcG9uZW50X3NoYTI1NiI6ICJhYjFkZjk0Y2VhMTM1OGQ5NTEyNWVkOTlmNGZiZTk2MDZmN2UwNjkwY2E2YjY5MzhlYWQ0MTlmODg0MzQzZjQ2IiwNCiAgImNhbmRpZGF0ZV9vcl9wcm9kdWN0aW9uX2FkbWl0dGVkIjogZmFsc2UsDQogICJxdWFsaXR5X3Njb3JlX2NvbXB1dGVkIjogZmFsc2UsDQogICJyZXBvcnRzIjogew0KICAgICI1Ijogew0KICAgICAgInN0YXR1cyI6ICJQQVNTX0lORkVSRU5DRV9ESUFHTk9TVElDX09OTFkiLA0KICAgICAgInJvd3MiOiA1LA0KICAgICAgImNsYXNzZXMiOiBbDQogICAgICAgICJjbGF1ZGUtZmFibGUtNSIsDQogICAgICAgICJjbGF1ZGUtb3B1cy00LTYiLA0KICAgICAgICAiY2xhdWRlLW9wdXMtNC04IiwNCiAgICAgICAgImNsYXVkZS1vcHVzLTUiLA0KICAgICAgICAiY2xhdWRlLXNvbm5ldC01IiwNCiAgICAgICAgImdlbWluaS0zLjEtcHJvLXByZXZpZXciLA0KICAgICAgICAiZ2VtaW5pLTMuNi1mbGFzaCIsDQogICAgICAgICJnZW1pbmktMy43LWZsYXNoIiwNCiAgICAgICAgImdlbWluaS0zLjgtZmxhc2giLA0KICAgICAgICAiZ2xtLTUuMyIsDQogICAgICAgICJnbG0tNS4zLWZsYXNoIiwNCiAgICAgICAgImdwdC01LjUiLA0KICAgICAgICAiZ3B0LTUuNi1sdW5hIiwNCiAgICAgICAgImdwdC01LjYtc29sIiwNCiAgICAgICAgImdwdC01LjYtdGVycmEiLA0KICAgICAgICAiZ3B0LTYtYXN0cmEiLA0KICAgICAgICAia2ltaS1rMyIsDQogICAgICAgICJxd2VuMy44LW1heCINCiAgICAgIF0sDQogICAgICAiaWRzIjogWw0KICAgICAgICAiMDAxOGIzNzY1OTcyIiwNCiAgICAgICAgIjAwNDFhYzg2ZWU5MyIsDQogICAgICAgICIwMDVhMTkwODNlYWUiLA0KICAgICAgICAiMDA5MzE5NTk2ZjEwIiwNCiAgICAgICAgIjAwZDNlOGM5MjU5NiINCiAgICAgIF0sDQogICAgICAib3V0cHV0X3NoYXBlIjogWw0KICAgICAgICA1LA0KICAgICAgICAxOA0KICAgICAgXSwNCiAgICAgICJkdHlwZSI6ICJmbG9hdDMyIiwNCiAgICAgICJmaW5pdGUiOiB0cnVlLA0KICAgICAgImlkX29yZGVyX3ByZXNlcnZlZCI6IHRydWUsDQogICAgICAib2JzZXJ2ZWRfcXVlcnlfYmF0Y2hfc2l6ZXMiOiBbDQogICAgICAgIDMsDQogICAgICAgIDINCiAgICAgIF0sDQogICAgICAiY2Fub25pY2FsX2FuY2hvcl9iYXRjaF9zaXplIjogMywNCiAgICAgICJhbmNob3JfcmVwbGF5Ijogew0KICAgICAgICAiZmVhdHVyZV9tYXhhYnMiOiAwLjAsDQogICAgICAgICJwcm9iYWJpbGl0eV9tYXhhYnMiOiB7DQogICAgICAgICAgInBsYWluIjogMC4wLA0KICAgICAgICAgICJjZW50ZXIiOiAwLjANCiAgICAgICAgfSwNCiAgICAgICAgImZlYXR1cmVfcnRvbCI6IDFFLTA1LA0KICAgICAgICAiZmVhdHVyZV9hdG9sIjogMUUtMDUsDQogICAgICAgICJwcm9iYWJpbGl0eV9hdG9sIjogMUUtMDYNCiAgICAgIH0sDQogICAgICAiQ1VEQV9WSVNJQkxFX0RFVklDRVMiOiAiR1BVLWFlOGZiZGZlLTZiNGEtMzY0Yy0wOTAyLTFjZTg4OWNmNWIyZSIsDQogICAgICAibG9naWNhbF9ncHVfdXVpZCI6ICJhZThmYmRmZS02YjRhLTM2NGMtMDkwMi0xY2U4ODljZjViMmUiLA0KICAgICAgImxvZ2ljYWxfZ3B1X25hbWUiOiAiVGVzbGEgVDQiLA0KICAgICAgInByZWRpY3Rpb25fY3VkYV9ldmVudF9tcyI6IDQwMy41OTIyMjQxMjEwOTM3NSwNCiAgICAgICJsb2FkX2FuZF9hbmNob3Jfd2FsbF9zZWNvbmRzIjogOTAuODEwMzI4MDY5OTk5OTcsDQogICAgICAicGVha19hbGxvY2F0ZWRfYnl0ZXMiOiA2NDU0ODgyMzA0LA0KICAgICAgInBlYWtfcmVzZXJ2ZWRfYnl0ZXMiOiA2NjAzOTMxNjQ4LA0KICAgICAgImZ1bGxfY2hlY2twb2ludF9iaW5kaW5nIjogew0KICAgICAgICAiYWN0dWFsX2Z1bGxfUUFfcGFzc2VkIjogdHJ1ZSwNCiAgICAgICAgInBhaXJlZF9nYWluX2FkbWl0dGVkX2J5X3Jvb3QiOiBmYWxzZSwNCiAgICAgICAgIm5vdGVib29rX2lkIjogMTM5MzQ3NDcwLA0KICAgICAgICAidmVyc2lvbiI6IDEsDQogICAgICAgICJzY3JpcHRfdmVyc2lvbl9pZCI6IDM1OTU2NDczOCwNCiAgICAgICAgIndvcmtlcl9zaGEyNTYiOiAiNGRlYTAwYzIzMTNkYWQ0MjZiYTEwODFmM2I1NzE1NDBjODRmNmU1ZTQ0MWM1NzA3NTY5OWU2Zjk1NTdlOTM1YiIsDQogICAgICAgICJjaGVja3BvaW50X3JlbGF0aXZlIjogImZpbmFsL2ZpbmlzaGVkX2NoZWNrcG9pbnQiLA0KICAgICAgICAiY29tcGxldGVfc2hhMjU2IjogIjMxYjUyMjUwZjVlYWFiN2NiZDliNjAxNTJlNDQ3MTQ1NTU2NmI2ZDQwZmFlOTExZjBmMTAyNWFjODI3ODU0ZmIiLA0KICAgICAgICAicmVwb3J0X3NoYTI1NiI6ICI1M2JlNGQ5Mzk0YmQxNjFlZDBkNThiNzVlY2RjMzlmOGRhZmI0MGUxMmRlYzhkOWFkNWIzZTc2ZDQyY2FmNzg0IiwNCiAgICAgICAgInBvcnRhYmxlX2ZpbGVzX3NoYTI1NiI6IHsNCiAgICAgICAgICAiYWRhcHRlci9hZGFwdGVyX21vZGVsLnNhZmV0ZW5zb3JzIjogImFjYzc3ZjJlOTVkODZlMGVkYzU0MWFmYTQ2YzFhZjE2YWU5NTdiZWQ5NTU0MzVjNTMxOWE0ZGM4YTYwYjRjNDEiLA0KICAgICAgICAgICJhZGFwdGVyL2FkYXB0ZXJfY29uZmlnLmpzb24iOiAiZmU2NGRmYTg0NzExNDkwMjVlYzkxMGQ3YTY3M2MwM2VlNjczODI3MDQxZWZjMWNkYTA5ODhkZWMxNmI0MmFmMyIsDQogICAgICAgICAgImhlYWRzLnNhZmV0ZW5zb3JzIjogImNiYzUwY2EyYWJjNjZhNjAwMjQ1OWEyMzQzYjAyYzA1OThlOGQ5NGZlMDk1NDMwMzk1OWQyOTdhZTJjMmZlOGIiLA0KICAgICAgICAgICJyZWFkb3V0cy5ucHoiOiAiMDQxMTU5MTBiMjQ1NjljNWUwODY0MTM0MmIwY2VmZTkzYzgxNTY1MTVmOTNkMGI5MjBlZGUwMWRmYmEyZGRjMiIsDQogICAgICAgICAgInJlcGxheV9hbmNob3JzLm5weiI6ICJlNDlhNzQ3YTMyYTU1MjlmMmQwZTEzNjNhM2Y1YjdkOWUxY2EzNGQ3ZmMxMTQ1NzFlYzYyNDI3NzIwN2EyOWIyIiwNCiAgICAgICAgICAiYWRhcHRlci9SRUFETUUubWQiOiAiNDM2MDQ5YTA1ZjFmMDg4NjYxZGRjZGMzM2M0MmIxZTEzOWQ1ZWQxNDdkZTE2MmI3NGIxNzhjNzNkYjY4OWJmYiINCiAgICAgICAgfSwNCiAgICAgICAgImZ1bGxfcWFfYXVkaXRfc2hhMjU2IjogIjk3NDJlYzYzZmQxMzIxMWYwMDBjOGU0NjRjM2U4OGUxOTdlYjAzMmY4YTA4NWNiNTdiNDBlNWQ0ZmMxNTRhN2UiDQogICAgICB9LA0KICAgICAgImNvbXBvbmVudF9zaGEyNTYiOiAiYWIxZGY5NGNlYTEzNThkOTUxMjVlZDk5ZjRmYmU5NjA2ZjdlMDY5MGNhNmI2OTM4ZWFkNDE5Zjg4NDM0M2Y0NiIsDQogICAgICAib3duZWRfcGF0aF9zaGEyNTYiOiAiMjBkYWJhZmQzZjQyMzE5Njg2M2NlM2QxZmJkNGNhYThiMTk2YmQ1MTdhODg0MjI1NjE2MTVkYjAwM2QxZTYyMCIsDQogICAgICAid29ya2VyX3NoYTI1NiI6ICI2ZDNlM2ZmZTQxYzM5ZDczMTQxMmU2NjhmOTRhZmRiYTQxMWVhY2VlMzE0MjU1MzBhYzIyOTY2ZDg2NDI5ZmI3IiwNCiAgICAgICJvdXRwdXRfc2hhMjU2IjogIjVkZjZlYzk1MTM5N2UxZmNhMjEwZjRiMTU0ZjMzNDJhNzU3NzY3MWUzNjliNjBjODRmZmM2YWQ4NThjODYxZTEiLA0KICAgICAgInRyYWluaW5nX3VwZGF0ZXMiOiAwLA0KICAgICAgInRhcmdldHNfbG9hZGVkIjogZmFsc2UsDQogICAgICAib3B0aW1pemVyX2xvYWRlZCI6IGZhbHNlLA0KICAgICAgInF1YW50aXphdGlvbiI6IGZhbHNlLA0KICAgICAgImNhbmRpZGF0ZV9vcl9wcm9kdWN0aW9uX2FkbWl0dGVkIjogZmFsc2UsDQogICAgICAid2FsbF9zZWNvbmRzIjogOTEuMzk1MDYwNTkyDQogICAgfSwNCiAgICAiMyI6IHsNCiAgICAgICJzdGF0dXMiOiAiUEFTU19JTkZFUkVOQ0VfRElBR05PU1RJQ19PTkxZIiwNCiAgICAgICJyb3dzIjogMywNCiAgICAgICJjbGFzc2VzIjogWw0KICAgICAgICAiY2xhdWRlLWZhYmxlLTUiLA0KICAgICAgICAiY2xhdWRlLW9wdXMtNC02IiwNCiAgICAgICAgImNsYXVkZS1vcHVzLTQtOCIsDQogICAgICAgICJjbGF1ZGUtb3B1cy01IiwNCiAgICAgICAgImNsYXVkZS1zb25uZXQtNSIsDQogICAgICAgICJnZW1pbmktMy4xLXByby1wcmV2aWV3IiwNCiAgICAgICAgImdlbWluaS0zLjYtZmxhc2giLA0KICAgICAgICAiZ2VtaW5pLTMuNy1mbGFzaCIsDQogICAgICAgICJnZW1pbmktMy44LWZsYXNoIiwNCiAgICAgICAgImdsbS01LjMiLA0KICAgICAgICAiZ2xtLTUuMy1mbGFzaCIsDQogICAgICAgICJncHQtNS41IiwNCiAgICAgICAgImdwdC01LjYtbHVuYSIsDQogICAgICAgICJncHQtNS42LXNvbCIsDQogICAgICAgICJncHQtNS42LXRlcnJhIiwNCiAgICAgICAgImdwdC02LWFzdHJhIiwNCiAgICAgICAgImtpbWktazMiLA0KICAgICAgICAicXdlbjMuOC1tYXgiDQogICAgICBdLA0KICAgICAgImlkcyI6IFsNCiAgICAgICAgIjAwMThiMzc2NTk3MiIsDQogICAgICAgICIwMDQxYWM4NmVlOTMiLA0KICAgICAgICAiMDA1YTE5MDgzZWFlIg0KICAgICAgXSwNCiAgICAgICJvdXRwdXRfc2hhcGUiOiBbDQogICAgICAgIDMsDQogICAgICAgIDE4DQogICAgICBdLA0KICAgICAgImR0eXBlIjogImZsb2F0MzIiLA0KICAgICAgImZpbml0ZSI6IHRydWUsDQogICAgICAiaWRfb3JkZXJfcHJlc2VydmVkIjogdHJ1ZSwNCiAgICAgICJvYnNlcnZlZF9xdWVyeV9iYXRjaF9zaXplcyI6IFsNCiAgICAgICAgMw0KICAgICAgXSwNCiAgICAgICJjYW5vbmljYWxfYW5jaG9yX2JhdGNoX3NpemUiOiAzLA0KICAgICAgImFuY2hvcl9yZXBsYXkiOiB7DQogICAgICAgICJmZWF0dXJlX21heGFicyI6IDAuMCwNCiAgICAgICAgInByb2JhYmlsaXR5X21heGFicyI6IHsNCiAgICAgICAgICAicGxhaW4iOiAwLjAsDQogICAgICAgICAgImNlbnRlciI6IDAuMA0KICAgICAgICB9LA0KICAgICAgICAiZmVhdHVyZV9ydG9sIjogMUUtMDUsDQogICAgICAgICJmZWF0dXJlX2F0b2wiOiAxRS0wNSwNCiAgICAgICAgInByb2JhYmlsaXR5X2F0b2wiOiAxRS0wNg0KICAgICAgfSwNCiAgICAgICJDVURBX1ZJU0lCTEVfREVWSUNFUyI6ICJHUFUtMGRiZDA4NTYtOTkyNi0zMjU5LWU4MzktMTg2OWE2NTQ1ODQ2IiwNCiAgICAgICJsb2dpY2FsX2dwdV91dWlkIjogIjBkYmQwODU2LTk5MjYtMzI1OS1lODM5LTE4NjlhNjU0NTg0NiIsDQogICAgICAibG9naWNhbF9ncHVfbmFtZSI6ICJUZXNsYSBUNCIsDQogICAgICAicHJlZGljdGlvbl9jdWRhX2V2ZW50X21zIjogMjU4LjE1MTA5MjUyOTI5NjksDQogICAgICAibG9hZF9hbmRfYW5jaG9yX3dhbGxfc2Vjb25kcyI6IDkwLjgxMDYxMTU4LA0KICAgICAgInBlYWtfYWxsb2NhdGVkX2J5dGVzIjogNjQ1NDg4MjMwNCwNCiAgICAgICJwZWFrX3Jlc2VydmVkX2J5dGVzIjogNjYwMzkzMTY0OCwNCiAgICAgICJmdWxsX2NoZWNrcG9pbnRfYmluZGluZyI6IHsNCiAgICAgICAgImFjdHVhbF9mdWxsX1FBX3Bhc3NlZCI6IHRydWUsDQogICAgICAgICJwYWlyZWRfZ2Fpbl9hZG1pdHRlZF9ieV9yb290IjogZmFsc2UsDQogICAgICAgICJub3RlYm9va19pZCI6IDEzOTM0NzQ3MCwNCiAgICAgICAgInZlcnNpb24iOiAxLA0KICAgICAgICAic2NyaXB0X3ZlcnNpb25faWQiOiAzNTk1NjQ3MzgsDQogICAgICAgICJ3b3JrZXJfc2hhMjU2IjogIjRkZWEwMGMyMzEzZGFkNDI2YmExMDgxZjNiNTcxNTQwYzg0ZjZlNWU0NDFjNTcwNzU2OTllNmY5NTU3ZTkzNWIiLA0KICAgICAgICAiY2hlY2twb2ludF9yZWxhdGl2ZSI6ICJmaW5hbC9maW5pc2hlZF9jaGVja3BvaW50IiwNCiAgICAgICAgImNvbXBsZXRlX3NoYTI1NiI6ICIzMWI1MjI1MGY1ZWFhYjdjYmQ5YjYwMTUyZTQ0NzE0NTU1NjZiNmQ0MGZhZTkxMWYwZjEwMjVhYzgyNzg1NGZiIiwNCiAgICAgICAgInJlcG9ydF9zaGEyNTYiOiAiNTNiZTRkOTM5NGJkMTYxZWQwZDU4Yjc1ZWNkYzM5ZjhkYWZiNDBlMTJkZWM4ZDlhZDViM2U3NmQ0MmNhZjc4NCIsDQogICAgICAgICJwb3J0YWJsZV9maWxlc19zaGEyNTYiOiB7DQogICAgICAgICAgImFkYXB0ZXIvYWRhcHRlcl9tb2RlbC5zYWZldGVuc29ycyI6ICJhY2M3N2YyZTk1ZDg2ZTBlZGM1NDFhZmE0NmMxYWYxNmFlOTU3YmVkOTU1NDM1YzUzMTlhNGRjOGE2MGI0YzQxIiwNCiAgICAgICAgICAiYWRhcHRlci9hZGFwdGVyX2NvbmZpZy5qc29uIjogImZlNjRkZmE4NDcxMTQ5MDI1ZWM5MTBkN2E2NzNjMDNlZTY3MzgyNzA0MWVmYzFjZGEwOTg4ZGVjMTZiNDJhZjMiLA0KICAgICAgICAgICJoZWFkcy5zYWZldGVuc29ycyI6ICJjYmM1MGNhMmFiYzY2YTYwMDI0NTlhMjM0M2IwMmMwNTk4ZThkOTRmZTA5NTQzMDM5NTlkMjk3YWUyYzJmZThiIiwNCiAgICAgICAgICAicmVhZG91dHMubnB6IjogIjA0MTE1OTEwYjI0NTY5YzVlMDg2NDEzNDJiMGNlZmU5M2M4MTU2NTE1ZjkzZDBiOTIwZWRlMDFkZmJhMmRkYzIiLA0KICAgICAgICAgICJyZXBsYXlfYW5jaG9ycy5ucHoiOiAiZTQ5YTc0N2EzMmE1NTI5ZjJkMGUxMzYzYTNmNWI3ZDllMWNhMzRkN2ZjMTE0NTcxZWM2MjQyNzcyMDdhMjliMiIsDQogICAgICAgICAgImFkYXB0ZXIvUkVBRE1FLm1kIjogIjQzNjA0OWEwNWYxZjA4ODY2MWRkY2RjMzNjNDJiMWUxMzlkNWVkMTQ3ZGUxNjJiNzRiMTc4YzczZGI2ODliZmIiDQogICAgICAgIH0sDQogICAgICAgICJmdWxsX3FhX2F1ZGl0X3NoYTI1NiI6ICI5NzQyZWM2M2ZkMTMyMTFmMDAwYzhlNDY0YzNlODhlMTk3ZWIwMzJmOGEwODVjYjU3YjQwZTVkNGZjMTU0YTdlIg0KICAgICAgfSwNCiAgICAgICJjb21wb25lbnRfc2hhMjU2IjogImFiMWRmOTRjZWExMzU4ZDk1MTI1ZWQ5OWY0ZmJlOTYwNmY3ZTA2OTBjYTZiNjkzOGVhZDQxOWY4ODQzNDNmNDYiLA0KICAgICAgIm93bmVkX3BhdGhfc2hhMjU2IjogIjIwZGFiYWZkM2Y0MjMxOTY4NjNjZTNkMWZiZDRjYWE4YjE5NmJkNTE3YTg4NDIyNTYxNjE1ZGIwMDNkMWU2MjAiLA0KICAgICAgIndvcmtlcl9zaGEyNTYiOiAiNmQzZTNmZmU0MWMzOWQ3MzE0MTJlNjY4Zjk0YWZkYmE0MTFlYWNlZTMxNDI1NTMwYWMyMjk2NmQ4NjQyOWZiNyIsDQogICAgICAib3V0cHV0X3NoYTI1NiI6ICJhNzNjYzc3YWMwZThiZTViNjkxNjMzMmNhMzYzMDQ3YzBiNDUzZjIyMTIwYmE4ZmYzNGU4OTMxOTA1MDg1MDRhIiwNCiAgICAgICJ0cmFpbmluZ191cGRhdGVzIjogMCwNCiAgICAgICJ0YXJnZXRzX2xvYWRlZCI6IGZhbHNlLA0KICAgICAgIm9wdGltaXplcl9sb2FkZWQiOiBmYWxzZSwNCiAgICAgICJxdWFudGl6YXRpb24iOiBmYWxzZSwNCiAgICAgICJjYW5kaWRhdGVfb3JfcHJvZHVjdGlvbl9hZG1pdHRlZCI6IGZhbHNlLA0KICAgICAgIndhbGxfc2Vjb25kcyI6IDkxLjI0MjQ3OTA2MDk5OTk4DQogICAgfQ0KICB9LA0KICAic3RhdHVzIjogIlBBU1NfQUNUVUFMX1NNT0xfRklOQUxfSU5GRVJFTkNFX0JPVEhfVDRfU1RSSUNUX09SSUdJTkFMX0FOQ0hPUlMiLA0KICAib3ZlcmxhcF9zYW1wbGVzIjogNCwNCiAgInNjcmlwdF92ZXJzaW9uX2lkIjogMzU5NTgxOTg4LA0KICAicm9vdCI6ICJEOlxcUHJvZ3JhbW1pbmdcXFB5dGhvbiBBSSBNTFxcS2FnZ2xlXFxuYW1lLXRoYXQtbGxtXFxleHBlcmltZW50c1xcc21vbGxtMy1sb3JhXFxpbmZlcmVuY2VcXHNtb2tlXFxyZXN1bHRzLXYzIiwNCiAgImFjdHVhbHNvdXJjZV9ieXRlY2hlY2tzIjogWw0KICAgIHsNCiAgICAgICJleGFjdCI6IHRydWUsDQogICAgICAic2hhMjU2IjogImFiMWRmOTRjZWExMzU4ZDk1MTI1ZWQ5OWY0ZmJlOTYwNmY3ZTA2OTBjYTZiNjkzOGVhZDQxOWY4ODQzNDNmNDYiLA0KICAgICAgInNvdXJjZSI6ICJjb21wb25lbnQucHkiDQogICAgfSwNCiAgICB7DQogICAgICAiZXhhY3QiOiB0cnVlLA0KICAgICAgInNoYTI1NiI6ICIyMGRhYmFmZDNmNDIzMTk2ODYzY2UzZDFmYmQ0Y2FhOGIxOTZiZDUxN2E4ODQyMjU2MTYxNWRiMDAzZDFlNjIwIiwNCiAgICAgICJzb3VyY2UiOiAib3duZWRfcGF0aC5weSINCiAgICB9LA0KICAgIHsNCiAgICAgICJleGFjdCI6IHRydWUsDQogICAgICAic2hhMjU2IjogIjhkNmVlODYzMWZmYTk5YmFhZjNmOWZkYjQ5ZTNkOGE0ZGJkNmRiZTIyODRlMmVhMTYxN2VhNTk2NTkwMjA1MmIiLA0KICAgICAgInNvdXJjZSI6ICJydW5fY29tcG9uZW50LnB5Ig0KICAgIH0sDQogICAgew0KICAgICAgImV4YWN0IjogdHJ1ZSwNCiAgICAgICJzaGEyNTYiOiAiNmQzZTNmZmU0MWMzOWQ3MzE0MTJlNjY4Zjk0YWZkYmE0MTFlYWNlZTMxNDI1NTMwYWMyMjk2NmQ4NjQyOWZiNyIsDQogICAgICAic291cmNlIjogIndvcmtlci5weSINCiAgICB9LA0KICAgIHsNCiAgICAgICJleGFjdCI6IHRydWUsDQogICAgICAic2hhMjU2IjogIjhmNjYyMDQ1ZTU3MTMxMTljOTI5Zjg5ZmE4N2QwN2ZkMzk3YzU4MThmOWY0Yjk4ZDFlYTExMzZlZDZiM2ZlNTAiLA0KICAgICAgInNvdXJjZSI6ICJydW5fZGlhZ25vc3RpYy5weSINCiAgICB9LA0KICAgIHsNCiAgICAgICJleGFjdCI6IHRydWUsDQogICAgICAic2hhMjU2IjogIjhmZDM4ODkyNTc5NDM0ZGFiYTliYjAwMDkxMTM3NjYzMzhmODkxYmFlMmFmMjc0ZGU4MzU0OTMzMDlmZmJiZTQiLA0KICAgICAgInNvdXJjZSI6ICJkaWFnbm9zdGljLXNwZWMuanNvbiINCiAgICB9DQogIF0sDQogICJoZWFkZXJzIjogew0KICAgICI1Ijogew0KICAgICAgImlkcy5ucHkiOiAieydkZXNjcic6ICc8VTEyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDUsKSwgfSIsDQogICAgICAicGxhaW5fbG9ncC5ucHkiOiAieydkZXNjcic6ICc8ZjQnLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoNSwgMTgpLCB9IiwNCiAgICAgICJjbGFzc2VzLm5weSI6ICJ7J2Rlc2NyJzogJzxVMjInLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoMTgsKSwgfSINCiAgICB9LA0KICAgICIzIjogew0KICAgICAgImlkcy5ucHkiOiAieydkZXNjcic6ICc8VTEyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDMsKSwgfSIsDQogICAgICAicGxhaW5fbG9ncC5ucHkiOiAieydkZXNjcic6ICc8ZjQnLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoMywgMTgpLCB9IiwNCiAgICAgICJjbGFzc2VzLm5weSI6ICJ7J2Rlc2NyJzogJzxVMjInLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoMTgsKSwgfSINCiAgICB9DQogIH0sDQogICJjaGVja2VkX3V0YyI6ICIyMDI2LTEwLTE4VDA0OjU4OjM3LjUyNDk2NThaIiwNCiAgInN0cmljdF9udW1lcmljYWxfYW5jaG9yX3JlcGxheV9wYXNzZWQiOiB0cnVlLA0KICAicGFyZW50X2NvbXBsZXRlIjogew0KICAgICJzdGF0dXMiOiAiUEFTU19CT1RIX0lORkVSRU5DRV9GSVhUVVJFU19PTkxZIiwNCiAgICAicm93cyI6IFsNCiAgICAgIDMsDQogICAgICA1DQogICAgXSwNCiAgICAiR1BVX21hcHBpbmciOiBbDQogICAgICB7DQogICAgICAgICJwaHlzaWNhbF9pbmRleCI6IDAsDQogICAgICAgICJ1dWlkIjogIkdQVS0wZGJkMDg1Ni05OTI2LTMyNTktZTgzOS0xODY5YTY1NDU4NDYiLA0KICAgICAgICAibmFtZSI6ICJUZXNsYSBUNCINCiAgICAgIH0sDQogICAgICB7DQogICAgICAgICJwaHlzaWNhbF9pbmRleCI6IDEsDQogICAgICAgICJ1dWlkIjogIkdQVS1hZThmYmRmZS02YjRhLTM2NGMtMDkwMi0xY2U4ODljZjViMmUiLA0KICAgICAgICAibmFtZSI6ICJUZXNsYSBUNCINCiAgICAgIH0NCiAgICBdLA0KICAgICJjYW5kaWRhdGVfb3JfcHJvZHVjdGlvbl9hZG1pdHRlZCI6IGZhbHNlLA0KICAgICJ0cmFpbmluZ191cGRhdGVzIjogMCwNCiAgICAid2FsbF9zZWNvbmRzIjogOTYuNDQ1MDQ0NTAwMDAwMDUNCiAgfSwNCiAgImFjdHVhbF9jb21wYWN0X2RhdGFzZXQiOiAibGlnaHRuaW5ndjA4L250bC1zbW9sbG0zLWZpbmFsLWluZmVyZW5jZS1hc3NldHMvMSIsDQogICJib3RoX3BoeXNpY2FsX1Q0X2ZvcndhcmRfQ1VEQV9ldmVudHNfcHJvdmVuIjogdHJ1ZSwNCiAgInRlbGVtZXRyeV9kZXJpdmVkX2Zyb21fYWN0dWFsX252aWRpYV9zbWlfY3N2IjogdHJ1ZSwNCiAgIm1heF9zYW1wbGVkX3V0aWxpemF0aW9uX3BlcmNlbnQiOiB7DQogICAgIjEiOiA5LjAsDQogICAgIjAiOiA4LjANCiAgfSwNCiAgIm1heF9zYW1wbGVkX21lbW9yeV9taWIiOiB7DQogICAgIjEiOiA2NDMxLjAsDQogICAgIjAiOiA2NDMxLjANCiAgfSwNCiAgIkNVREFfZXZlbnRfcHJvb2Zfbm90X2RlcGVuZGVudF9vbl8yc2VjX3NhbXBsZXIiOiB0cnVlDQp9DQo=\",\"selection-evidence.json\":\"ew0KICAic2VsZWN0ZWRfYnlfcm9vdCI6IHRydWUsDQogICJzZWxlY3RlZF9icmFuY2giOiAic21vbF9jZW50ZXJfYWRkX3Byb3RlY3RlZF9wZm4xMCIsDQogICJub3RlYm9va19pZCI6IDEzOTM1NTEyOSwNCiAgInZlcnNpb24iOiAyLA0KICAic2NyaXB0X3ZlcnNpb25faWQiOiAzNTk1ODM0MjEsDQogICJyZXBvcnRfc2hhMjU2IjogImYyNjI4NWQwZTQ2M2VkODE0ZGU0NjNjYzE5MDU3NGQ3ZDk4MzkyM2JiNDA4N2EzOGU0MjM4NjZlOGVjNzM2NzYiLA0KICAiYWN0dWFsX2FybSI6IHsNCiAgICAiYXJtIjogIlNtb2xfY2VudGVyX2FkZF9QRk4xMCIsDQogICAgImNvbnRleHRfQkEiOiBbDQogICAgICAwLjUyMTA3Mjc5NjkzNDg2NTksDQogICAgICAwLjU1MDUxMDg1NTY4MzI2OTQsDQogICAgICAwLjU5ODg1MDU3NDcxMjY0MzYNCiAgICBdLA0KICAgICJtZWFuX0JBIjogMC41NTY4MTE0MDkxMTAyNTk3LA0KICAgICJjb21wYXJpc29ucyI6IHsNCiAgICAgICJCNTU2Ijogew0KICAgICAgICAiY29udHJvbF9jb250ZXh0X0JBIjogWw0KICAgICAgICAgIDAuNTExODEzNTM3Njc1NjA2NywNCiAgICAgICAgICAwLjU0MTE4NzczOTQ2MzYwMTQsDQogICAgICAgICAgMC41OTY5MzQ4NjU5MDAzODMNCiAgICAgICAgXSwNCiAgICAgICAgImNvbnRyb2xfbWVhbl9CQSI6IDAuNTQ5OTc4NzE0MzQ2NTMwNiwNCiAgICAgICAgImNvbnRleHRfZGVsdGEiOiBbDQogICAgICAgICAgMC4wMDkyNTkyNTkyNTkyNTkyNTcsDQogICAgICAgICAgMC4wMDkzMjMxMTYyMTk2Njc5NDUsDQogICAgICAgICAgMC4wMDE5MTU3MDg4MTIyNjA1MzQxDQogICAgICAgIF0sDQogICAgICAgICJtZWFuX2RlbHRhIjogMC4wMDY4MzI2OTQ3NjM3MjkyNDYsDQogICAgICAgICJjbGFzc3dpc2VfY29udGV4dF9yZWNhbGxfZGVsdGEiOiBbDQogICAgICAgICAgWw0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgLTAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wNjY2NjY2NjY2NjY2NjY2NSwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2DQogICAgICAgICAgXSwNCiAgICAgICAgICBbDQogICAgICAgICAgICAwLjAzNDQ4Mjc1ODYyMDY4OTY0LA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjA2NjY2NjY2NjY2NjY2NjY1LA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzOCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMA0KICAgICAgICAgIF0sDQogICAgICAgICAgWw0KICAgICAgICAgICAgMC4wMzQ0ODI3NTg2MjA2ODk2MSwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMA0KICAgICAgICAgIF0NCiAgICAgICAgXSwNCiAgICAgICAgImNsYXNzd2lzZV9tZWFuX3JlY2FsbF9kZWx0YSI6IFsNCiAgICAgICAgICAwLjAyMjk4ODUwNTc0NzEyNjQyLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEwOCwNCiAgICAgICAgICAwLjAyMjIyMjIyMjIyMjIyMjIxNiwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEyNywNCiAgICAgICAgICAtMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAyMjIyMjIyMjIyMjIyMjIxNiwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEwOCwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEwOCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMDExMTExMTExMTExMTExMTA4DQogICAgICAgIF0sDQogICAgICAgICJjb3JyZWN0ZWRfcGVyX2NvbnRleHQiOiBbDQogICAgICAgICAgNywNCiAgICAgICAgICA1LA0KICAgICAgICAgIDINCiAgICAgICAgXSwNCiAgICAgICAgImJyb2tlbl9wZXJfY29udGV4dCI6IFsNCiAgICAgICAgICAyLA0KICAgICAgICAgIDAsDQogICAgICAgICAgMQ0KICAgICAgICBdDQogICAgICB9LA0KICAgICAgInByb3RlY3RlZF9QRk5fb25seSI6IHsNCiAgICAgICAgImNvbnRyb2xfY29udGV4dF9CQSI6IFsNCiAgICAgICAgICAwLjUyMTA3Mjc5NjkzNDg2NTksDQogICAgICAgICAgMC41NDY3NDMyOTUwMTkxNTcsDQogICAgICAgICAgMC41OTY5MzQ4NjU5MDAzODMxDQogICAgICAgIF0sDQogICAgICAgICJjb250cm9sX21lYW5fQkEiOiAwLjU1NDkxNjk4NTk1MTQ2ODcsDQogICAgICAgICJjb250ZXh0X2RlbHRhIjogWw0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAwMzc2NzU2MDY2NDExMjM5LA0KICAgICAgICAgIDAuMDAxOTE1NzA4ODEyMjYwNTI3OA0KICAgICAgICBdLA0KICAgICAgICAibWVhbl9kZWx0YSI6IDAuMDAxODk0NDIzMTU4NzkwOTcyNywNCiAgICAgICAgImNsYXNzd2lzZV9jb250ZXh0X3JlY2FsbF9kZWx0YSI6IFsNCiAgICAgICAgICBbDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjANCiAgICAgICAgICBdLA0KICAgICAgICAgIFsNCiAgICAgICAgICAgIDAuMDM0NDgyNzU4NjIwNjg5NjQsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzgsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAtMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjANCiAgICAgICAgICBdLA0KICAgICAgICAgIFsNCiAgICAgICAgICAgIDAuMDM0NDgyNzU4NjIwNjg5NjEsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIC0wLjAzMzMzMzMzMzMzMzMzMzQ0LA0KICAgICAgICAgICAgMC4wDQogICAgICAgICAgXQ0KICAgICAgICBdLA0KICAgICAgICAiY2xhc3N3aXNlX21lYW5fcmVjYWxsX2RlbHRhIjogWw0KICAgICAgICAgIDAuMDIyOTg4NTA1NzQ3MTI2NDIsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEwOCwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEyNywNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIC0wLjAxMTExMTExMTExMTExMTE0NiwNCiAgICAgICAgICAwLjANCiAgICAgICAgXSwNCiAgICAgICAgImNvcnJlY3RlZF9wZXJfY29udGV4dCI6IFsNCiAgICAgICAgICAwLA0KICAgICAgICAgIDMsDQogICAgICAgICAgMg0KICAgICAgICBdLA0KICAgICAgICAiYnJva2VuX3Blcl9jb250ZXh0IjogWw0KICAgICAgICAgIDAsDQogICAgICAgICAgMSwNCiAgICAgICAgICAxDQogICAgICAgIF0NCiAgICAgIH0sDQogICAgICAiR2VtbWFfb25seSI6IHsNCiAgICAgICAgImNvbnRyb2xfY29udGV4dF9CQSI6IFsNCiAgICAgICAgICAwLjUxMzY2NTM4OTUyNzQ1ODMsDQogICAgICAgICAgMC41NDEyNTE1OTY0MjQwMTAxLA0KICAgICAgICAgIDAuNTk4ODUwNTc0NzEyNjQzNg0KICAgICAgICBdLA0KICAgICAgICAiY29udHJvbF9tZWFuX0JBIjogMC41NTEyNTU4NTM1NTQ3MDQxLA0KICAgICAgICAiY29udGV4dF9kZWx0YSI6IFsNCiAgICAgICAgICAwLjAwNzQwNzQwNzQwNzQwNzQwNiwNCiAgICAgICAgICAwLjAwOTI1OTI1OTI1OTI1OTI1NywNCiAgICAgICAgICAwLjANCiAgICAgICAgXSwNCiAgICAgICAgIm1lYW5fZGVsdGEiOiAwLjAwNTU1NTU1NTU1NTU1NTU1NCwNCiAgICAgICAgImNsYXNzd2lzZV9jb250ZXh0X3JlY2FsbF9kZWx0YSI6IFsNCiAgICAgICAgICBbDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAtMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAtMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYNCiAgICAgICAgICBdLA0KICAgICAgICAgIFsNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wNjY2NjY2NjY2NjY2NjY2NSwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjANCiAgICAgICAgICBdLA0KICAgICAgICAgIFsNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIC0wLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wDQogICAgICAgICAgXQ0KICAgICAgICBdLA0KICAgICAgICAiY2xhc3N3aXNlX21lYW5fcmVjYWxsX2RlbHRhIjogWw0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAtMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAtMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMjIyMjIyMjIyMjIyMjIyMTYsDQogICAgICAgICAgMC4wMjIyMjIyMjIyMjIyMjIyMTYsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMDExMTExMTExMTExMTExMTA4LA0KICAgICAgICAgIDAuMDExMTExMTExMTExMTExMTA4LA0KICAgICAgICAgIDAuMDExMTExMTExMTExMTExMTA4DQogICAgICAgIF0sDQogICAgICAgICJjb3JyZWN0ZWRfcGVyX2NvbnRleHQiOiBbDQogICAgICAgICAgNywNCiAgICAgICAgICA2LA0KICAgICAgICAgIDINCiAgICAgICAgXSwNCiAgICAgICAgImJyb2tlbl9wZXJfY29udGV4dCI6IFsNCiAgICAgICAgICAzLA0KICAgICAgICAgIDEsDQogICAgICAgICAgMg0KICAgICAgICBdDQogICAgICB9LA0KICAgICAgIlNtb2xfY2VudGVyX29ubHkiOiB7DQogICAgICAgICJjb250cm9sX2NvbnRleHRfQkEiOiBbDQogICAgICAgICAgMC41MTE4MTM1Mzc2NzU2MDY3LA0KICAgICAgICAgIDAuNTQ2ODA3MTUxOTc5NTY1NiwNCiAgICAgICAgICAwLjYwMDcwMjQyNjU2NDQ5NTQNCiAgICAgICAgXSwNCiAgICAgICAgImNvbnRyb2xfbWVhbl9CQSI6IDAuNTUzMTA3NzA1NDA2NTU2LA0KICAgICAgICAiY29udGV4dF9kZWx0YSI6IFsNCiAgICAgICAgICAwLjAwOTI1OTI1OTI1OTI1OTI1NywNCiAgICAgICAgICAwLjAwMzcwMzcwMzcwMzcwMzcwMywNCiAgICAgICAgICAtMC4wMDE4NTE4NTE4NTE4NTE4NTE1DQogICAgICAgIF0sDQogICAgICAgICJtZWFuX2RlbHRhIjogMC4wMDM3MDM3MDM3MDM3MDM3MDMsDQogICAgICAgICJjbGFzc3dpc2VfY29udGV4dF9yZWNhbGxfZGVsdGEiOiBbDQogICAgICAgICAgWw0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgLTAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wLA0KICAgICAgICAgICAgMC4wNjY2NjY2NjY2NjY2NjY2NSwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2DQogICAgICAgICAgXSwNCiAgICAgICAgICBbDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMDY2NjY2NjY2NjY2NjY2NjUsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAtMC4wMzMzMzMzMzMzMzMzMzMzMjYsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjAsDQogICAgICAgICAgICAwLjANCiAgICAgICAgICBdLA0KICAgICAgICAgIFsNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIC0wLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIC0wLjAzMzMzMzMzMzMzMzMzMzMyNiwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMCwNCiAgICAgICAgICAgIDAuMDMzMzMzMzMzMzMzMzMzMzI2LA0KICAgICAgICAgICAgMC4wDQogICAgICAgICAgXQ0KICAgICAgICBdLA0KICAgICAgICAiY2xhc3N3aXNlX21lYW5fcmVjYWxsX2RlbHRhIjogWw0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMjIyMjIyMjIyMjIyMjIyMTYsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIC0wLjAyMjIyMjIyMjIyMjIyMjIxNiwNCiAgICAgICAgICAtMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wLA0KICAgICAgICAgIDAuMDIyMjIyMjIyMjIyMjIyMjE2LA0KICAgICAgICAgIDAuMCwNCiAgICAgICAgICAwLjAxMTExMTExMTExMTExMTEwOCwNCiAgICAgICAgICAwLjAsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgsDQogICAgICAgICAgMC4wMTExMTExMTExMTExMTExMDgNCiAgICAgICAgXSwNCiAgICAgICAgImNvcnJlY3RlZF9wZXJfY29udGV4dCI6IFsNCiAgICAgICAgICA3LA0KICAgICAgICAgIDMsDQogICAgICAgICAgMQ0KICAgICAgICBdLA0KICAgICAgICAiYnJva2VuX3Blcl9jb250ZXh0IjogWw0KICAgICAgICAgIDIsDQogICAgICAgICAgMSwNCiAgICAgICAgICAyDQogICAgICAgIF0NCiAgICAgIH0NCiAgICB9LA0KICAgICJmYW1pbHlfcmV2ZXJ0c19wZXJfY29udGV4dCI6IFsNCiAgICAgIDIsDQogICAgICAyLA0KICAgICAgMA0KICAgIF0NCiAgfSwNCiAgImhlbHBlcl9zaGEyNTYiOiAiOWQ5ZjhhODliOTI5MzUwNzUxMzFkMWUzYTI2YTc5NDhjYmJlNmViMTQ5NWQ1Y2E4MDgzMjlhNWVjZGEzMTg0YSIsDQogICJtYWluMThfc2hhMjU2IjogIjgyYWIwNzQzMmE2M2Q2NzY3ZmUzYjk4ZTU4NDBkZDAzYzlkMzE2NmRjNjUzZWYyNDk4ZDg0MDg5NmRlM2M2MGIiLA0KICAiY2FuZGlkYXRlX2FkbWl0dGVkIjogZmFsc2UsDQogICJwdWJsaWNfYXNzZXRzX2F1ZGl0X3Bhc3NlZCI6IGZhbHNlLA0KICAiaW50ZXJwcmV0YXRpb24iOiAiUm9vdCBzZWxlY3Rpb24gZnJvbSBvbmUgcmV1c2VkNTM5LXJvdzMwLXByb21wdCBjb25kaXRpb25hbCByZWFkb3V0OyBub3QgZnVsbC1maXZlZm9sZCBPT0YsIGluZGVwZW5kZW50IGhvbGRvdXQsIENJIG9yIGxlYWRlcmJvYXJkIGZvcmVjYXN0Ig0KfQ==\",\"runtime_config.json\":\"ew0KICAicmVmZXJlbmNlX25vdGVib29rIjogIm9ic3RhY2xlZGV2ZWxvcGVyL2xvY2FsLWNhbmRpZGF0ZS1hZGFwdGl2ZS1jZW50ZXJlZC1zdHlsZWdyYW0iLA0KICAicmVmZXJlbmNlX3ZlcnNpb24iOiAyLA0KICAicmVmZXJlbmNlX3Njb3JlZF9zY3JpcHRfdmVyc2lvbl9pZCI6IDM1OTMxMzM0OCwNCiAgInJlZmVyZW5jZV9zb3VyY2Vfc2hhMjU2IjogImI4NTZlNzM2ZDVjYjQ3ZDhmYTMwOWNlMmFjZDVjZTA1ODhhNGY5MTQ3OGJkZWFlOTljNDkxZjMxYjdjMjZhNGMiLA0KICAidGFicGZuIjogew0KICAgICJjaGVja3BvaW50X2ZpbGVuYW1lIjogInRhYnBmbi12My41LTIwMjYwOTA5LnNhZmV0ZW5zb3JzIiwNCiAgICAiY2hlY2twb2ludF9zaGEyNTYiOiAiZWNlNGQ2N2VhZGZlYTQyZWIwZTYxMGRmNTE4OWJlYTYwY2I3ZjMxMDczZDgxZTljN2EwMTliNzZlYWNmMGJlMyIsDQogICAgImNoZWNrcG9pbnRfYnl0ZXMiOiA4NzYwMjc5MzIsDQogICAgInBhY2thZ2VfdmVyc2lvbiI6ICI5LjAuMCIsDQogICAgIm5fZXN0aW1hdG9ycyI6IDEyOA0KICB9LA0KICAib2ZmbGluZV9hc3NldF9kaXJlY3RvcnlfbmFtZSI6ICJudGwtdGFidWxhci1vZmZsaW5lLWFzc2V0cyIsDQogICJwY2FfY29tcG9uZW50c19wZXJfdmlldyI6IDY0LA0KICAicGNhX2l0ZXJhdGVkX3Bvd2VyIjogMywNCiAgInJhbmRvbV9zdGF0ZSI6IDQyLA0KICAicHJvYmFiaWxpdHlfdHJhbnNmb3JtIjogIm5vbmUiLA0KICAib3J0aG9ncmFwaHlfcG9zdF9wZW5hbHR5IjogMCwNCiAgInNlY29uZF9jb2x1bW5fYmFsYW5jZSI6IGZhbHNlLA0KICAiYmFzZWxpbmVfdGltZW91dF9zZWNvbmRzIjogMTMyMDAsDQogICJwcmVwYXJlX3RpbWVvdXRfc2Vjb25kcyI6IDMwMCwNCiAgIm1vZGVsX3RpbWVvdXRfc2Vjb25kcyI6IDM2MDAsDQogICJwYWNrYWdlX2luc3RhbGxfdGltZW91dF9zZWNvbmRzIjogMzAwLA0KICAidmVyaWZpZWRfcnVudGltZV9pbXBvcnRzIjogew0KICAgICJudW1weSI6ICIyLjEuMyIsDQogICAgInBhbmRhcyI6ICIyLjMuMyIsDQogICAgInNjaXB5IjogIjEuMTYuMyIsDQogICAgInNrbGVhcm4iOiAiMS42LjEiLA0KICAgICJ0b3JjaCI6ICIyLjExLjArY3UxMjgiLA0KICAgICJ0YWJwZm4iOiAiOS4wLjAiDQogIH0sDQogICJmZWF0dXJlX2NvdW50IjogMTg0LA0KICAidGV4dF9mZWF0dXJlX2FybSI6ICJtYXNrZWRfc2VxdWVuY2UiLA0KICAibWFza2VkX2ZlYXR1cmVfbW9kdWxlX3NoYTI1NiI6ICJjZTYzMTM5NzViMGY4ZWI4YTNiN2M4Y2U5Y2QwZDE2NTZkMGUxNmJkMWY3MzQwZDUwZGJjNDJlYzFhYzk5MWIyIiwNCiAgInRleHRfYmVuY2htYXJrX3NvdXJjZV9zaGEyNTYiOiAiNjAxZjUxMjQ2ODIzMjBhOTcwZjRiZTUxODczZDcwYjAzNWY1YzExZmUzNTY5Y2UyNDRiMGJhZDIxYTRkZDY2NSIsDQogICJtYXNrZWRfc3ZkX2NvbXBvbmVudHNfcGVyX3ZpZXciOiAxNiwNCiAgInJ1bnRpbWVfZGlhZ25vc3RpY19zY3JpcHRfdmVyc2lvbl9pZCI6IDM1OTE2MjM2OCwNCiAgInJ1bnRpbWVfZGlhZ25vc3RpY19xdWVyeV9jYXBhY2l0eV9yb3dzIjogMywNCiAgImxhcmdlcl9mZWF0dXJlX2NhcGFjaXR5X3JlZmVyZW5jZSI6IHsNCiAgICAic2NyaXB0X3ZlcnNpb25faWQiOiAzNTkxNTQ3MjksDQogICAgImZlYXR1cmVzIjogMjE2LA0KICAgICJxdWVyeV9yb3dzIjogNDA5Ng0KICB9LA0KICAicmVmZXJlbmNlX3Njb3JlZF92ZXJzaW9uIjogMiwNCiAgImZpbmFsX2NsYXVkZV9nYXRlIjogdHJ1ZSwNCiAgImJhc2VsaW5lX2JhbGFuY2VfYWxwaGEiOiAwLjUsDQogICJjZW50ZXJlZF9zdHlsZWdyYW1fc2hyaW5rYWdlIjogMS4wLA0KICAiY2VudGVyZWRfc3R5bGVncmFtX2FkYXB0ZXJfc2hhMjU2IjogIjRlMTE1OGY5ZDMwZDIzZjBjZWNlODQ3N2Y3MzMwMDM4ZWRmOWVlNDFlMDk2YWFlNzhiNWM2ZjAwNmQ5NTUyMTQiLA0KICAicGFyZW50X293bl9zb3VyY2Vfc2hhMjU2IjogImIwNDBmZDU5Y2IyMDVhMjdlNjliNmExODlhMjhjMmMyMzllMDlhNTdiNDUyYmExY2NjZjdmMmU4ODE5OGMzYTQiLA0KICAicmVmZXJlbmNlX25vdGVib29rX2lkIjogMTM5MjAwMTAwLA0KICAiY2FuZGlkYXRlX2FkbWl0dGVkIjogdHJ1ZSwNCiAgInByb3RlY3RlZF93ZWlnaHQiOiAwLjEsDQogICJwcm90ZWN0ZWRfb3BlcmF0b3Jfc2hhMjU2IjogIjlkOWY4YTg5YjkyOTM1MDc1MTMxZDFlM2EyNmE3OTQ4Y2JiZTZlYjE0OTVkNWNhODA4MzI5YTVlY2RhMzE4NGEiLA0KICAicmF3X2V4dGVuc2lvbiI6ICJzbW9sX2NlbnRlcl9hZGQiLA0KICAidGFiaWNsIjogew0KICAgICJjaGVja3BvaW50X2ZpbGVuYW1lIjogInRhYmljbC1jbGFzc2lmaWVyLXYxLjEtMjAyNTA1MDYuY2twdCIsDQogICAgImNoZWNrcG9pbnRfc2hhMjU2IjogImJiNDJkMTczM2I0OWZhYjNhN2E5ZTQyZGI2NDhjOGZmNjRkOGQ1NzA1ZDkzYzAzOTE3OWFkZDNiZmIyZGIzMmIiLA0KICAgICJjaGVja3BvaW50X2J5dGVzIjogMTA4MzE4NjIxLA0KICAgICJwYWNrYWdlX3ZlcnNpb24iOiAiMi4yLjAiLA0KICAgICJuX2VzdGltYXRvcnMiOiA2NCwNCiAgICAid2hlZWxfZmlsZW5hbWUiOiAidGFiaWNsLTIuMi4wLXB5My1ub25lLWFueS53aGwiLA0KICAgICJ3aGVlbF9zaGEyNTYiOiAiYzgwYjhlYzI3MTliZTI2YmMwYmYxZGQ3NThhMjM2MWY1ODAxOWQ4Zjc4NzcwMzhiNzg1NTI5YTQyNTc4OWNlNCINCiAgfQ0KfQ==\",\"optional-config.json\":\"ew0KICAiY2FuZGlkYXRlX2FkbWl0dGVkIjogdHJ1ZSwNCiAgInNlbGVjdGVkX2J5X3Jvb3QiOiB0cnVlLA0KICAiYXNzZXRzX3B1YmxpY19hdWRpdGVkIjogdHJ1ZSwNCiAgImRpYWdub3N0aWNfUEFTU19iaW5kaW5nIjogew0KICAgICJub3RlYm9va19pZCI6IDEzOTM1MzQ1NSwNCiAgICAidmVyc2lvbiI6IDMsDQogICAgInNjcmlwdF92ZXJzaW9uX2lkIjogMzU5NTgxOTg4LA0KICAgICJhdWRpdF9zaGEyNTYiOiAiOGU3NjcwOGZiMWNjYTkyMzlmZmFiZmU3ZThjZDAyOWI5NTRkMzk1NzU3N2YwMzQxNTZjM2NmZGE0YTliYTUyNSIsDQogICAgIm5vdGVib29rX3NoYTI1NiI6ICI4Yjk1NTczZTBmYzYxZTYxZWU0YWU5NmY0NDMxMTJjY2ZmOWEyYzEwZGY0OTI5ZWM2NjlkMGU5YjNjOThiNzdkIg0KICB9LA0KICAicmF3X2V4dGVuc2lvbiI6ICJzbW9sX2NlbnRlcl9hZGQiLA0KICAiY29lZmZpY2llbnQiOiAwLjAzLA0KICAiYWxwaGEiOiAwLjUsDQogICJwcm90ZWN0ZWRfd2VpZ2h0IjogMC4xLA0KICAicG9zdCI6IDAsDQogICJjb21wYWN0X2Fzc2V0X3JlZiI6ICJsaWdodG5pbmd2MDgvbnRsLXNtb2xsbTMtZmluYWwtaW5mZXJlbmNlLWFzc2V0cy8xIiwNCiAgImNvbXBhY3RfYXNzZXRfaWQiOiAxMjY3MTU4MCwNCiAgImNvbXBhY3RfYXNzZXRfbWFuaWZlc3Rfc2hhMjU2IjogIjE3NzI3MDNmYWFlZTVkMmIzMGU3MmZlMzdkMDgwMmMwYzIwZWFiMjMzMmU0ZDY3YTA2MWNkYTY5MTJmYjYxMWEiLA0KICAiYmFzZV9yZWYiOiAibGlnaHRuaW5ndjA4L3Ntb2xsbTMtM2ItYmFzZS1vZmZsaW5lLXdlaWdodHMvMSIsDQogICJiYXNlX25vdGVib29rX2lkIjogMTM5MzM5ODIxLA0KICAic291cmNlX3NoYTI1NiI6IHsNCiAgICAiY29tcG9uZW50LnB5IjogImFiMWRmOTRjZWExMzU4ZDk1MTI1ZWQ5OWY0ZmJlOTYwNmY3ZTA2OTBjYTZiNjkzOGVhZDQxOWY4ODQzNDNmNDYiLA0KICAgICJvd25lZF9wYXRoLnB5IjogIjIwZGFiYWZkM2Y0MjMxOTY4NjNjZTNkMWZiZDRjYWE4YjE5NmJkNTE3YTg4NDIyNTYxNjE1ZGIwMDNkMWU2MjAiLA0KICAgICJydW5fY29tcG9uZW50LnB5IjogIjhkNmVlODYzMWZmYTk5YmFhZjNmOWZkYjQ5ZTNkOGE0ZGJkNmRiZTIyODRlMmVhMTYxN2VhNTk2NTkwMjA1MmIiLA0KICAgICJjZW50ZXJfcmVhZG91dC5weSI6ICI0OTliOTlmNmNhOTRiNmI0YjhlYjFmY2ExNWE0ZWYwNTE5MDc5MTk0NDA2NDQxZTcwZjhkMGUyNDQzMDY4MDcyIiwNCiAgICAic291cmNlX2JhbGFuY2UucHkiOiAiZGE4NjgyYmM4Mjk4YWVkNjNjODk2MDdjZWIwODQ5OGE0N2EyMGJhN2FiNzQ3NTJkZGRiZGRhOTk2ZWFiMjM1MSIsDQogICAgImNsYXVkZV9nYXRlLnB5IjogIjNmZDE5NzFhMTBlYjE1ZGYyMWRkMmMyNWFjZjY1MTZkMjJlN2ZiMDc3MDhiNGNjMGQ0YTg3MjY4MWZiMzdhZTQiLA0KICAgICJwcm90ZWN0ZWRfb3BlcmF0b3IucHkiOiAiOWQ5ZjhhODliOTI5MzUwNzUxMzFkMWUzYTI2YTc5NDhjYmJlNmViMTQ5NWQ1Y2E4MDgzMjlhNWVjZGEzMTg0YSIsDQogICAgInJ1bl9wYXJhbGxlbC5weSI6ICIzZmYzMGQyN2Q4MTFkZWNiZTBhMTk2MjhhOTkzM2FlOWE3OTdmNTE5MWQzMDNmN2NkMzI1Nzg4M2JhNjY1OTlmIiwNCiAgICAicnVuX2NhbmRpZGF0ZV9vcHRpb25hbC5weSI6ICJhYjE3MGNmYTg3YWJiZmViY2ZmYjg3N2E0YjlkYjZiNGNmYTI1OTQ0NTMxYjMyMGUyZTM1ZGEyMDgyMmUwMTNiIiwNCiAgICAiY29tcHV0ZV9jZW50ZXIucHkiOiAiM2RiZjdjYTBjMzc2OWZiZDRhZGUwNTM0NmU0ZGFlYjc5MWQ1ODlkN2E3MzYxZTk2N2YwOGQ0NGM0Njk5OWJiZCIsDQogICAgImZpbmlzaF9jZW50ZXJfcHJvdGVjdGVkLnB5IjogImFhNGU1NGUyZmRjOGVhNGYyMDlhZTQyMTM4ZDZiODA4YWMwN2M0NWJmMjMyYzY4ZGIwMzE4YzVkZjhkYTBhNDciLA0KICAgICJhZG1pc3Npb25fcHJlZmxpZ2h0LnB5IjogIjU5ZGUzYzQxYjJhMTE0NmExMTc5NzAwOGI4MGU3Nzg1OWEwMjgyYjBkOGIxNGM2YTE1MmZhZmM5ZGJhMTllZWIiDQogIH0sDQogICJzbW9sX3RpbWVvdXRfc2Vjb25kcyI6IDM2MDAsDQogICJkaWFnbm9zdGljX2FjdHVhbF9QQVNTX2JvdW5kIjogdHJ1ZSwNCiAgInJvb3RfZGVjaXNpb25fYmluZGluZyI6IHsNCiAgICAiZXhwbGljaXRfY2FuZGlkYXRlX2FkbWlzc2lvbiI6IHRydWUsDQogICAgInNlbGVjdGVkX2JyYW5jaCI6ICJzbW9sX2NlbnRlcl9hZGRfcHJvdGVjdGVkX3BmbjEwIiwNCiAgICAiZGVjaXNpb25fc2hhMjU2IjogIjIwZTZkZWRiYjc4MzU3YzliNTcxNGQ3MDEzYTJkNTMwZGU1YWU4NTg1NjUxYjA0YTcyNGZiMDE3MGVlOTk4Y2YiLA0KICAgICJqb2ludF9yZXN1bHRfc2hhMjU2IjogImYyNjI4NWQwZTQ2M2VkODE0ZGU0NjNjYzE5MDU3NGQ3ZDk4MzkyM2JiNDA4N2EzOGU0MjM4NjZlOGVjNzM2NzYiLA0KICAgICJwdWJsaWNfYXNzZXRzX2F1ZGl0X3NoYTI1NiI6ICJiYmMzY2I4NTg2MzBlNWNiNWQ0NzNjNjc3ZDVjMDc0NGM4YjY4OGQ0MWE0MDI3ZmU2NDdlYzJjNzBmMDlmM2RkIg0KICB9DQp9\",\"smol-spec.json\":\"ew0KICAibW9kZSI6ICJTT1VSQ0VfT05MWV9PUFRJT05BTF9TTU9MX0NFTlRFUl9BREQiLA0KICAiaW5mZXJlbmNlX2F1dGhvcml6ZWQiOiB0cnVlLA0KICAiZGlhZ25vc3RpY19vbmx5X2F1dGhvcml6ZWQiOiBmYWxzZSwNCiAgImFjdHVhbF9maW5hbF9iaW5kaW5nIjogew0KICAgICJhY3R1YWxfZnVsbF9RQV9wYXNzZWQiOiB0cnVlLA0KICAgICJwYWlyZWRfZ2Fpbl9hZG1pdHRlZF9ieV9yb290IjogdHJ1ZSwNCiAgICAibm90ZWJvb2tfaWQiOiAxMzkzNDc0NzAsDQogICAgInZlcnNpb24iOiAxLA0KICAgICJzY3JpcHRfdmVyc2lvbl9pZCI6IDM1OTU2NDczOCwNCiAgICAid29ya2VyX3NoYTI1NiI6ICI0ZGVhMDBjMjMxM2RhZDQyNmJhMTA4MWYzYjU3MTU0MGM4NGY2ZTVlNDQxYzU3MDc1Njk5ZTZmOTU1N2U5MzViIiwNCiAgICAiY2hlY2twb2ludF9yZWxhdGl2ZSI6ICJmaW5hbC9maW5pc2hlZF9jaGVja3BvaW50IiwNCiAgICAiY29tcGxldGVfc2hhMjU2IjogIjMxYjUyMjUwZjVlYWFiN2NiZDliNjAxNTJlNDQ3MTQ1NTU2NmI2ZDQwZmFlOTExZjBmMTAyNWFjODI3ODU0ZmIiLA0KICAgICJyZXBvcnRfc2hhMjU2IjogIjUzYmU0ZDkzOTRiZDE2MWVkMGQ1OGI3NWVjZGMzOWY4ZGFmYjQwZTEyZGVjOGQ5YWQ1YjNlNzZkNDJjYWY3ODQiLA0KICAgICJwb3J0YWJsZV9maWxlc19zaGEyNTYiOiB7DQogICAgICAiYWRhcHRlci9hZGFwdGVyX21vZGVsLnNhZmV0ZW5zb3JzIjogImFjYzc3ZjJlOTVkODZlMGVkYzU0MWFmYTQ2YzFhZjE2YWU5NTdiZWQ5NTU0MzVjNTMxOWE0ZGM4YTYwYjRjNDEiLA0KICAgICAgImFkYXB0ZXIvYWRhcHRlcl9jb25maWcuanNvbiI6ICJmZTY0ZGZhODQ3MTE0OTAyNWVjOTEwZDdhNjczYzAzZWU2NzM4MjcwNDFlZmMxY2RhMDk4OGRlYzE2YjQyYWYzIiwNCiAgICAgICJoZWFkcy5zYWZldGVuc29ycyI6ICJjYmM1MGNhMmFiYzY2YTYwMDI0NTlhMjM0M2IwMmMwNTk4ZThkOTRmZTA5NTQzMDM5NTlkMjk3YWUyYzJmZThiIiwNCiAgICAgICJyZWFkb3V0cy5ucHoiOiAiMDQxMTU5MTBiMjQ1NjljNWUwODY0MTM0MmIwY2VmZTkzYzgxNTY1MTVmOTNkMGI5MjBlZGUwMWRmYmEyZGRjMiIsDQogICAgICAicmVwbGF5X2FuY2hvcnMubnB6IjogImU0OWE3NDdhMzJhNTUyOWYyZDBlMTM2M2EzZjViN2Q5ZTFjYTM0ZDdmYzExNDU3MWVjNjI0Mjc3MjA3YTI5YjIiLA0KICAgICAgImFkYXB0ZXIvUkVBRE1FLm1kIjogIjQzNjA0OWEwNWYxZjA4ODY2MWRkY2RjMzNjNDJiMWUxMzlkNWVkMTQ3ZGUxNjJiNzRiMTc4YzczZGI2ODliZmIiDQogICAgfSwNCiAgICAiZnVsbF9xYV9hdWRpdF9zaGEyNTYiOiAiOTc0MmVjNjNmZDEzMjExZjAwMGM4ZTQ2NGMzZTg4ZTE5N2ViMDMyZjhhMDg1Y2I1N2I0MGU1ZDRmYzE1NGE3ZSINCiAgfSwNCiAgInRyYWluaW5nX3dvcmtlcl9zaGEyNTYiOiAiNGRlYTAwYzIzMTNkYWQ0MjZiYTEwODFmM2I1NzE1NDBjODRmNmU1ZTQ0MWM1NzA3NTY5OWU2Zjk1NTdlOTM1YiIsDQogICJleHBlY3RlZF9mdWxsX25vdGVib29rX2lkIjogMTM5MzQ3NDcwLA0KICAiZXhwZWN0ZWRfZnVsbF9ub3RlYm9va192ZXJzaW9uIjogMSwNCiAgImJhc2VfcmV2aXNpb24iOiAiZDc4YTQyZjc5MTk4NjAzZTYxNDA5NTc1MzQ4NGEwNGMxMGMyYjk0MCIsDQogICJiYXNlX21hbmlmZXN0X3NoYTI1NiI6ICIxMjAxMTJiZDAwMmNkZThiZDc3MDU1MTQ0NjcwODFkYjRkNzg2NDZjMjIyYjM5NThiNmUzYWIwZDA5MDk4OTExIiwNCiAgImJhc2VfZmlsZXMiOiB7DQogICAgIlJFQURNRS5tZCI6IHsNCiAgICAgICJzaXplIjogMTIzNDAsDQogICAgICAiZ2l0X2Jsb2IiOiAiNDhiODYxOGUwZDIwZTQ3ZTgyNmEwYTg0MmM3OGEyZGM0NjE2ZDZhOSIsDQogICAgICAibGZzX3NoYTI1NiI6IG51bGwNCiAgICB9LA0KICAgICJjb25maWcuanNvbiI6IHsNCiAgICAgICJzaXplIjogOTQzLA0KICAgICAgImdpdF9ibG9iIjogIjk3NWViOWEwYWNlYTNiNzM5ZWNkYmM0YTRmNjE1ZDY4OGIyN2I0MDciLA0KICAgICAgImxmc19zaGEyNTYiOiBudWxsDQogICAgfSwNCiAgICAiZ2VuZXJhdGlvbl9jb25maWcuanNvbiI6IHsNCiAgICAgICJzaXplIjogMTI2LA0KICAgICAgImdpdF9ibG9iIjogImUwYjg1ZDgwNDBlNTFlZDVlZmRkZjZhMjUwMWMzYTQ2MmI0MGU2ZGEiLA0KICAgICAgImxmc19zaGEyNTYiOiBudWxsDQogICAgfSwNCiAgICAibW9kZWwtMDAwMDEtb2YtMDAwMDIuc2FmZXRlbnNvcnMiOiB7DQogICAgICAic2l6ZSI6IDQ5NjYzMTUyNjQsDQogICAgICAiZ2l0X2Jsb2IiOiAiODFkY2Q2OWQ3ZDU0M2NhMmY3NGVmY2IzNzUwZWU5NDcyMzFlZDE4MyIsDQogICAgICAibGZzX3NoYTI1NiI6ICI3ZTI3MGFjNTY4ZWUxODgwZGRiYWRhZDY2Y2NkY2Q5OTA2ZDUyNDE1ZTg5MDRlMmYzMDBjNzUyNTBiOWM3ZDQ5Ig0KICAgIH0sDQogICAgIm1vZGVsLTAwMDAyLW9mLTAwMDAyLnNhZmV0ZW5zb3JzIjogew0KICAgICAgInNpemUiOiAxMTgzOTE5NzQ0LA0KICAgICAgImdpdF9ibG9iIjogIjBkM2RhYWI2ODA1YjNjZDU1ZmMxZGQ4OGY1ZjFkOTBiYzY2YzE4M2IiLA0KICAgICAgImxmc19zaGEyNTYiOiAiYzZhNmU3NjkwYTY2ZGNjMzg2YTZhOWI0NTY2ODZlN2MzMDhkNDVjYmE4ODRkMTE2YzkyOGRhZmE3ZmE5ODdhZSINCiAgICB9LA0KICAgICJtb2RlbC5zYWZldGVuc29ycy5pbmRleC5qc29uIjogew0KICAgICAgInNpemUiOiAyNjg5OSwNCiAgICAgICJnaXRfYmxvYiI6ICJlZTZjMDI4YzAxMjgxNGQzODViZTUxYzFkNTM3YzM0NWM5ZmE5NDU3IiwNCiAgICAgICJsZnNfc2hhMjU2IjogbnVsbA0KICAgIH0sDQogICAgInNwZWNpYWxfdG9rZW5zX21hcC5qc29uIjogew0KICAgICAgInNpemUiOiAxNTEsDQogICAgICAiZ2l0X2Jsb2IiOiAiZDJhMWJiY2E5ZGY1NjYzZGJkNjNjODA1NjA4MjRmOGFlZjAyY2Y5ZSIsDQogICAgICAibGZzX3NoYTI1NiI6IG51bGwNCiAgICB9LA0KICAgICJ0b2tlbml6ZXIuanNvbiI6IHsNCiAgICAgICJzaXplIjogMTcyMDg4NjEsDQogICAgICAiZ2l0X2Jsb2IiOiAiMGNkOGI3ZmE3YzFmMTc4NDVlMGQ1ZTc0ZjlkNWFmZTkwY2EzMTA1YyIsDQogICAgICAibGZzX3NoYTI1NiI6ICJhYjRkYTZiMmFhNjgyNDdlOWMwZmE5Yjk3ZmM3ZmNjNzk2NTA1MDM4ZDAxZjdlMTQ0NTIyYTY1Y2UwZGJkMmU1Ig0KICAgIH0sDQogICAgInRva2VuaXplcl9jb25maWcuanNvbiI6IHsNCiAgICAgICJzaXplIjogNTA0MDUsDQogICAgICAiZ2l0X2Jsb2IiOiAiYzU3ZDQxOWYyYTk2NDcwNzc2Zjk5ZmEzN2E5MTQzMzYzZDNhZTA0NyIsDQogICAgICAibGZzX3NoYTI1NiI6IG51bGwNCiAgICB9DQogIH0sDQogICJjb3JlX3ZlcnNpb25zIjogew0KICAgICJ0b3JjaCI6ICIyLjExLjArY3UxMjgiLA0KICAgICJ0cmFuc2Zvcm1lcnMiOiAiNS4xNi4xIiwNCiAgICAicGVmdCI6ICIwLjIwLjAiLA0KICAgICJhY2NlbGVyYXRlIjogIjEuMTQuMCIsDQogICAgInNhZmV0ZW5zb3JzIjogIjAuOC4wIiwNCiAgICAibnVtcHkiOiAiMi4xLjMiLA0KICAgICJwYW5kYXMiOiAiMi4zLjMiDQogIH0sDQogICJjbGFzc2VzIjogWw0KICAgICJjbGF1ZGUtZmFibGUtNSIsDQogICAgImNsYXVkZS1vcHVzLTQtNiIsDQogICAgImNsYXVkZS1vcHVzLTQtOCIsDQogICAgImNsYXVkZS1vcHVzLTUiLA0KICAgICJjbGF1ZGUtc29ubmV0LTUiLA0KICAgICJnZW1pbmktMy4xLXByby1wcmV2aWV3IiwNCiAgICAiZ2VtaW5pLTMuNi1mbGFzaCIsDQogICAgImdlbWluaS0zLjctZmxhc2giLA0KICAgICJnZW1pbmktMy44LWZsYXNoIiwNCiAgICAiZ2xtLTUuMyIsDQogICAgImdsbS01LjMtZmxhc2giLA0KICAgICJncHQtNS41IiwNCiAgICAiZ3B0LTUuNi1sdW5hIiwNCiAgICAiZ3B0LTUuNi1zb2wiLA0KICAgICJncHQtNS42LXRlcnJhIiwNCiAgICAiZ3B0LTYtYXN0cmEiLA0KICAgICJraW1pLWszIiwNCiAgICAicXdlbjMuOC1tYXgiDQogIF0sDQogICJ0cmFpbl9zaGEyNTYiOiAiNDBhMjJjZThkODhjNWY0NmZlYjQ2MzdhMDk5ZGI4MTViZWY1ZjcxM2ZmZTUzZTg5ZGJlMWI3YjIxZGU3NzcwNyIsDQogICJkdHlwZSI6ICJGUDE2IFNEUEEgYmFzZTsgRlAzMiBhZGFwdGVycy9ub3Jtcy9yZWFkb3V0czsgRlAzMiBhZmZpbmUvbG9nX3NvZnRtYXgiLA0KICAibWF4X2xlbmd0aCI6IDQ0OCwNCiAgIm1pY3JvYmF0Y2giOiAzLA0KICAicG9vbF9sYXllcnMiOiBbDQogICAgNiwNCiAgICAxMiwNCiAgICAxOCwNCiAgICAyNCwNCiAgICAzMCwNCiAgICAzNg0KICBdLA0KICAiZmVhdHVyZV93aWR0aCI6IDEyMjg4LA0KICAib3V0cHV0X2NsYXNzZXMiOiAxOCwNCiAgImZpcnN0X3Rva2VuX2V4Y2x1ZGVkX2Zyb21fbWVhbiI6IHRydWUsDQogICJ0ZXN0X2lkX29yZGVyX3ByZXNlcnZlZCI6IHRydWUsDQogICJmZWF0dXJlX3JlcGxheV9ydG9sIjogMUUtMDUsDQogICJmZWF0dXJlX3JlcGxheV9hdG9sIjogMUUtMDUsDQogICJwcm9iYWJpbGl0eV9yZXBsYXlfbWF4YWJzIjogMUUtMDYsDQogICJ0cmFpbmluZyI6IGZhbHNlLA0KICAib3B0aW1pemVyX3N0YXRlX2xvYWRlZCI6IGZhbHNlLA0KICAidGFyZ2V0c19sb2FkZWQiOiBmYWxzZSwNCiAgInN1Ym1pc3Npb25fYXV0aG9yaXplZCI6IGZhbHNlLA0KICAic291cmNlX2hhc2hlcyI6IHsNCiAgICAiY29tcG9uZW50LnB5IjogImFiMWRmOTRjZWExMzU4ZDk1MTI1ZWQ5OWY0ZmJlOTYwNmY3ZTA2OTBjYTZiNjkzOGVhZDQxOWY4ODQzNDNmNDYiLA0KICAgICJvd25lZF9wYXRoLnB5IjogIjIwZGFiYWZkM2Y0MjMxOTY4NjNjZTNkMWZiZDRjYWE4YjE5NmJkNTE3YTg4NDIyNTYxNjE1ZGIwMDNkMWU2MjAiLA0KICAgICJydW5fY29tcG9uZW50LnB5IjogIjhkNmVlODYzMWZmYTk5YmFhZjNmOWZkYjQ5ZTNkOGE0ZGJkNmRiZTIyODRlMmVhMTYxN2VhNTk2NTkwMjA1MmIiDQogIH0NCn0=\",\"tabicl_default_worker.py\":\"IiIiSXNvbGF0ZWQgb2ZmbGluZSBUYWJJQ0wgdjEuMSBkZWZhdWx0NjQgaW5mZXJlbmNlIG9uIHRoZSB1bmNoYW5nZWQgMTUyLWNvbHVtbiBwcmVmaXguIiIiCmltcG9ydCBvcwphc3NlcnQgb3MubmFtZSAhPSAnbnQnIGFuZCBvcy5wYXRoLmlzZGlyKCcva2FnZ2xlL2lucHV0JyksICdLYWdnbGUgZXhlY3V0aW9uIG9ubHknCmZvciBrZXkgaW4gKCdPTVBfTlVNX1RIUkVBRFMnLCdNS0xfTlVNX1RIUkVBRFMnLCdPUEVOQkxBU19OVU1fVEhSRUFEUycpOgogICAgb3MuZW52aXJvbltrZXldID0gJzQnCmFzc2VydCBvcy5lbnZpcm9uLmdldCgnQ1VEQV9WSVNJQkxFX0RFVklDRVMnKSwgJ1BhcmVudCBtdXN0IGJpbmQgdGhlIGFzc2lnbmVkIHBoeXNpY2FsIEdQVScKb3MuZW52aXJvbi51cGRhdGUoSEZfSFVCX09GRkxJTkU9JzEnLCBUUkFOU0ZPUk1FUlNfT0ZGTElORT0nMScpCmltcG9ydCBoYXNobGliCmltcG9ydCBpbXBvcnRsaWIubWV0YWRhdGEgYXMgbWV0YWRhdGEKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0YWJpY2wgaW1wb3J0IFRhYklDTENsYXNzaWZpZXIKZnJvbSB0aHJlYWRwb29sY3RsIGltcG9ydCB0aHJlYWRwb29sX2xpbWl0cwoKdGhyZWFkcG9vbF9saW1pdHMoNCkKdG9yY2guc2V0X251bV90aHJlYWRzKDQpCmhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmNvbmZpZyA9IGpzb24ubG9hZHMoKGhlcmUvJ3J1bnRpbWVfY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKc2V0dGluZ3MgPSBjb25maWdbJ3RhYmljbCddCmNhY2hlID0gUGF0aChvcy5lbnZpcm9uWydOVExfVEFCVUxBUl9URU1QJ10pLnJlc29sdmUoKQphc3NlcnQgY2FjaGUuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQp3aXRoIG5wLmxvYWQoY2FjaGUvJ3RhYmxlcy5ucHonLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIHNhdmVkOgogICAgdHJhaW4xODQsIHF1ZXJ5MTg0LCB5LCBpZHMsIGxhYmVscyA9IChzYXZlZFtrXS5jb3B5KCkgZm9yIGsgaW4gKCd4X3RyYWluJywneF90ZXN0JywneScsJ3Rlc3RfaWRzJywnbGFiZWxzJykpCndpdGggbnAubG9hZChjYWNoZS8nYWxpZ25tZW50Lm5weicsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgc2F2ZWQ6CiAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChzYXZlZFsneSddLCB5KQogICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoc2F2ZWRbJ3Rlc3RfaWRzJ10sIGlkcykKICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHNhdmVkWydsYWJlbHMnXSwgbGFiZWxzKQphc3NlcnQgdHJhaW4xODQuc2hhcGVbMV0gPT0gcXVlcnkxODQuc2hhcGVbMV0gPT0gMTg0CmZpdF94LCBxdWVyeV94ID0gdHJhaW4xODRbOiwgOjE1Ml0uY29weSgpLCBxdWVyeTE4NFs6LCA6MTUyXS5jb3B5KCkKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoZml0X3gsIHRyYWluMTg0WzosIDoxNTJdKQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChxdWVyeV94LCBxdWVyeTE4NFs6LCA6MTUyXSkKYXNzZXJ0IGZpdF94LmR0eXBlID09IHF1ZXJ5X3guZHR5cGUgPT0gbnAuZmxvYXQzMgphc3NlcnQgbnAuaXNmaW5pdGUoZml0X3gpLmFsbCgpIGFuZCBucC5pc2Zpbml0ZShxdWVyeV94KS5hbGwoKQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChucC51bmlxdWUoeSksIG5wLmFyYW5nZSgxOCkpCmFzc2VydCBtZXRhZGF0YS52ZXJzaW9uKCd0YWJpY2wnKSA9PSBzZXR0aW5nc1sncGFja2FnZV92ZXJzaW9uJ10gPT0gJzIuMi4wJwphc3NlcnQgbWV0YWRhdGEudmVyc2lvbignZWlub3BzJykgPT0gJzAuOC4yJwphc3NlcnQgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQphc3NldCA9IFBhdGgob3MuZW52aXJvblsnTlRMX1RBQklDTF9BU1NFVCddKQpjaGVja3BvaW50ID0gYXNzZXQvc2V0dGluZ3NbJ2NoZWNrcG9pbnRfZmlsZW5hbWUnXQphc3NlcnQgY2hlY2twb2ludC5pc19maWxlKCkgYW5kIGNoZWNrcG9pbnQuc3RhdCgpLnN0X3NpemUgPT0gc2V0dGluZ3NbJ2NoZWNrcG9pbnRfYnl0ZXMnXQp3aXRoIGNoZWNrcG9pbnQub3BlbigncmInKSBhcyBzdHJlYW06CiAgICBhc3NlcnQgaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sICdzaGEyNTYnKS5oZXhkaWdlc3QoKSA9PSBzZXR0aW5nc1snY2hlY2twb2ludF9zaGEyNTYnXQphc3NlcnQgc2V0dGluZ3NbJ25fZXN0aW1hdG9ycyddID09IDY0LCAnRnJvemVuIGRlZmF1bHQ2NCBjb25maWd1cmF0aW9uJwpwYXJhbXMgPSBkaWN0KGNoZWNrcG9pbnRfdmVyc2lvbj1jaGVja3BvaW50Lm5hbWUsbW9kZWxfcGF0aD1zdHIoY2hlY2twb2ludCksYWxsb3dfYXV0b19kb3dubG9hZD1GYWxzZSwKICAgIG5fZXN0aW1hdG9ycz1zZXR0aW5nc1snbl9lc3RpbWF0b3JzJ10sbm9ybV9tZXRob2RzPU5vbmUsc29mdG1heF90ZW1wZXJhdHVyZT0uOSwKICAgIGF2ZXJhZ2VfbG9naXRzPUZhbHNlLHN1cHBvcnRfbWFueV9jbGFzc2VzPVRydWUsYmF0Y2hfc2l6ZT00LGt2X2NhY2hlPUZhbHNlLGRldmljZT0nY3VkYTowJywKICAgIHVzZV9hbXA9VHJ1ZSx1c2VfZmEzPUZhbHNlLG9mZmxvYWRfbW9kZT0nYXV0bycscmFuZG9tX3N0YXRlPTc3NyxuX2pvYnM9NCkKZnJlZSwgdG90YWwgPSB0b3JjaC5jdWRhLm1lbV9nZXRfaW5mbygpCmFzc2VydCBmcmVlID4gOCoxMDI0KiozCnRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQptb2RlbCA9IFRhYklDTENsYXNzaWZpZXIoKipwYXJhbXMpCnRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgYmVmb3JlID0gdGltZS5tb25vdG9uaWMoKQptb2RlbC5maXQoZml0X3gsIHkpCnRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgZml0X3NlY29uZHMgPSB0aW1lLm1vbm90b25pYygpLWJlZm9yZQpiZWZvcmUgPSB0aW1lLm1vbm90b25pYygpOyByYXcgPSBtb2RlbC5wcmVkaWN0X3Byb2JhKHF1ZXJ5X3gpCnRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgcHJlZGljdF9zZWNvbmRzID0gdGltZS5tb25vdG9uaWMoKS1iZWZvcmUKY29sdW1ucyA9IG5wLmFzYXJyYXkobW9kZWwuY2xhc3Nlc18sIGR0eXBlPWludCkKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwobnAuc29ydChjb2x1bW5zKSwgbnAuYXJhbmdlKDE4KSkKcHJvYmEgPSBucC56ZXJvcygobGVuKHF1ZXJ5X3gpLCAxOCksIGR0eXBlPW5wLmZsb2F0NjQpCnByb2JhWzosIGNvbHVtbnNdID0gcmF3CmFzc2VydCBucC5pc2Zpbml0ZShwcm9iYSkuYWxsKCkgYW5kIChwcm9iYSA+PSAwKS5hbGwoKQpucC50ZXN0aW5nLmFzc2VydF9hbGxjbG9zZShwcm9iYS5zdW0oMSksIDEsIHJ0b2w9MCwgYXRvbD0xZS01KQpucC5zYXZleihjYWNoZS8ndGFiaWNsX3Byb2JhLm5weicsIHByb2JhPXByb2JhLCBpZHM9aWRzLCBsYWJlbHM9bGFiZWxzKQpQYXRoKCcva2FnZ2xlL3dvcmtpbmcvdGFiaWNsX2luZmVyZW5jZV9hdWRpdC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3QoCiAgICBzdGF0dXM9J0NPTVBMRVRFJyxjaGVja3BvaW50X3NoYTI1Nj1zZXR0aW5nc1snY2hlY2twb2ludF9zaGEyNTYnXSxwYXJhbWV0ZXJzPXBhcmFtcywKICAgIHJlc29sdmVkX25vcm1fbWV0aG9kcz1tb2RlbC5lbnNlbWJsZV9nZW5lcmF0b3JfLm5vcm1fbWV0aG9kc18sdHJhaW5fcm93cz1sZW4oeSkscXVlcnlfcm93cz1sZW4oaWRzKSwKICAgIGZlYXR1cmVfY29sdW1ucz0xNTIsc291cmNlX3RhYmxlX2NvbHVtbnM9MTg0LHVuY2hhbmdlZF8xNTJfY29sdW1uX3ByZWZpeD1UcnVlLAogICAgZml0X3NlY29uZHM9Zml0X3NlY29uZHMscHJlZGljdF9zZWNvbmRzPXByZWRpY3Rfc2Vjb25kcywKICAgIHBlYWtfY3VkYV9hbGxvY2F0ZWRfYnl0ZXM9aW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSksCiAgICBwZWFrX2N1ZGFfcmVzZXJ2ZWRfYnl0ZXM9aW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9yZXNlcnZlZCgpKSwKICAgIHBhY2thZ2VzPXtwOm1ldGFkYXRhLnZlcnNpb24ocCkgZm9yIHAgaW4gKCd0YWJpY2wnLCd0b3JjaCcsJ251bXB5Jywnc2Npa2l0LWxlYXJuJywnZWlub3BzJyl9LAogICAgcHJvYmFiaWxpdHlfdHJhbnNmb3JtPSdub25lJyxyb3dfbWFzc19tYXhhYnM9ZmxvYXQobnAuYWJzKHByb2JhLnN1bSgxKS0xKS5tYXgoKSksCiAgICBvb2Y9RmFsc2UsbW9kZWxfY29udGV4dF9leHBvcnRlZD1GYWxzZSksaW5kZW50PTIpKQoKYXNzZXJ0IG1vZGVsLmVuc2VtYmxlX2dlbmVyYXRvcl8ubm9ybV9tZXRob2RzXyA9PSBbJ25vbmUnLCAncG93ZXInXQppbXBvcnQgb3MgYXMgX29zCl9hdWRpdF9wYXRoID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL3RhYmljbF9pbmZlcmVuY2VfYXVkaXQuanNvbicpCl9hdWRpdCA9IGpzb24ubG9hZHMoX2F1ZGl0X3BhdGgucmVhZF90ZXh0KCkpCl9hdWRpdC51cGRhdGUoUElEPV9vcy5nZXRwaWQoKSwgQ1VEQV9WSVNJQkxFX0RFVklDRVM9X29zLmVudmlyb25bJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ10pCl9hdWRpdF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhfYXVkaXQsIGluZGVudD0yKSkK\",\"setup_icl.py\":\"IiIiSW5zdGFsbCB0aGUgcmV2aWV3ZWQgVGFiSUNMIGFkZG9uIGludG8gdGhlIGV4aXN0aW5nIGlzb2xhdGVkIFBGTiBzaXRlLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKCmFzc2VydCBvcy5uYW1lICE9ICdudCcgYW5kIFBhdGgoJy9rYWdnbGUvaW5wdXQnKS5pc19kaXIoKQpoZXJlID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApjb25maWcgPSBqc29uLmxvYWRzKChoZXJlIC8gJ3J1bnRpbWVfY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKc2V0dGluZ3MgPSBjb25maWdbJ3RhYmljbCddCmFzc2V0ID0gUGF0aChvcy5lbnZpcm9uWydOVExfVEFCSUNMX0FTU0VUJ10pCmNhY2hlID0gUGF0aChvcy5lbnZpcm9uWydOVExfVEFCVUxBUl9URU1QJ10pLnJlc29sdmUoKQphc3NlcnQgY2FjaGUuaXNfcmVsYXRpdmVfdG8oUGF0aCgnL2thZ2dsZS90ZW1wJykucmVzb2x2ZSgpKQpzaXRlID0gY2FjaGUgLyAnc2l0ZScKCgpkZWYgc2hhKHBhdGgpOgogICAgd2l0aCBwYXRoLm9wZW4oJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIHJldHVybiBoYXNobGliLmZpbGVfZGlnZXN0KHN0cmVhbSwgJ3NoYTI1NicpLmhleGRpZ2VzdCgpCgoKY2hlY2twb2ludCA9IGFzc2V0IC8gc2V0dGluZ3NbJ2NoZWNrcG9pbnRfZmlsZW5hbWUnXQphc3NlcnQgY2hlY2twb2ludC5zdGF0KCkuc3Rfc2l6ZSA9PSBzZXR0aW5nc1snY2hlY2twb2ludF9ieXRlcyddCmFzc2VydCBzaGEoY2hlY2twb2ludCkgPT0gc2V0dGluZ3NbJ2NoZWNrcG9pbnRfc2hhMjU2J10KbG9jayA9IGFzc2V0IC8gJ3RhYmljbC1hZGRvbi1sb2NrLnR4dCcKYXNzZXJ0IGxvY2sucmVhZF90ZXh0KCkuc3RyaXAoKSA9PSAndGFiaWNsPT0yLjIuMCAtLWhhc2g9c2hhMjU2OicgKyBzZXR0aW5nc1snd2hlZWxfc2hhMjU2J10KYXNzZXJ0IHNoYShhc3NldCAvIHNldHRpbmdzWyd3aGVlbF9maWxlbmFtZSddKSA9PSBzZXR0aW5nc1snd2hlZWxfc2hhMjU2J10KZm9yIG5hbWUgaW4gKCdUYWJJQ0xfQ09ERV9MSUNFTlNFJywgJ01PREVMX1dFSUdIVF9MSUNFTlNFX0RFQ0xBUkFUSU9OLnR4dCcsICdtb2RlbF9saWNlbnNlX3Byb3ZlbmFuY2UuanNvbicpOgogICAgYXNzZXJ0IChhc3NldCAvIG5hbWUpLmlzX2ZpbGUoKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICctbScsICdwaXAnLCAnaW5zdGFsbCcsICctLW5vLWluZGV4JywgJy0tbm8tZGVwcycsICctLXJlcXVpcmUtaGFzaGVzJywKICAgICctLWZpbmQtbGlua3MnLCBzdHIoYXNzZXQpLCAnLS10YXJnZXQnLCBzdHIoc2l0ZSksICctcicsIHN0cihsb2NrKV0sIGNoZWNrPVRydWUsIHRpbWVvdXQ9MzAwKQplbnYgPSBvcy5lbnZpcm9uLmNvcHkoKQplbnZbJ1BZVEhPTlBBVEgnXSA9IHN0cihzaXRlKQpleHBlY3RlZCA9IGRpY3QodGFicGZuPSc5LjAuMCcsIHRhYmljbD0nMi4yLjAnLCBlaW5vcHM9JzAuOC4yJywgdG9yY2g9JzIuMTEuMCtjdTEyOCcsCiAgICAgICAgICAgICAgICBudW1weT0nMi4xLjMnLCBza2xlYXJuPScxLjYuMScpCmNvZGUgPSAoImltcG9ydCBqc29uLGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtO2Zyb20gcGF0aGxpYiBpbXBvcnQgUGF0aDsiCiAgICAgICAgImZyb20gdGFicGZuIGltcG9ydCBUYWJQRk5DbGFzc2lmaWVyO2Zyb20gdGFiaWNsIGltcG9ydCBUYWJJQ0xDbGFzc2lmaWVyO2ltcG9ydCBlaW5vcHMsdG9yY2gsbnVtcHksc2tsZWFybjsiCiAgICAgICAgImFjdHVhbD1kaWN0KHRhYnBmbj1tLnZlcnNpb24oJ3RhYnBmbicpLHRhYmljbD1tLnZlcnNpb24oJ3RhYmljbCcpLGVpbm9wcz1laW5vcHMuX192ZXJzaW9uX18sIgogICAgICAgICJ0b3JjaD10b3JjaC5fX3ZlcnNpb25fXyxudW1weT1udW1weS5fX3ZlcnNpb25fXyxza2xlYXJuPXNrbGVhcm4uX192ZXJzaW9uX18pOyIKICAgICAgICAiYXNzZXJ0IGFjdHVhbD09IiArIHJlcHIoZXhwZWN0ZWQpICsgIiwoYWN0dWFsLCdFeGFjdCBzaGFyZWQgcnVudGltZSByZXF1aXJlZCcpOyIKICAgICAgICAiUGF0aCgnL2thZ2dsZS93b3JraW5nL3RhYnVsYXJfam9pbnRfZW52aXJvbm1lbnRfYXVkaXQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcygiCiAgICAgICAgImRpY3Qoc3RhdHVzPSdWRVJJRklFRF9TSEFSRURfQURET05TX05PX01PREVMUycsaW1wb3J0cz1hY3R1YWwsbW9kZWxzX2luaXRpYWxpemVkPTApLGluZGVudD0yKSk7IgogICAgICAgICJwcmludCgnT2ZmbGluZSBQRk4gKyBkZWZhdWx0IFRhYklDTCBpbXBvcnRzIHBhc3NlZDsgbm8gbW9kZWxzIGluaXRpYWxpemVkJyxmbHVzaD1UcnVlKSIpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwgJy1jJywgY29kZV0sIGVudj1lbnYsIGNoZWNrPVRydWUsIHRpbWVvdXQ9OTApCg==\",\"run_parallel_icl.py\":\"IiIiUEZOIG9uIHBoeXNpY2FsMDsgc2VxdWVudGlhbCBTbW9sIHRoZW4gZGVmYXVsdCBUYWJJQ0wgb24gcGh5c2ljYWwxLiIiIgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIGljbF9hZG1pc3Npb24gaW1wb3J0IHJlcXVpcmVfYWRtaXNzaW9uCgpoZXJlID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApjZmcgPSByZXF1aXJlX2FkbWlzc2lvbihoZXJlKQphc3NlcnQgb3MubmFtZSAhPSAnbnQnIGFuZCBQYXRoKCcva2FnZ2xlL2lucHV0JykuaXNfZGlyKCkKbWFwcGluZyA9IFtdCmZvciBsaW5lIGluIHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCx1dWlkLG5hbWUnLCAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSwgdGV4dD1UcnVlKS5zdHJpcCgpLnNwbGl0bGluZXMoKToKICAgIGluZGV4LCB1dWlkLCBuYW1lID0gW3ZhbHVlLnN0cmlwKCkgZm9yIHZhbHVlIGluIGxpbmUuc3BsaXQoJywnLCAyKV0KICAgIG1hcHBpbmcuYXBwZW5kKGRpY3QocGh5c2ljYWxfaW5kZXg9aW50KGluZGV4KSwgdXVpZD11dWlkLCBuYW1lPW5hbWUpKQptYXBwaW5nLnNvcnQoa2V5PWxhbWJkYSBncHU6IGdwdVsncGh5c2ljYWxfaW5kZXgnXSkKYXNzZXJ0IFtncHVbJ3BoeXNpY2FsX2luZGV4J10gZm9yIGdwdSBpbiBtYXBwaW5nXSA9PSBbMCwgMV0KYXNzZXJ0IGFsbCgnVDQnIGluIGdwdVsnbmFtZSddIGFuZCBncHVbJ3V1aWQnXS5zdGFydHN3aXRoKCdHUFUtJykgZm9yIGdwdSBpbiBtYXBwaW5nKQphc3NlcnQgbWFwcGluZ1swXVsndXVpZCddICE9IG1hcHBpbmdbMV1bJ3V1aWQnXQpvdXQgPSBQYXRoKCcva2FnZ2xlL3dvcmtpbmcvc21vbF9wZm5faWNsX3BhcmFsbGVsJykKb3V0Lm1rZGlyKGV4aXN0X29rPUZhbHNlKQpjaGlsZHJlbiwgbG9ncywgcmVjb3JkcyA9IHt9LCBbXSwge30Kc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKCgpkZWYgc3Bhd24ocm91dGUsIHBoeXNpY2FsLCBmaWxlbmFtZSwgYXJncz0oKSwgYWRkb25zPUZhbHNlKToKICAgIGVudiA9IG9zLmVudmlyb24uY29weSgpCiAgICBlbnZbJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ10gPSBtYXBwaW5nW3BoeXNpY2FsXVsndXVpZCddCiAgICBlbnZbJ0hGX0hVQl9PRkZMSU5FJ10gPSBlbnZbJ1RSQU5TRk9STUVSU19PRkZMSU5FJ10gPSAnMScKICAgIGlmIGFkZG9uczoKICAgICAgICBlbnZbJ1BZVEhPTlBBVEgnXSA9IG9zLmVudmlyb25bJ05UTF9UQUJVTEFSX1NJVEUnXQogICAgbG9nID0gKG91dCAvIChyb3V0ZSArICcubG9nJykpLm9wZW4oJ3cnLCBidWZmZXJpbmc9MSkKICAgIGxvZ3MuYXBwZW5kKGxvZykKICAgIGNoaWxkID0gc3VicHJvY2Vzcy5Qb3Blbihbc3lzLmV4ZWN1dGFibGUsICctdScsIHN0cihoZXJlIC8gZmlsZW5hbWUpLCAqYXJnc10sIGVudj1lbnYsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3Rkb3V0PWxvZywgc3RkZXJyPXN1YnByb2Nlc3MuU1RET1VULCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgY2hpbGRyZW5bcm91dGVdID0gY2hpbGQKICAgIHJlY29yZHNbcm91dGVdID0gZGljdChQSUQ9Y2hpbGQucGlkLCBHUFU9bWFwcGluZ1twaHlzaWNhbF0sIHN0YXJ0ZWRfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZmluaXNoZWRfc2Vjb25kcz1Ob25lLCByZXR1cm5jb2RlPU5vbmUpCgoKdHJ5OgogICAgc3Bhd24oJ1BGTjEyOCcsIDAsICd0YWJ1bGFyX3dvcmtlci5weScsICgndGFicGZuJywpLCBUcnVlKQogICAgc3Bhd24oJ1Ntb2xDRU5URVInLCAxLCAnY29tcHV0ZV9jZW50ZXIucHknKQogICAgdGljayA9IDAKICAgIHdoaWxlIFRydWU6CiAgICAgICAgZWxhcHNlZCA9IHRpbWUubW9ub3RvbmljKCktc3RhcnRlZAogICAgICAgIGFzc2VydCBlbGFwc2VkIDwgY2ZnWydwYXJhbGxlbF9waGFzZV90aW1lb3V0X3NlY29uZHMnXSwgJ1BoYXNlIHRpbWVvdXQ7IG5vIHJldHJ5JwogICAgICAgIGZvciByb3V0ZSwgY2hpbGQgaW4gbGlzdChjaGlsZHJlbi5pdGVtcygpKToKICAgICAgICAgICAgY29kZSA9IGNoaWxkLnBvbGwoKQogICAgICAgICAgICBpZiBjb2RlIGlzIG5vdCBOb25lIGFuZCByZWNvcmRzW3JvdXRlXVsnZmluaXNoZWRfc2Vjb25kcyddIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZWNvcmRzW3JvdXRlXS51cGRhdGUoZmluaXNoZWRfc2Vjb25kcz1lbGFwc2VkLCByZXR1cm5jb2RlPWNvZGUpCiAgICAgICAgICAgIGFzc2VydCBjb2RlIGluIChOb25lLCAwKSwgKHJvdXRlLCBjb2RlLCAnTm8gcmV0cnkgb3IgZmluYWxpemF0aW9uJykKICAgICAgICAgICAgaWYgY29kZSBpcyBOb25lOgogICAgICAgICAgICAgICAgYXNzZXJ0IGVsYXBzZWQtcmVjb3Jkc1tyb3V0ZV1bJ3N0YXJ0ZWRfc2Vjb25kcyddIDwgMzYwMCwgKHJvdXRlLCAndGltZW91dCcpCiAgICAgICAgaWYgY2hpbGRyZW5bJ1Ntb2xDRU5URVInXS5wb2xsKCkgPT0gMCBhbmQgJ1RhYklDTDY0REVGQVVMVCcgbm90IGluIGNoaWxkcmVuOgogICAgICAgICAgICAjIHdhaXQoKSByZWFwcyB0aGUgY29tcGxldGVkIHByb2Nlc3MgYmVmb3JlIHRoZSBuZXh0IG1vZGVsIGNhbiBvY2N1cHkgcGh5c2ljYWwxLgogICAgICAgICAgICBjaGlsZHJlblsnU21vbENFTlRFUiddLndhaXQoKQogICAgICAgICAgICBzcGF3bignVGFiSUNMNjRERUZBVUxUJywgMSwgJ3RhYmljbF9kZWZhdWx0X3dvcmtlci5weScsIGFkZG9ucz1UcnVlKQogICAgICAgIGdwdSA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCx1dWlkLHV0aWxpemF0aW9uLmdwdSxtZW1vcnkudXNlZCcsICctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLCB0ZXh0PVRydWUsIHRpbWVvdXQ9MTUpCiAgICAgICAgYXBwcyA9IHN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWNvbXB1dGUtYXBwcz1waWQsZ3B1X3V1aWQsdXNlZF9tZW1vcnknLCAnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSwgdGV4dD1UcnVlLCB0aW1lb3V0PTE1KQogICAgICAgIHdpdGggKG91dCAvICdncHUtdGVsZW1ldHJ5Lmpzb25sJykub3BlbignYScpIGFzIHN0cmVhbToKICAgICAgICAgICAgc3RyZWFtLndyaXRlKGpzb24uZHVtcHMoZGljdChlbGFwc2VkX3NlY29uZHM9ZWxhcHNlZCwgR1BVPWdwdSwgY29tcHV0ZV9hcHBzPWFwcHMsIHdvcmtlcnM9cmVjb3JkcykpKydcbicpCiAgICAgICAgaWYgdGljayAlIDYgPT0gMDoKICAgICAgICAgICAgcHJpbnQoanNvbi5kdW1wcyhkaWN0KHN0YWdlPSdQRk5fU01PTF9USEVOX0lDTCcsIGVsYXBzZWRfc2Vjb25kcz1lbGFwc2VkLCB3b3JrZXJzPXJlY29yZHMpKSwgZmx1c2g9VHJ1ZSkKICAgICAgICBpZiBsZW4oY2hpbGRyZW4pID09IDMgYW5kIGFsbChjaGlsZC5wb2xsKCkgPT0gMCBmb3IgY2hpbGQgaW4gY2hpbGRyZW4udmFsdWVzKCkpOgogICAgICAgICAgICBicmVhawogICAgICAgIHRpY2sgKz0gMQogICAgICAgIHRpbWUuc2xlZXAoNSkKICAgIHNtb2wgPSBqc29uLmxvYWRzKFBhdGgoJy9rYWdnbGUvd29ya2luZy9zbW9sX2NlbnRlcl9pbmZlcmVuY2VfYXVkaXQuanNvbicpLnJlYWRfdGV4dCgpKQogICAgaWNsID0ganNvbi5sb2FkcyhQYXRoKCcva2FnZ2xlL3dvcmtpbmcvdGFiaWNsX2luZmVyZW5jZV9hdWRpdC5qc29uJykucmVhZF90ZXh0KCkpCiAgICBhc3NlcnQgc21vbFsnc3RhdHVzJ10gPT0gJ09XTkVEX0ZJTkFMX0NFTlRFUl9GRUFUVVJFX1BBU1NfQ09NUExFVEUnIGFuZCBzbW9sWydxdWVyeV9jdWRhX2V2ZW50X21zJ10gPiAwCiAgICBhc3NlcnQgc21vbFsnQ1VEQV9WSVNJQkxFX0RFVklDRVMnXSA9PSBpY2xbJ0NVREFfVklTSUJMRV9ERVZJQ0VTJ10gPT0gbWFwcGluZ1sxXVsndXVpZCddCiAgICBhc3NlcnQgc21vbFsnUElEJ10gPT0gcmVjb3Jkc1snU21vbENFTlRFUiddWydQSUQnXSBhbmQgaWNsWydQSUQnXSA9PSByZWNvcmRzWydUYWJJQ0w2NERFRkFVTFQnXVsnUElEJ10KICAgIGFzc2VydCByZWNvcmRzWydTbW9sQ0VOVEVSJ11bJ2ZpbmlzaGVkX3NlY29uZHMnXSA8PSByZWNvcmRzWydUYWJJQ0w2NERFRkFVTFQnXVsnc3RhcnRlZF9zZWNvbmRzJ10KICAgIChvdXQgLyAnQ09NUExFVEUuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KHN0YXR1cz0nUEZOX0FORF9TRVFVRU5USUFMX1NNT0xfSUNMX0NPTVBMRVRFJywgd29ya2Vycz1yZWNvcmRzLAogICAgICAgIG5vX2NvbmN1cnJlbnRfU21vbF9JQ0xfcHJvY2Vzc2VzPVRydWUsIHdhbGxfc2Vjb25kcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQsIGF1dG9tYXRpY19yZXRyeT1GYWxzZSksIGluZGVudD0yKSkKZmluYWxseToKICAgIGZvciBjaGlsZCBpbiBjaGlsZHJlbi52YWx1ZXMoKToKICAgICAgICBpZiBjaGlsZC5wb2xsKCkgaXMgTm9uZToKICAgICAgICAgICAgb3Mua2lsbHBnKGNoaWxkLnBpZCwgc2lnbmFsLlNJR1RFUk0pCiAgICBmb3IgY2hpbGQgaW4gY2hpbGRyZW4udmFsdWVzKCk6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjaGlsZC53YWl0KHRpbWVvdXQ9MTApCiAgICAgICAgZXhjZXB0IHN1YnByb2Nlc3MuVGltZW91dEV4cGlyZWQ6CiAgICAgICAgICAgIG9zLmtpbGxwZyhjaGlsZC5waWQsIHNpZ25hbC5TSUdLSUxMKQogICAgICAgICAgICBjaGlsZC53YWl0KCkKICAgIGZvciBsb2cgaW4gbG9nczoKICAgICAgICBsb2cuY2xvc2UoKQo=\",\"run_candidate_icl.py\":\"IiIiT2ZmbGluZSBzZXF1ZW50aWFsIGJhc2VsaW5lLCBmaXhlZCB0YWJ1bGFyIG1lbWJlcnMgYW5kIENQVSBmaW5hbGl6YXRpb24uIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgphc3NlcnQgb3MubmFtZSE9J250JyBhbmQgUGF0aCgnL2thZ2dsZS9pbnB1dCcpLmlzX2RpcigpLCdLYWdnbGUgZXhlY3V0aW9uIG9ubHknCmhlcmU9UGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApjb25maWc9anNvbi5sb2FkcygoaGVyZS8ncnVudGltZV9jb25maWcuanNvbicpLnJlYWRfdGV4dCgpKQpmcm9tIGljbF9hZG1pc3Npb24gaW1wb3J0IHJlcXVpcmVfYWRtaXNzaW9uCnJlcXVpcmVfYWRtaXNzaW9uKGhlcmUpCmFzc2VydCBjb25maWdbJ3JlZmVyZW5jZV9zY29yZWRfc2NyaXB0X3ZlcnNpb25faWQnXT09MzU5MzEzMzQ4CmFzc2VydCBjb25maWdbJ2NhbmRpZGF0ZV9hZG1pdHRlZCddLCAnQ2FuZGlkYXRlIGFkbWlzc2lvbiByZXF1aXJlZCcKYXNzZXJ0IGNvbmZpZ1sncHJvdGVjdGVkX3dlaWdodCddPT0uMTAgYW5kIGNvbmZpZ1sncmF3X2V4dGVuc2lvbiddID09ICdzbW9sX2NlbnRlcl9hZGQnCmFzc2VydCBub3QgY29uZmlnWydzZWNvbmRfY29sdW1uX2JhbGFuY2UnXSBhbmQgY29uZmlnWydwcm9iYWJpbGl0eV90cmFuc2Zvcm0nXT09J25vbmUnCm1hdGNoZXM9bGlzdChQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoY29uZmlnWyd0YWJwZm4nXVsnY2hlY2twb2ludF9maWxlbmFtZSddKSkKYXNzZXJ0IGxlbihtYXRjaGVzKT09MSwnQXR0YWNoIHRoZSBleGFjdCByZXZpZXdlZCBvZmZsaW5lIHRhYnVsYXIgYXNzZXQgcGFja2FnZScKYXNzZXQ9bWF0Y2hlc1swXS5wYXJlbnQKbWFuaWZlc3Q9anNvbi5sb2FkcygoYXNzZXQvJ3J1bnRpbWVfbWFuaWZlc3QuanNvbicpLnJlYWRfdGV4dCgpKQphc3NlcnQgbWFuaWZlc3RbJ3N0YXR1cyddPT0nQ1BVX1NFVFVQX0NPTVBMRVRFX05PX01PREVMX0VYRUNVVElPTicKYXNzZXJ0IG1hbmlmZXN0WydnbG9iYWxfZW52aXJvbm1lbnRfdW5jaGFuZ2VkJ10gYW5kIG5vdCBtYW5pZmVzdFsnbW9kZWxfZml0J10gYW5kIG5vdCBtYW5pZmVzdFsnbW9kZWxfcHJlZGljdCddCmFzc2VydCBtYW5pZmVzdFsnY2hlY2twb2ludF9zaGEyNTYnXT09Y29uZmlnWyd0YWJwZm4nXVsnY2hlY2twb2ludF9zaGEyNTYnXQphc3NlcnQgbWF0Y2hlc1swXS5zdGF0KCkuc3Rfc2l6ZT09Y29uZmlnWyd0YWJwZm4nXVsnY2hlY2twb2ludF9ieXRlcyddCndpdGggbWF0Y2hlc1swXS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgIGFzc2VydCBoYXNobGliLmZpbGVfZGlnZXN0KHN0cmVhbSwnc2hhMjU2JykuaGV4ZGlnZXN0KCk9PWNvbmZpZ1sndGFicGZuJ11bJ2NoZWNrcG9pbnRfc2hhMjU2J10KbG9jaz1hc3NldC8ncnVudGltZS1hZGRvbi1sb2NrLnR4dCcKd2hlZWxzPWFzc2V0CmFzc2VydCBsb2NrLmlzX2ZpbGUoKSBhbmQgYW55KHdoZWVscy5nbG9iKCcqLndobCcpKSwnUmV2aWV3ZWQgb2ZmbGluZSBhZGRvbiBsb2NrL3doZWVscyBhcmUgcmVxdWlyZWQnCmZvciBpdGVtIGluIG1hbmlmZXN0WydzZWxlY3RlZF9hZGRvbnMnXS52YWx1ZXMoKToKICAgIHdpdGggKHdoZWVscy9pdGVtWyd3aGVlbCddKS5vcGVuKCdyYicpIGFzIHN0cmVhbToKICAgICAgICBhc3NlcnQgaGFzaGxpYi5maWxlX2RpZ2VzdChzdHJlYW0sJ3NoYTI1NicpLmhleGRpZ2VzdCgpPT1pdGVtWydzaGEyNTYnXQpmb3IgbGluZSBpbiBsb2NrLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKToKICAgIGlmICc9PScgaW4gbGluZSBhbmQgbm90IGxpbmUubHN0cmlwKCkuc3RhcnRzd2l0aCgnIycpOgogICAgICAgIHBhY2thZ2U9cmUuc3BsaXQocic9PScsbGluZS5zdHJpcCgpLG1heHNwbGl0PTEpWzBdLmxvd2VyKCkucmVwbGFjZSgnXycsJy0nKQogICAgICAgIGFzc2VydCBwYWNrYWdlIG5vdCBpbiAoJ3RvcmNoJywndHJhbnNmb3JtZXJzJywncGVmdCcsJ2FjY2VsZXJhdGUnLCdzYWZldGVuc29ycycpCiAgICAgICAgYXNzZXJ0IG5vdCBwYWNrYWdlLnN0YXJ0c3dpdGgoJ252aWRpYS0nKSwnUmV1c2UgdGhlIG9yaWdpbmFsIENVREEvVG9yY2ggcnVudGltZScKUGF0aCgnL2thZ2dsZS90ZW1wJykubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCnRlbXA9UGF0aCh0ZW1wZmlsZS5ta2R0ZW1wKHByZWZpeD0nbnRsLXRhYnVsYXItJyxkaXI9Jy9rYWdnbGUvdGVtcCcpKQpzaXRlPXRlbXAvJ3NpdGUnCmJhc2VfZW52PW9zLmVudmlyb24uY29weSgpCmJhc2VfZW52WydOVExfVEFCVUxBUl9URU1QJ109c3RyKHRlbXApCmJhc2VfZW52WydOVExfVEFCVUxBUl9BU1NFVCddPXN0cihhc3NldCkKc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCctbScsJ3BpcCcsJ2luc3RhbGwnLCctLW5vLWluZGV4JywnLS1maW5kLWxpbmtzJyxzdHIod2hlZWxzKSwKICAgICctLW5vLWRlcHMnLCctLXJlcXVpcmUtaGFzaGVzJywnLS10YXJnZXQnLHN0cihzaXRlKSwnLXInLHN0cihsb2NrKV0sCiAgICBjaGVjaz1UcnVlLHRpbWVvdXQ9Y29uZmlnWydwYWNrYWdlX2luc3RhbGxfdGltZW91dF9zZWNvbmRzJ10pCnRhYnVsYXJfZW52PWJhc2VfZW52LmNvcHkoKQp0YWJ1bGFyX2VudlsnUFlUSE9OUEFUSCddPXN0cihzaXRlKQp0YWJ1bGFyX2VudlsnSEZfSFVCX09GRkxJTkUnXT0nMScKc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsCiAgICAiaW1wb3J0IGpzb24sIGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtOyBmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGg7IGltcG9ydCBudW1weSwgcGFuZGFzLCBzY2lweSwgc2tsZWFybiwgdG9yY2g7IGZyb20gdGFicGZuIGltcG9ydCBUYWJQRk5DbGFzc2lmaWVyOyBmcm9tIHRhYnBmbi5jb25zdGFudHMgaW1wb3J0IE1vZGVsVmVyc2lvbjsgYWN0dWFsPWRpY3QobnVtcHk9bnVtcHkuX192ZXJzaW9uX18sIHBhbmRhcz1wYW5kYXMuX192ZXJzaW9uX18sIHNjaXB5PXNjaXB5Ll9fdmVyc2lvbl9fLCBza2xlYXJuPXNrbGVhcm4uX192ZXJzaW9uX18sIHRvcmNoPXRvcmNoLl9fdmVyc2lvbl9fLCB0YWJwZm49bS52ZXJzaW9uKCd0YWJwZm4nKSk7IGV4cGVjdGVkPSIrcmVwcihjb25maWdbJ3ZlcmlmaWVkX3J1bnRpbWVfaW1wb3J0cyddKSsiOyBhc3NlcnQgYWN0dWFsPT1leHBlY3RlZCwgKGFjdHVhbCxleHBlY3RlZCk7IGFzc2VydCBNb2RlbFZlcnNpb24uVjNfNTsgUGF0aCgnL2thZ2dsZS93b3JraW5nL3RhYnVsYXJfZW52aXJvbm1lbnRfYXVkaXQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KHN0YXR1cz0nVkVSSUZJRURfT0ZGTElORV9SVU5USU1FX0lNUE9SVFMnLGltcG9ydHM9YWN0dWFsLGRpYWdub3N0aWNfc2NyaXB0X3ZlcnNpb25faWQ9MzU5MTYyMzY4KSxpbmRlbnQ9MikpOyBwcmludCgnT2ZmbGluZSBUYWJQRk4gaW1wb3J0cyBwYXNzZWQnLCBmbHVzaD1UcnVlKSJdLAogICAgZW52PXRhYnVsYXJfZW52LGNoZWNrPVRydWUsdGltZW91dD05MCkKaWNsX21hdGNoZXM9bGlzdChQYXRoKCcva2FnZ2xlL2lucHV0Jykucmdsb2IoY29uZmlnWyd0YWJpY2wnXVsnY2hlY2twb2ludF9maWxlbmFtZSddKSkKYXNzZXJ0IGxlbihpY2xfbWF0Y2hlcyk9PTEsICdBdHRhY2ggb25lIGV4YWN0IHJldmlld2VkIHB1YmxpYyBUYWJJQ0wgYXNzZXQnCmJhc2VfZW52WydOVExfVEFCSUNMX0FTU0VUJ109c3RyKGljbF9tYXRjaGVzWzBdLnBhcmVudCkKYmFzZV9lbnZbJ05UTF9UQUJVTEFSX1NJVEUnXT1zdHIoc2l0ZSkKc3VicHJvY2Vzcy5ydW4oW3N5cy5leGVjdXRhYmxlLHN0cihoZXJlLydzZXR1cF9pY2wucHknKV0sZW52PWJhc2VfZW52LGNoZWNrPVRydWUsdGltZW91dD0zOTApCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSxzdHIoaGVyZS8ncnVuX2Jhc2VsaW5lLnB5JyldLGN3ZD0nL2thZ2dsZS93b3JraW5nJyxlbnY9YmFzZV9lbnYsCiAgICBjaGVjaz1UcnVlLHRpbWVvdXQ9Y29uZmlnWydiYXNlbGluZV90aW1lb3V0X3NlY29uZHMnXSkKYmFzZWxpbmU9UGF0aCgnL2thZ2dsZS93b3JraW5nL3N1Ym1pc3Npb24ucGFycXVldCcpCmFzc2VydCBiYXNlbGluZS5pc19maWxlKCkKY29udHJvbD1QYXRoKCcva2FnZ2xlL3dvcmtpbmcvQjU1Nl9nYXRlZF9jb250cm9sLnBhcnF1ZXQnKQphc3NlcnQgbm90IGNvbnRyb2wuZXhpc3RzKCkKYmFzZWxpbmUucmVuYW1lKGNvbnRyb2wpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSxzdHIoaGVyZS8ndGFidWxhcl93b3JrZXIucHknKSwncHJlcGFyZSddLGVudj10YWJ1bGFyX2VudiwKICAgIGNoZWNrPVRydWUsdGltZW91dD1jb25maWdbJ3ByZXBhcmVfdGltZW91dF9zZWNvbmRzJ10pCmJhc2VfZW52WydOVExfVEFCVUxBUl9TSVRFJ109c3RyKHNpdGUpCnN1YnByb2Nlc3MucnVuKFtzeXMuZXhlY3V0YWJsZSwnLXUnLHN0cihoZXJlLydydW5fcGFyYWxsZWxfaWNsLnB5JyldLGVudj1iYXNlX2VudixjaGVjaz1UcnVlLAogICAgdGltZW91dD03NTAwKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsc3RyKGhlcmUvJ2ZpbmlzaF9zZXJpYWxfaWNsLnB5JyldLGVudj1iYXNlX2VudixjaGVjaz1UcnVlLHRpbWVvdXQ9MTIwKQphc3NlcnQgUGF0aCgnL2thZ2dsZS93b3JraW5nL3N1Ym1pc3Npb24ucGFycXVldCcpLmlzX2ZpbGUoKQphc3NlcnQganNvbi5sb2FkcyhQYXRoKCcva2FnZ2xlL3dvcmtpbmcvcHJvdGVjdGVkX3Jlc2lkdWFsX2F1ZGl0Lmpzb24nKS5yZWFkX3RleHQoKSlbJ3N0YXR1cyddPT0nU01PTF9DRU5URVJfUEZOMTBfREVGQVVMVElDTDA1X1NFUklBTF9DT01QTEVURSc=\",\"finish_serial_icl.py\":\"IiIiT3B0aW9uYWwgcmF3Ky4wMypTQywgb3JpZ2luYWwgYWxwaGEuNSBiYWxhbmNlL2dhdGUsIHRoZW4gc2hhcmVkIHByb3RlY3RlZCBQRk4uMTAuIiIiCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBjbGF1ZGVfZ2F0ZSBpbXBvcnQgZ2F0ZWRfcHJlZGljdCwgbG9hZF9nYXRlX3N0YXRlCmZyb20gcHJvdGVjdGVkX29wZXJhdG9yIGltcG9ydCBwcm90ZWN0ZWQKZnJvbSBzb3VyY2VfYmFsYW5jZSBpbXBvcnQgc291cmNlX2JhbGFuY2UKCmhlcmUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmZyb20gYWRtaXNzaW9uX3ByZWZsaWdodCBpbXBvcnQgdmFsaWRhdGUKY29uZmlnID0gdmFsaWRhdGUoaGVyZSkKZnJvbSBpY2xfYWRtaXNzaW9uIGltcG9ydCByZXF1aXJlX2FkbWlzc2lvbgpyZXF1aXJlX2FkbWlzc2lvbihoZXJlKQphc3NlcnQgY29uZmlnWydjYW5kaWRhdGVfYWRtaXR0ZWQnXSBhbmQgY29uZmlnWydzZWxlY3RlZF9ieV9yb290J10KYXNzZXJ0IGNvbmZpZ1snYXNzZXRzX3B1YmxpY19hdWRpdGVkJ10gYW5kIGNvbmZpZ1snZGlhZ25vc3RpY19hY3R1YWxfUEFTU19ib3VuZCddCmFzc2VydCBjb25maWdbJ3Jhd19leHRlbnNpb24nXSA9PSAnc21vbF9jZW50ZXJfYWRkJyBhbmQgY29uZmlnWydjb2VmZmljaWVudCddID09IC4wMwphc3NlcnQgY29uZmlnWydhbHBoYSddID09IC41IGFuZCBjb25maWdbJ3Byb3RlY3RlZF93ZWlnaHQnXSA9PSAuMTAgYW5kIGNvbmZpZ1sncG9zdCddID09IDAKZm9yIG5hbWUsIGRpZ2VzdCBpbiBjb25maWdbJ3NvdXJjZV9zaGEyNTYnXS5pdGVtcygpOgogICAgYXNzZXJ0IGhhc2hsaWIuc2hhMjU2KChoZXJlIC8gbmFtZSkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSA9PSBkaWdlc3QsIG5hbWUKY2FjaGUgPSBQYXRoKG9zLmVudmlyb25bJ05UTF9UQUJVTEFSX1RFTVAnXSkucmVzb2x2ZSgpCmFzc2VydCBjYWNoZS5pc19yZWxhdGl2ZV90byhQYXRoKCcva2FnZ2xlL3RlbXAnKS5yZXNvbHZlKCkpCgoKZGVmIGxvYWQobmFtZSk6CiAgICB3aXRoIG5wLmxvYWQoY2FjaGUgLyBuYW1lLCBhbGxvd19waWNrbGU9RmFsc2UpIGFzIHZhbHVlczoKICAgICAgICByZXR1cm4ge2tleTogdmFsdWVzW2tleV0uY29weSgpIGZvciBrZXkgaW4gdmFsdWVzLmZpbGVzfQoKCm9ic2VydmVkLCBzbW9sLCBwZm4gPSAobG9hZChuYW1lKSBmb3IgbmFtZSBpbgogICAgKCdiYXNlbGluZV9vYnNlcnZhdGlvbi5ucHonLCAnc21vbF9jZW50ZXJfYWRkLm5weicsICd0YWJwZm5fcHJvYmEubnB6JykpCnJhdywgUCwgaWRzLCBsYWJlbHMgPSAob2JzZXJ2ZWRba2V5XSBmb3Iga2V5IGluICgncmF3JywgJ2JhbGFuY2VkJywgJ2lkcycsICdsYWJlbHMnKSkKYXNzZXJ0IHJhdy5kdHlwZSA9PSBQLmR0eXBlID09IG5wLmZsb2F0NjQgYW5kIHJhdy5zaGFwZSA9PSBQLnNoYXBlID09IChsZW4oaWRzKSwgMTgpCmZvciB2YWx1ZXMgaW4gKHNtb2wsIHBmbik6CiAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbCh2YWx1ZXNbJ2lkcyddLCBpZHMpCiAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbCh2YWx1ZXNbJ2xhYmVscyddLCBsYWJlbHMpCmZvciBrZXkgaW4gKCdwb29sX2lkcycsICdyZWNvdmVyZWRfZ3JvdXBfaWRzJywgJ2FkYXB0aXZlX21hc2snKToKICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHNtb2xba2V5XSwgb2JzZXJ2ZWRba2V5XSkKIyBUaGUgZXh0cmFjdGVkIG9yaWdpbmFsIHNvdXJjZSBiYWxhbmNlIG11c3QgcmVwcm9kdWNlIHRoZSBvYnNlcnZlZCBiYXNlbGluZSBiaXQtZm9yLWJpdC4KbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoc291cmNlX2JhbGFuY2UocmF3KSwgUCkKZ2F0ZSA9IGxvYWRfZ2F0ZV9zdGF0ZShjYWNoZSwgUCwgaWRzLCBsYWJlbHMpCmJhc2VsaW5lID0gZ2F0ZWRfcHJlZGljdChQLCBsYWJlbHMsICoqZ2F0ZSkKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoYmFzZWxpbmUsIG9ic2VydmVkWydnYXRlZF9wcmVkaWN0aW9ucyddKQpjb250cm9sID0gcGQucmVhZF9wYXJxdWV0KCcva2FnZ2xlL3dvcmtpbmcvQjU1Nl9nYXRlZF9jb250cm9sLnBhcnF1ZXQnKQpucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChjb250cm9sLmlkLnRvX251bXB5KGR0eXBlPXN0ciksIGlkcykKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoY29udHJvbC5tb2RlbC50b19udW1weShkdHlwZT1zdHIpLCBsYWJlbHNbYmFzZWxpbmVdKQpTQyA9IHNtb2xbJ1NDJ10KYXNzZXJ0IFNDLmR0eXBlID09IG5wLmZsb2F0MzIgYW5kIFNDLnNoYXBlID09IHJhdy5zaGFwZSBhbmQgbnAuaXNmaW5pdGUoU0MpLmFsbCgpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKFNDW35vYnNlcnZlZFsnYWRhcHRpdmVfbWFzayddXSwgMCkKIyBTYW1lIEZQMzIgbXVsdGlwbGljYXRpb24gYW5kIHByb21vdGlvbiBvcmRlciBhcyB0aGUgYWN0dWFsIGM2NmM1IGNvbmRpdGlvbmFsIENQVSBhcm0uCmV4dGVuZGVkX3JhdyA9IHJhdyArIC4wMyAqIFNDCnEgPSBzb3VyY2VfYmFsYW5jZShleHRlbmRlZF9yYXcpCmV4dGVuZGVkX2xhYmVscyA9IGdhdGVkX3ByZWRpY3QocSwgbGFiZWxzLCAqKmdhdGUpCmNsYXVkZSA9IGdhdGVbJ0NMQVVERV9JRFgnXQpub25jbGF1ZGUgPSBucC5mbGF0bm9uemVybyh+bnAuY2hhci5zdGFydHN3aXRoKGxhYmVscywgJ2NsYXVkZScpKQphc3NlcnQgbGVuKGNsYXVkZSkgPT0gNSBhbmQgbGVuKG5vbmNsYXVkZSkgPT0gMTMKdGFidWxhciA9IHBmblsncHJvYmEnXQphc3NlcnQgdGFidWxhci5kdHlwZSA9PSBucC5mbG9hdDY0IGFuZCB0YWJ1bGFyLnNoYXBlID09IHEuc2hhcGUgYW5kIG5wLmlzZmluaXRlKHRhYnVsYXIpLmFsbCgpCmNhbmRpZGF0ZSwgcHJlZGljdGlvbiwgcmV2ZXJ0ZWQgPSBwcm90ZWN0ZWQocVtOb25lXSwgdGFidWxhciwgZXh0ZW5kZWRfbGFiZWxzW05vbmVdLCBjbGF1ZGUsIG5vbmNsYXVkZSwgd2VpZ2h0PS4xMCkKaWNsID0gbG9hZCgndGFiaWNsX3Byb2JhLm5weicpCm5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKGljbFsnaWRzJ10sIGlkcykKbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwoaWNsWydsYWJlbHMnXSwgbGFiZWxzKQphc3NlcnQgaWNsWydwcm9iYSddLmR0eXBlID09IG5wLmZsb2F0NjQgYW5kIGljbFsncHJvYmEnXS5zaGFwZSA9PSBxLnNoYXBlCmFzc2VydCBucC5pc2Zpbml0ZShpY2xbJ3Byb2JhJ10pLmFsbCgpIGFuZCAoaWNsWydwcm9iYSddID49IDApLmFsbCgpCm5wLnRlc3RpbmcuYXNzZXJ0X2FsbGNsb3NlKGljbFsncHJvYmEnXS5zdW0oMSksIDEsIHJ0b2w9MCwgYXRvbD0xZS01KQpwZm5fY2FuZGlkYXRlLCBwZm5fcHJlZGljdGlvbiwgcGZuX3JldmVydGVkID0gY2FuZGlkYXRlLCBwcmVkaWN0aW9uLCByZXZlcnRlZApjYW5kaWRhdGUsIHByZWRpY3Rpb24sIHJldmVydGVkID0gcHJvdGVjdGVkKHBmbl9jYW5kaWRhdGUsIGljbFsncHJvYmEnXSwgcGZuX3ByZWRpY3Rpb24sIGNsYXVkZSwgbm9uY2xhdWRlLCB3ZWlnaHQ9LjA1KQpucC5zYXZleihjYWNoZS8nc2VyaWFsLXByb3RlY3RlZC1vdXRwdXQubnB6JywgaWRzPWlkcywgbGFiZWxzPWxhYmVscywgcHJvYmE9Y2FuZGlkYXRlWzBdLCBwcmVkaWN0aW9ucz1wcmVkaWN0aW9uWzBdKQpzdWJtaXNzaW9uID0gcGQuRGF0YUZyYW1lKGRpY3QoaWQ9aWRzLCBtb2RlbD1sYWJlbHNbcHJlZGljdGlvblswXV0pKQphc3NlcnQgc3VibWlzc2lvbi5pZC5pc191bmlxdWUgYW5kIGxlbihzdWJtaXNzaW9uKSA9PSBsZW4oY29udHJvbCkgYW5kIHN1Ym1pc3Npb24ubW9kZWwuaXNpbihsYWJlbHMpLmFsbCgpCnN1Ym1pc3Npb24udG9fcGFycXVldCgnL2thZ2dsZS93b3JraW5nL3N1Ym1pc3Npb24ucGFycXVldCcsIGluZGV4PUZhbHNlKQpQYXRoKCcva2FnZ2xlL3dvcmtpbmcvcHJvdGVjdGVkX3Jlc2lkdWFsX2F1ZGl0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdCgKICAgIHN0YXR1cz0nU01PTF9DRU5URVJfUEZOMTBfREVGQVVMVElDTDA1X1NFUklBTF9DT01QTEVURScsIGJhc2VsaW5lX2JhbGFuY2VfYW5kX2dhdGVfcmVwbGF5X2JpdGV4YWN0PVRydWUsCiAgICBmb3JtdWxhPSdvcmlnaW5hbF9yYXcgKyAuMDMqU0MgLT4gb3JpZ2luYWxfYWxwaGEuNV9iYWxhbmNlIC0+IG9yaWdpbmFsX0NsYXVkZV9nYXRlIC0+IHByb3RlY3RlZF9QRk4uMTAgLT4gcHJvdGVjdGVkX2RlZmF1bHRJQ0wuMDUnLAogICAgc291cmNlX25vdGVib29rX2lkPTEzOTIwMDEwMCwgc291cmNlX3ZlcnNpb249MiwgY29lZmZpY2llbnQ9LjAzLCBhbHBoYT0uNSwgcG9zdD0wLAogICAgcmVzaWR1YWxfd2VpZ2h0PS4xMCwgc2VyaWFsX2ljbF93ZWlnaHQ9LjA1LCBjb25kaXRpb25hbF9leHBvbmVudHNfbm9ucmV2ZXJ0ZWQ9Wy44NTUsLjA5NSwuMDVdLCBxdWVyeV9yb3dzPWxlbihpZHMpLCBleHRyYV9nbG9iYWxfbm9ybWFsaXphdGlvbj1GYWxzZSwKICAgIHByb3RlY3RlZF9mYW1pbHlfbWVtYmVyc2hpcF9yZWxhdGl2ZV90bz0nU21vbENFTlRFUiBleHRlbmRlZCBvcmlnaW5hbCBnYXRlIGRlY2lzaW9ucycsCiAgICByZXZlcnRlZF9mYW1pbHlfZmxpcHM9aW50KHJldmVydGVkLnN1bSgpKSwgcGZuX3JldmVydGVkX2ZhbWlseV9mbGlwcz1pbnQocGZuX3JldmVydGVkLnN1bSgpKSwgdHJhaW5pbmc9RmFsc2UpLCBpbmRlbnQ9MikpCg==\",\"icl_admission.py\":\"IiIiQ2xvc2VkIHByb2R1Y3Rpb24gYWRtaXNzaW9uIGZvciB0aGUgZml4ZWQgc2VyaWFsIGRlZmF1bHQtVGFiSUNMIHJlc2lkdWFsLiIiIgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgcmVxdWlyZV9hZG1pc3Npb24oaGVyZSk6CiAgICBoZXJlID0gUGF0aChoZXJlKQogICAgY29uZmlnID0ganNvbi5sb2FkcygoaGVyZSAvICdpY2wtY29uZmlnLmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGFzc2VydCBjb25maWdbJ3Byb2R1Y3Rpb25fYWRtaXR0ZWQnXSBhbmQgY29uZmlnWydsYXVuY2hfYXV0aG9yaXplZCddIGFuZCBjb25maWdbJ3NlbGVjdGVkX2J5X3Jvb3QnXQogICAgYXNzZXJ0IGNvbmZpZ1snYXNzZXRzX3B1YmxpY19hdWRpdGVkJ10gYW5kIGNvbmZpZ1sncm9vdF9kZWNpc2lvbl9iaW5kaW5nJ10KICAgIGFzc2VydCBjb25maWdbJ2ZpeGVkX3JlY2lwZSddID09IGRpY3Qoc21vbF9jZW50ZXI9LjAzLCBhbHBoYT0uNSwgcG9zdD0wLCBwcm90ZWN0ZWRfcGZuPS4xMCwgcHJvdGVjdGVkX2ljbD0uMDUpCiAgICBhc3NlcnQgY29uZmlnWydzZWxlY3RlZF93ZWlnaHRzJ10gPT0gY29uZmlnWydmaXhlZF9yZWNpcGUnXQogICAgYmluZGluZyA9IGNvbmZpZ1snZGlhZ25vc3RpY19hY3R1YWxfUEFTU19iaW5kaW5nJ10KICAgIGFzc2VydCBiaW5kaW5nIGFuZCBiaW5kaW5nWydub3RlYm9va19pZCddIGFuZCBiaW5kaW5nWyd2ZXJzaW9uJ10gYW5kIGJpbmRpbmdbJ3NjcmlwdF92ZXJzaW9uX2lkJ10KICAgIHJhdyA9IChoZXJlIC8gJ2ljbC1hY3R1YWwtZGlhZ25vc3RpYy1hdWRpdC5qc29uJykucmVhZF9ieXRlcygpCiAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKSA9PSBiaW5kaW5nWydhdWRpdF9zaGEyNTYnXQogICAgcmVwb3J0ID0ganNvbi5sb2FkcyhyYXcpCiAgICBhc3NlcnQgcmVwb3J0WydzdGF0dXMnXSA9PSAnUEFTU19TQU1FNTM5X0ZJUlNUNF9ERUZBVUxUNjRfQU5DSE9SUycKICAgIGFzc2VydCByZXBvcnRbJ21vZGVsX3dvcmtlcl9zaGEyNTYnXSA9PSBjb25maWdbJ3NvdXJjZV9zaGEyNTYnXVsndGFiaWNsX2RlZmF1bHRfd29ya2VyLnB5J10KICAgIGFzc2VydCByZXBvcnRbJ2lucHV0X3NoYTI1NiddID09IGNvbmZpZ1snY2Fub25pY2FsX2lucHV0X3NoYTI1NiddCiAgICBhc3NlcnQgcmVwb3J0WydyZWZlcmVuY2Vfc2hhMjU2J10gPT0gY29uZmlnWydjYW5vbmljYWxfcmVmZXJlbmNlX3NoYTI1NiddCiAgICBhc3NlcnQgcmVwb3J0WydmdWxsX3F1ZXJ5X3Jvd3MnXSA9PSA1MzkgYW5kIHJlcG9ydFsnYW5jaG9ycyddID09IDQKICAgIGFzc2VydCByZXBvcnRbJ3NhbWU1MzlfcXVlcnlfb3JkZXInXSBhbmQgcmVwb3J0WydhcmdtYXhfZXhhY3QnXSBhbmQgcmVwb3J0Wydub19ncmFkaWVudF90cmFpbmluZyddCiAgICBhc3NlcnQgcmVwb3J0Wydwcm9iYWJpbGl0eV9ydG9sJ10gPT0gMCBhbmQgcmVwb3J0Wydwcm9iYWJpbGl0eV9hdG9sJ10gPT0gMWUtNgogICAgYXNzZXJ0IHJlcG9ydFsncGFja2FnZXMnXSA9PSBjb25maWdbJ2V4cGVjdGVkX3BhY2thZ2VzJ10KICAgIGZvciBuYW1lLCBkaWdlc3QgaW4gY29uZmlnWydzb3VyY2Vfc2hhMjU2J10uaXRlbXMoKToKICAgICAgICBhc3NlcnQgaGFzaGxpYi5zaGEyNTYoKGhlcmUgLyBuYW1lKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpID09IGRpZ2VzdCwgbmFtZQogICAgcmV0dXJuIGNvbmZpZwo=\",\"icl-config.json\":\"ew0KICAic3RhdHVzIjogIlJPT1RfU0VMRUNURURfUFVCTElDQVRJT05fUkVBRFlfTE9DQUxfRFJBRlQiLA0KICAicHJvZHVjdGlvbl9hZG1pdHRlZCI6IHRydWUsDQogICJsYXVuY2hfYXV0aG9yaXplZCI6IHRydWUsDQogICJzZWxlY3RlZF9ieV9yb290IjogdHJ1ZSwNCiAgImFzc2V0c19wdWJsaWNfYXVkaXRlZCI6IHRydWUsDQogICJkaWFnbm9zdGljX2xhdW5jaF9hdXRob3JpemVkIjogZmFsc2UsDQogICJkaWFnbm9zdGljX2FjdHVhbF9QQVNTX2JpbmRpbmciOiB7DQogICAgIm5vdGVib29rX2lkIjogMTM5Mzc3OTI0LA0KICAgICJ2ZXJzaW9uIjogMSwNCiAgICAic2NyaXB0X3ZlcnNpb25faWQiOiAzNTk2MTg4NTAsDQogICAgImF1ZGl0X3NoYTI1NiI6ICIxNTJmMTY1NTZhYzA3OWU5MzA0NTA5YzZmZGJjODYwMjg0ZmY3ODM1ZGE1NmQ3YTg5NjYzNmVkNWM2M2FhYzgwIiwNCiAgICAicnVudGltZV9hdWRpdF9zaGEyNTYiOiAiMTAzNTAxZmUxMWYwNDQ2MjYzNzM0NGQ2YmQ4YmQ0M2RiZDBkYzUxYjQ0NjA0NWY0NDA3MDgxZWUxMWE4Nzg3ZiIsDQogICAgImRpYWdub3N0aWNfbm90ZWJvb2tfc2hhMjU2IjogIjc3OWMxNDhhMTM1MTM1YTc2NGQ1ZjAxYTI0MWVhYjI0NTRkZDlkM2I1MzBlOGZjOTVjZmM0ZWJkNzMxZDE1MmIiDQogIH0sDQogICJyb290X2RlY2lzaW9uX2JpbmRpbmciOiB7DQogICAgImV4cGxpY2l0X2NhbmRpZGF0ZV9hZG1pc3Npb24iOiB0cnVlLA0KICAgICJzY29wZSI6ICJMT0NBTF9QVUJMSUNBVElPTl9SRUFEWV9EUkFGVF9PTkxZIiwNCiAgICAic2VsZWN0ZWRfYnJhbmNoIjogIlNtb2xDRU5URVIwM19vcmlnaW5hbF9hbHBoYTA1X0NsYXVkZV9QRk4xMjhwcm90ZWN0ZWQxMF9kZWZhdWx0SUNMNjRwcm90ZWN0ZWQwNSIsDQogICAgInBhaXJlZF9ub3RlYm9va19pZCI6IDEzOTM3NTEyOSwNCiAgICAicGFpcmVkX3ZlcnNpb24iOiAxLA0KICAgICJwYWlyZWRfc2NyaXB0X3ZlcnNpb25faWQiOiAzNTk2MTM1NDUsDQogICAgInBhaXJlZF9yZXBvcnRfc2hhMjU2IjogImU1NmQyNGQ1Y2I1ZmZlYjVlNTVmNzFhOTZkZDc1NTc0YjQ3MGRlZGI3ZTUxMGFlNWQ4OGQzYjhhYWM4OGIwNmIiLA0KICAgICJwYWlyZWRfaW5jcmVtZW50X3ZzX3NlbGVjdGVkX1Ntb2xfY29udHJvbCI6IDAuMDAxODUxODUxODUxODUxODUxNSwNCiAgICAicGFpcmVkX2NvbnRleHRfaW5jcmVtZW50cyI6IFsNCiAgICAgIDAsDQogICAgICAwLjAwMTg1MTg1MTg1MTg1MTg1LA0KICAgICAgMC4wMDM3MDM3MDM3MDM3MDM3DQogICAgXSwNCiAgICAiaW5kZXBlbmRlbnRfZml2ZWZvbGRfdmFsaWRhdGlvbiI6IGZhbHNlLA0KICAgICJDbGF1ZGVfYXV0aG9yX2V4Y2x1c2lvbl9hc3N1bXB0aW9uX3JldGFpbmVkIjogdHJ1ZSwNCiAgICAiYWJzb2x1dGVfbGVhZGVyYm9hcmRfZm9yZWNhc3QiOiBudWxsLA0KICAgICJuZXdfcmVtb3RlX3J1bl9vcl9zdWJtaXNzaW9uX2F1dGhvcml6ZWRfYnlfcGFja2FnZXIiOiBmYWxzZQ0KICB9LA0KICAic2VsZWN0ZWRfd2VpZ2h0cyI6IHsNCiAgICAic21vbF9jZW50ZXIiOiAwLjAzLA0KICAgICJhbHBoYSI6IDAuNSwNCiAgICAicG9zdCI6IDAsDQogICAgInByb3RlY3RlZF9wZm4iOiAwLjEsDQogICAgInByb3RlY3RlZF9pY2wiOiAwLjA1DQogIH0sDQogICJmaXhlZF9yZWNpcGUiOiB7DQogICAgInNtb2xfY2VudGVyIjogMC4wMywNCiAgICAiYWxwaGEiOiAwLjUsDQogICAgInBvc3QiOiAwLA0KICAgICJwcm90ZWN0ZWRfcGZuIjogMC4xLA0KICAgICJwcm90ZWN0ZWRfaWNsIjogMC4wNQ0KICB9LA0KICAicGFyYWxsZWxfcGhhc2VfdGltZW91dF9zZWNvbmRzIjogNzUwMCwNCiAgImVhY2hfbmV1cmFsX3RpbWVvdXRfc2Vjb25kcyI6IDM2MDAsDQogICJiYXNlbGluZV90aW1lb3V0X3NlY29uZHMiOiAxMzIwMCwNCiAgImRpYWdub3N0aWNfcGh5c2ljYWxfZ3B1X2luZGV4IjogIjEiLA0KICAiY2Fub25pY2FsX2lucHV0X3NoYTI1NiI6ICI1NzM0Yzk4NmFkMDYwZmE4MWYxZmRjZTZjMmM4ZmI0MTgzMmYzNjMzNmU2NTRlMzIyNDRhZmFiM2JlZGVmMDQ5IiwNCiAgImNhbm9uaWNhbF9yZWZlcmVuY2Vfc2hhMjU2IjogImI0YWNlMDJiZjRkMDQ3OTRlZjc2MGU3YWY2ZjA3MDE3MmE5MzQyZWNlMmMzZmQzYzYyM2YzODY4YzE4OThjZTkiLA0KICAiY2Fub25pY2FsX3JlcG9ydF9zaGEyNTYiOiAiYTRhOWZkZmZiNTAwYTNhMjFlMTM3OWE4ODlkZDAwOWRhOTUxMDk2MjUxZGYxZWQ4ZGNjMzg4ZTZkZTEyMzgxZSIsDQogICJjaGVja3BvaW50X3NoYTI1NiI6ICJiYjQyZDE3MzNiNDlmYWIzYTdhOWU0MmRiNjQ4YzhmZjY0ZDhkNTcwNWQ5M2MwMzkxNzlhZGQzYmZiMmRiMzJiIiwNCiAgIndoZWVsX2ZpbGVuYW1lIjogInRhYmljbC0yLjIuMC1weTMtbm9uZS1hbnkud2hsIiwNCiAgIndoZWVsX3NoYTI1NiI6ICJjODBiOGVjMjcxOWJlMjZiYzBiZjFkZDc1OGEyMzYxZjU4MDE5ZDhmNzg3NzAzOGI3ODU1MjlhNDI1Nzg5Y2U0IiwNCiAgInRhYmljbF9sb2NrX3NoYTI1NiI6ICIzMzcxZmM1ZGMzZGVlOGMyYWJjNDIxNzg5MWU4NTZkZmY3ZGNkNTM5OTVhOTdhYTA0YzBmZDMzYWM3NjZmNGZjIiwNCiAgInBmbl9sb2NrX3NoYTI1NiI6ICI2OTdiYTY4MTJkZDIwMzY0ZGEyNzNkOGZkNWMzNTYyMzE4ZmJkYjI0OWMyYTE0MjY4Y2YzMTBmMDUyMTczM2I5IiwNCiAgImV4cGVjdGVkX3BhY2thZ2VzIjogew0KICAgICJ0YWJpY2wiOiAiMi4yLjAiLA0KICAgICJ0b3JjaCI6ICIyLjExLjArY3UxMjgiLA0KICAgICJudW1weSI6ICIyLjEuMyIsDQogICAgInNjaWtpdC1sZWFybiI6ICIxLjYuMSIsDQogICAgImVpbm9wcyI6ICIwLjguMiINCiAgfSwNCiAgInNvdXJjZV9zaGEyNTYiOiB7DQogICAgInRhYmljbF9kZWZhdWx0X3dvcmtlci5weSI6ICJmODYwYjEyNDYyMmEzNjdiNGQwMmQ2NWVkYjU5N2ZmM2FkZTI1M2FhN2YzNDlkMmU3MWQyMDU5MmI0ZGMyYWFiIiwNCiAgICAic2V0dXBfaWNsLnB5IjogIjAyYmZjNjViNDMxNjQ2MGQzZDM3NGM5YWIzNDNkZGNmOGFmZWVlYmU3YmExZWIyZTczYjQwMmQyMjkxZjNlOTgiLA0KICAgICJydW5fcGFyYWxsZWxfaWNsLnB5IjogIjJhYThkMTFmNTQyOGVkMzM1NWM1Njc2M2Q4OWRjZGQxODg4NDg2NzFjNTBmNDlkZjZhY2UyODk2MTY1ZTlmM2QiLA0KICAgICJydW5fY2FuZGlkYXRlX2ljbC5weSI6ICJjNTE1MmYwNDcyZTE5ZGIwM2FjZTUzYWU4ODEzYTAwNzU4Yjg3OTVlOWE4YWJmNjg0NDViZGIwNDhlMzJiZDgyIiwNCiAgICAiZmluaXNoX3NlcmlhbF9pY2wucHkiOiAiOTY5NjU1ZDY3ZTllZWRhYzUzNGI5MmZkZmIwMDU4NmFiMTRkYTg5OTc5ZDI4MTY5YWUzMjI2MTgyMmMyNDM4ZCIsDQogICAgImljbF9hZG1pc3Npb24ucHkiOiAiYTQ1MTJjNmY2YTExM2NkYzIxZWEwODEwM2ZjZTVlYjgwNDM5NTU3ZjI3ZjA4ZDViNDNhNjI3MTE5NDhmZjIyYSIsDQogICAgInJ1bnRpbWVfY29uZmlnLmpzb24iOiAiMWRlYjNkMzc0NjRhYmZmYjdjY2NmNDlmYjJmNmZhMjIwMzQ3YzJhOWM5ZTIwYTNmMDdjNWFjN2RlZmI4NWVmNiINCiAgfSwNCiAgImRpYWdub3N0aWNfc291cmNlX3NoYTI1NiI6IHsNCiAgICAidGFiaWNsX2RlZmF1bHRfd29ya2VyLnB5IjogImY4NjBiMTI0NjIyYTM2N2I0ZDAyZDY1ZWRiNTk3ZmYzYWRlMjUzYWE3ZjM0OWQyZTcxZDIwNTkyYjRkYzJhYWIiLA0KICAgICJzZXR1cF9pY2wucHkiOiAiMDJiZmM2NWI0MzE2NDYwZDNkMzc0YzlhYjM0M2RkY2Y4YWZlZWViZTdiYTFlYjJlNzNiNDAyZDIyOTFmM2U5OCIsDQogICAgImRpYWdub3N0aWMucHkiOiAiZjBiZjU2MTZjZjRhMDc2YTJkNTMzZmZmOTc5YzAwZThhM2Q2MjI0NWQxMWQ0ZDE5NmM1Y2VlMWFjNWIwYmZhOCIsDQogICAgInJ1bnRpbWVfY29uZmlnLmpzb24iOiAiNjMzNTUyMzVlYzhiMTA4MWUwMmNlMmUxMzQ5Njk5YTg1NTZlNGZhOTQ1YzY0OTdlNWFhMmUwNTA5MjFjMDdkYiIsDQogICAgImNhbm9uaWNhbF9vdXRlcjAuanNvbiI6ICI0ZGZhODgxZDQ2OWVlM2NmNTFhNzE0ZmY1N2EwMDdkYjFhMjAzZDVhYTJkZWIxMDU3OGVhYTQ4MzA4NWQ4YTFkIg0KICB9DQp9\",\"icl-actual-diagnostic-audit.json\":\"ewogICJzdGF0dXMiOiAiUEFTU19TQU1FNTM5X0ZJUlNUNF9ERUZBVUxUNjRfQU5DSE9SUyIsCiAgImFuY2hvcnMiOiA0LAogICJmdWxsX3F1ZXJ5X3Jvd3MiOiA1MzksCiAgInNhbWU1MzlfcXVlcnlfb3JkZXIiOiB0cnVlLAogICJhcmdtYXhfZXhhY3QiOiB0cnVlLAogICJwcm9iYWJpbGl0eV9ydG9sIjogMCwKICAicHJvYmFiaWxpdHlfYXRvbCI6IDFlLTA2LAogICJmaXJzdDRfbWF4YWJzIjogNy40NTA1ODA1OTY5MjM4MjhlLTA4LAogICJmdWxsNTM5X21heGFic19kaWFnbm9zdGljIjogMS43ODgxMzkzNDMyNjE3MTg4ZS0wNywKICAibm9fZ3JhZGllbnRfdHJhaW5pbmciOiB0cnVlLAogICJtb2RlbF93b3JrZXJfc2hhMjU2IjogImY4NjBiMTI0NjIyYTM2N2I0ZDAyZDY1ZWRiNTk3ZmYzYWRlMjUzYWE3ZjM0OWQyZTcxZDIwNTkyYjRkYzJhYWIiLAogICJpbnB1dF9zaGEyNTYiOiAiNTczNGM5ODZhZDA2MGZhODFmMWZkY2U2YzJjOGZiNDE4MzJmMzYzMzZlNjU0ZTMyMjQ0YWZhYjNiZWRlZjA0OSIsCiAgInJlZmVyZW5jZV9zaGEyNTYiOiAiYjRhY2UwMmJmNGQwNDc5NGVmNzYwZTdhZjZmMDcwMTcyYTkzNDJlY2UyYzNmZDNjNjIzZjM4NjhjMTg5OGNlOSIsCiAgInBhY2thZ2VzIjogewogICAgInRhYmljbCI6ICIyLjIuMCIsCiAgICAidG9yY2giOiAiMi4xMS4wK2N1MTI4IiwKICAgICJudW1weSI6ICIyLjEuMyIsCiAgICAic2Npa2l0LWxlYXJuIjogIjEuNi4xIiwKICAgICJlaW5vcHMiOiAiMC44LjIiCiAgfSwKICAid29ya2VyIjogewogICAgInN0YXR1cyI6ICJDT01QTEVURSIsCiAgICAiY2hlY2twb2ludF9zaGEyNTYiOiAiYmI0MmQxNzMzYjQ5ZmFiM2E3YTllNDJkYjY0OGM4ZmY2NGQ4ZDU3MDVkOTNjMDM5MTc5YWRkM2JmYjJkYjMyYiIsCiAgICAicGFyYW1ldGVycyI6IHsKICAgICAgImNoZWNrcG9pbnRfdmVyc2lvbiI6ICJ0YWJpY2wtY2xhc3NpZmllci12MS4xLTIwMjUwNTA2LmNrcHQiLAogICAgICAibW9kZWxfcGF0aCI6ICIva2FnZ2xlL2lucHV0L2RhdGFzZXRzL2xpZ2h0bmluZ3YwOC9udGwtdGFiaWNsLW9mZmxpbmUtYXNzZXRzL3RhYmljbC1jbGFzc2lmaWVyLXYxLjEtMjAyNTA1MDYuY2twdCIsCiAgICAgICJhbGxvd19hdXRvX2Rvd25sb2FkIjogZmFsc2UsCiAgICAgICJuX2VzdGltYXRvcnMiOiA2NCwKICAgICAgIm5vcm1fbWV0aG9kcyI6IG51bGwsCiAgICAgICJzb2Z0bWF4X3RlbXBlcmF0dXJlIjogMC45LAogICAgICAiYXZlcmFnZV9sb2dpdHMiOiBmYWxzZSwKICAgICAgInN1cHBvcnRfbWFueV9jbGFzc2VzIjogdHJ1ZSwKICAgICAgImJhdGNoX3NpemUiOiA0LAogICAgICAia3ZfY2FjaGUiOiBmYWxzZSwKICAgICAgImRldmljZSI6ICJjdWRhOjAiLAogICAgICAidXNlX2FtcCI6IHRydWUsCiAgICAgICJ1c2VfZmEzIjogZmFsc2UsCiAgICAgICJvZmZsb2FkX21vZGUiOiAiYXV0byIsCiAgICAgICJyYW5kb21fc3RhdGUiOiA3NzcsCiAgICAgICJuX2pvYnMiOiA0CiAgICB9LAogICAgInJlc29sdmVkX25vcm1fbWV0aG9kcyI6IFsKICAgICAgIm5vbmUiLAogICAgICAicG93ZXIiCiAgICBdLAogICAgInRyYWluX3Jvd3MiOiAyMTYwLAogICAgInF1ZXJ5X3Jvd3MiOiA1MzksCiAgICAiZmVhdHVyZV9jb2x1bW5zIjogMTUyLAogICAgInNvdXJjZV90YWJsZV9jb2x1bW5zIjogMTg0LAogICAgInVuY2hhbmdlZF8xNTJfY29sdW1uX3ByZWZpeCI6IHRydWUsCiAgICAiZml0X3NlY29uZHMiOiAxLjc1OTg0NzU3NzAwMDAxNjQsCiAgICAicHJlZGljdF9zZWNvbmRzIjogMjAuODY2OTExMjQ4MDAwMDA4LAogICAgInBlYWtfY3VkYV9hbGxvY2F0ZWRfYnl0ZXMiOiA3MDEzOTk3MDU2LAogICAgInBlYWtfY3VkYV9yZXNlcnZlZF9ieXRlcyI6IDc0NjU4NjExMjAsCiAgICAicGFja2FnZXMiOiB7CiAgICAgICJ0YWJpY2wiOiAiMi4yLjAiLAogICAgICAidG9yY2giOiAiMi4xMS4wK2N1MTI4IiwKICAgICAgIm51bXB5IjogIjIuMS4zIiwKICAgICAgInNjaWtpdC1sZWFybiI6ICIxLjYuMSIsCiAgICAgICJlaW5vcHMiOiAiMC44LjIiCiAgICB9LAogICAgInByb2JhYmlsaXR5X3RyYW5zZm9ybSI6ICJub25lIiwKICAgICJyb3dfbWFzc19tYXhhYnMiOiAyLjAwMTMwMTAyNzEyMTM3MTZlLTA3LAogICAgIm9vZiI6IGZhbHNlLAogICAgIm1vZGVsX2NvbnRleHRfZXhwb3J0ZWQiOiBmYWxzZSwKICAgICJQSUQiOiAxMDQsCiAgICAiQ1VEQV9WSVNJQkxFX0RFVklDRVMiOiAiMSIKICB9LAogICJwcm9kdWN0aW9uX2FkbWl0dGVkIjogZmFsc2UsCiAgImZpdF9pc19pbl9jb250ZXh0X3ByZXBhcmF0aW9uIjogdHJ1ZQp9\"}")
digests=json.loads("{\"ntl_reference.py\":\"b856e736d5cb47d8fa309ce2acd5ce0588a4f91478bdeae99c491f31b7c26a4c\",\"capture_runtime.py\":\"172aa2b5c336d6b277cdcf19d4d54e63749dc2b076f6b17a4cc56b71386275d6\",\"claude_gate.py\":\"3fd1971a10eb15df21dd2c25acf6516d22e7fb07708b4cc0d4a872681fb37ae4\",\"masked_features.py\":\"ce6313975b0f8eb8a3b7c8ce9cd0d1656d0e16bd1f7340d50dbc42ec1ac991b2\",\"tabular_worker.py\":\"71fa6fda5562f012a2d4b96bb8dcd862378f26fc39ece6068b97fce5e43af067\",\"run_baseline.py\":\"28e4e8f93a14615532e07a0a141b06dd6630437c4cc1ed330b515436b2b8826a\",\"protected_operator.py\":\"9d9f8a89b92935075131d1e3a26a7948cbbe6eb1495d5ca808329a5ecda3184a\",\"QWEN35_LICENSE\":\"bbedc3fda3305820b977265f01b8619d87570a6739de3a5582c3464840f1e57a\",\"QWEN35_NOTICE.md\":\"5ce58b5e8ad74b3d9bf751e66b84721a0663311bd001ecc0d7a57d0c4c348c61\",\"component.py\":\"ab1df94cea1358d95125ed99f4fbe9606f7e0690ca6b6938ead419f884343f46\",\"owned_path.py\":\"20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620\",\"run_component.py\":\"8d6ee8631ffa99baaf3f9fdb49e3d8a4dbd6dbe2284e2ea1617ea5965902052b\",\"center_readout.py\":\"499b99f6ca94b6b4b8eb1fca15a4ef0519079194406441e70f8d0e2443068072\",\"source_balance.py\":\"da8682bc8298aed63c89607ceb08498a47a20ba7ab74752dddbdda996eab2351\",\"compute_center.py\":\"3dbf7ca0c3769fbd4ade05346e4daeb791d589d7a7361e967f08d44c46999bbd\",\"finish_center_protected.py\":\"aa4e54e2fdc8ea4f209ae42138d6b808ac07c45bf232c68db0318c5df8da0a47\",\"admission_preflight.py\":\"59de3c41b2a1146a11797008b80e77859a0282b0d8b14c6a152fafc9dba19eeb\",\"run_parallel.py\":\"3ff30d27d811decbe0a19628a9933ae9a797f5191d303f7cd3257883ba66599f\",\"run_candidate_optional.py\":\"ab170cfa87abbfebcffb877a4b9db6b4cfa25944531b320e2e35da20822e013b\",\"actual-diagnostic-audit.json\":\"8e76708fb1cca9239ffabfe7e8cd029b954d3957577f034156c3cfda4a9ba525\",\"selection-evidence.json\":\"33c8a8a9e822ff608924e37007cfe93f40f9fd80f9386927097ca86cd95dbe67\",\"runtime_config.json\":\"1deb3d37464abffb7cccf49fb2f6fa220347c2a9c9e20a3f07c5ac7defb85ef6\",\"optional-config.json\":\"06573fd83a350bf60cbf302bab73dc4c89c57fc11621d558d1518e4e8342abbe\",\"smol-spec.json\":\"a37480c78e4c53f1fed653482370a9c1867b34e775eac01a3d837a85d4fe9529\",\"tabicl_default_worker.py\":\"f860b124622a367b4d02d65edb597ff3ade253aa7f349d2e71d20592b4dc2aab\",\"setup_icl.py\":\"02bfc65b4316460d3d374c9ab343ddcf8afeeebe7ba1eb2e73b402d2291f3e98\",\"run_parallel_icl.py\":\"2aa8d11f5428ed3355c56763d89dcdd188848671c50f49df6ace2896165e9f3d\",\"run_candidate_icl.py\":\"c5152f0472e19db03ace53ae8813a00758b8795e9a8abf68445bdb048e32bd82\",\"finish_serial_icl.py\":\"969655d67e9eedac534b92fdfb00586ab14da89979d28169ae32261822c2438d\",\"icl_admission.py\":\"a4512c6f6a113cdc21ea08103fce5eb80439557f27f08d5b43a62711948ff22a\",\"icl-config.json\":\"5f7484c1016256494549d602078011b8e422bb389315190e045eca853139d64d\",\"icl-actual-diagnostic-audit.json\":\"152f16556ac079e9304509c6fdbc860284ff7835da56d7a896636ed5c63aac80\"}")
for name, encoded in payload.items():
    assert '/' not in name and '\\' not in name
    raw=base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest()==digests[name],name
    (here/name).write_bytes(raw)
    if name.endswith('.py'): ast.parse(raw,filename=name)
owned={Path(name).stem for name in payload if name.endswith('.py')}
known={'os','sys','pathlib','json','hashlib','subprocess','signal','time','tempfile','re','importlib','numpy','pandas','torch','threadpoolctl','tabpfn','tabicl','einops','sklearn','scipy','collections','glob','gc','functools','argparse','transformers','peft','safetensors','lightgbm','ast','csv'}
for name in payload:
    if name.endswith('.py'):
        for node in ast.walk(ast.parse((here/name).read_text())):
            modules=([node.module] if isinstance(node,ast.ImportFrom) else [v.name for v in node.names] if isinstance(node,ast.Import) else [])
            for module in modules:
                assert module and module.split('.')[0] in owned|known,(name,module)
print(json.dumps(dict(status='EXACT_PAYLOAD_AST_IMPORT_CLOSURE_PASS',files=digests)),flush=True)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u',str(here/'run_candidate_icl.py')],cwd=here,check=True,timeout=22000)
